# SC3000 DPO Lab — Instructor Steps 1–8

**Final model: Round9 Random E5.** This Notebook follows the original course template structure (**Step 1–Step 8**) and retains the instructor's character-level NanoGPT architecture, Chosen/Rejected teacher forcing, whole-sequence mean preference loss, AdamW + cosine scheduler and greedy inference. The original Steps 1–8 remain in place, with the historical experiment integrated below them.

**Historical lineage (not “5 epochs from SFT”):** SFT → Round3 (5 epochs) → Round4 Treatment (E1 of 2-epoch schedule) → Round5 Original (1) → Round7 Replay (1) → Random E1 (1) → Random E2 (1, optimizer reset) → Random E3, E4, E5 (3 of a 4-epoch schedule). **13 training epochs total**, 20,306 gradient updates on 100k-row stages.

**Run All** runs/restarts those stages sequentially, writes a checkpoint at each epoch, and finally evaluates on fixed development splits and exports `dpo/dpo.pt`. It requires the provided SFT checkpoint, 74-character tokenizer and model.py; the reproduction code and generator implementations are included in this notebook. Exact GPU parity with archived weights is measured rather than assumed.

**Note:** Original teacher Step 1 was general installation guidance; this environment uses the already prepared `.venv`. Do not reinstall PyTorch inside an existing CUDA environment. Full 13-epoch reproduction may require multiple TC1 Slurm allocations; rerunning Run All verifies and resumes completed epochs. The graded final test is not used for model selection.

### Step 1: Install necesscary packages

In [ ]:
# Original teacher Step 1 installation guidance (already installed on TC1):
# %pip install matplotlib
# %pip install torch numpy transformers datasets tiktoken wandb tqdm
# Do not unconditionally re-run pip here: it could replace the tested CUDA/PyTorch versions.
import sys, importlib.util
for package in ("torch","numpy","tqdm"):
    if importlib.util.find_spec(package) is None:
        raise ImportError(f"{package} is missing from the selected Jupyter kernel")
print("Step 1 PASS: using the selected existing Python kernel:",sys.executable)

### Step 2: Package imports and configuration

In [ ]:
# Step 2 — Instructor imports, configuration, tokenizer and environment checks
from __future__ import annotations
import os, sys, json, pickle, random, math, hashlib, shutil, time, re, subprocess
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone

import torch
import torch.nn.functional as F

workdir = Path.cwd().resolve()
if (workdir / "model.py").is_file() and (workdir / "dpo" / "dpo.ipynb").exists():
    repo_root = workdir
elif (workdir / "dpo.ipynb").is_file() and (workdir.parent / "model.py").is_file():
    repo_root = workdir.parent
else:
    raise RuntimeError("Open from NanoGPT-Math root/ or NanoGPT-Math/dpo/.")

sys.path.insert(0, str(repo_root))
sys.path.insert(0, str(repo_root / "scripts"))
from model import GPT, GPTConfig

# Keep the originally supplied pretrained 8.84M-parameter NanoGPT checkpoint.
SFT = repo_root / "sft" / "gpt.pt"
META = repo_root / "sft" / "meta.pkl"
for required in (SFT, META):
    if not required.is_file():
        raise FileNotFoundError(f"Missing REQUIRED course-provided input: {required}")

with META.open("rb") as fh:
    vocab = pickle.load(fh)
stoi, itos = vocab["stoi"], vocab["itos"]
assert len(stoi) == len(itos) == 74, "Expected original 74-character vocabulary"
assert stoi["\n"] == 0 and stoi["."] == 7, "Original stopping IDs changed"

# FULL: 13 historical 100k training epochs across seven stages.
# SMOKE: replace each 100k stage by a 128-row mini-stage and NEVER publish dpo.pt.
FAST_SMOKE = False
# Preserve the historically selected E5 as the submitted model when present.
# Full replay weights are separately retained and parity-audited; do not
# silently replace a 614/768 selected checkpoint with a divergent rerun.
FINAL_EXPORT_POLICY = "archived_if_available"
AUTO_BUILD_MISSING_DATA = True
MAX_NEW_TOKENS = 80
BATCH_SIZE = 64
SEED = 42
BETA = 0.5
TRAIN_BASE_LENGTH = 64
device = "cuda" if torch.cuda.is_available() else "cpu"
RUN_DIR = (repo_root / "dpo" / "experiments" /
           ("final_notebook_smoke" if FAST_SMOKE else "final_notebook_e5_reproduction"))
RUN_DIR.mkdir(parents=True, exist_ok=True)

def sha256(path):
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as fh:
        for part in iter(lambda: fh.read(1024 * 1024), b""):
            h.update(part)
    return h.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def save_json_once(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        raise FileExistsError(f"Will not overwrite frozen artifact: {path}")
    temporary = path.with_suffix(path.suffix + ".partial")
    if temporary.exists():
        raise RuntimeError(f"Stale incomplete artifact, inspect first: {temporary}")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False) + "\n",
                         encoding="utf-8")
    temporary.rename(path)

def record_environment():
    try:
        git_rev = subprocess.run(["git", "rev-parse", "HEAD"], cwd=repo_root,
                                text=True, check=True, capture_output=True).stdout.strip()
    except (OSError, subprocess.CalledProcessError):
        git_rev = "unknown (archive export)"
    return {"utc":datetime.now(timezone.utc).isoformat(),
            "python":sys.version, "torch":torch.__version__,
            "cuda_runtime":torch.version.cuda,
            "gpu":torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
            "git_revision":git_rev,
            "sft_sha256":sha256(SFT), "tokenizer_meta_sha256":sha256(META)}

env_record=record_environment()
print(json.dumps(env_record, indent=2), flush=True)
if FAST_SMOKE:
    print("SMOKE mode: never export its model as formal dpo.pt")
elif device != "cuda":
    raise RuntimeError("Full 13-epoch reproduction requires CUDA. "
                       "Use a Slurm GPU/Jupyter allocation or set FAST_SMOKE=True.")

### Step 3: Define helper functions

The instructor's `compute_logprob` and `get_batches` semantics are preserved. Chosen and Rejected are teacher-forced character sequences, with normalized full-sequence log probabilities. The final model uses the original course loss, not the optional answer-focused ablation.

In [ ]:
# Step 3 — Original teacher-forced token log-probabilities and batch helpers
from generate_round1 import parse_relation, stable_split

def encode(s):
    return [stoi[ch] for ch in s]

def pad_or_truncate(chars, max_length):
    return chars[-max_length:] if len(chars)>max_length else chars+[0]*(max_length-len(chars))

def compute_logprob(gpt, batch_ids):
    inputs = batch_ids[:, :-1]
    targets = batch_ids[:, 1:]
    logits, _ = gpt(inputs, full_seq=True)
    loss = F.cross_entropy(logits.reshape(-1,logits.size(-1)),
                           targets.reshape(-1), ignore_index=0,
                           reduction="none").reshape_as(targets)
    mask = (targets!=0).float()
    return -(loss*mask).sum(dim=1)/mask.sum(dim=1)

def course_preference_loss(gpt, negatives, positives):
    neg_lp = compute_logprob(gpt, negatives)
    pos_lp = compute_logprob(gpt, positives)
    return -F.logsigmoid((pos_lp-neg_lp)/BETA).mean()-0.1*pos_lp.mean()

def get_batches(lines, batch_size, max_length, target_device):
    # EXACT instructor rule: shuffle rows in place, discard trailing <batch.
    random.shuffle(lines)
    for i in range(0,len(lines),batch_size):
        batch=lines[i:i+batch_size]
        if len(batch)<batch_size:continue
        neg=[pad_or_truncate(encode(x["negative"]+"\n\n\n\n"),max_length) for x in batch]
        pos=[pad_or_truncate(encode(x["positive"]+"\n\n\n\n"),max_length) for x in batch]
        yield (torch.tensor(neg,dtype=torch.long,device=target_device),
               torch.tensor(pos,dtype=torch.long,device=target_device))

def verify_pairs(path, expected_count=100000, smoke=False):
    source = read_json(path)
    if len(source)!=expected_count:
        raise RuntimeError(f"{path}: expected {expected_count}, found {len(source)}")
    rows = source[:128] if smoke else source
    max_required = TRAIN_BASE_LENGTH
    unique_prompts=set()
    unique_facts=set()
    for i,row in enumerate(rows):
        pos,neg=row["positive"],row["negative"]
        marker=" The answer is "
        if marker not in pos or " because " not in pos:
            raise ValueError(f"{path.name} row {i}: incorrect chosen format")
        question,answer_suffix=pos.split(marker,1)
        answer=answer_suffix.split(" because ",1)[0]
        if not re.fullmatch(r"[+-]?\d+",answer) or not pos.endswith("."):
            raise ValueError(f"{path.name} row {i}: malformed integer chosen")
        if neg!=question+" Sorry, I don't know.":
            raise ValueError(f"{path.name} row {i}: changed rejection/prompt")
        if set(pos+neg)-set(stoi):
            raise ValueError(f"{path.name} row {i}: tokenizer OOV")
        max_required=max(max_required,len(pos)+4,len(neg)+4)
        unique_prompts.add(question)
        unique_facts.add(parse_relation(question,answer))
    # Original NanoGPT block size=256; 64 was only the initial pack length.
    if max_required>256:
        raise ValueError(f"{path.name}: {max_required} exceeds model block_size")
    info={"source":str(path.relative_to(repo_root)),"sha256":sha256(path),
          "rows":len(rows),"unique_prompts":len(unique_prompts),
          "unique_facts":len(unique_facts),"max_length":max_required}
    return rows,info

def model_from_checkpoint(path):
    obj=torch.load(path,map_location="cpu",weights_only=True)
    args=obj["model_args"]
    model=GPT(GPTConfig(**args))
    state=obj.get("model_state_dict",obj.get("model"))
    if state is None:raise RuntimeError(f"{path}: missing state dict")
    model.load_state_dict({k.removeprefix("_orig_mod."):v for k,v in state.items()},strict=True)
    return model,args

# Sanity checks: answer-first comparison, no generated completion needed in training.
assert encode("3*17=?")
assert abs(float(F.logsigmoid(torch.tensor(0.0)))+math.log(2))<1e-6
print("PASS: course loss, tokenizer, model loading and batch conventions",flush=True)

### Step 4: Load the pretrained NanoGPT model

In [ ]:
# Step 4 — Load and check the course-provided pretrained NanoGPT (teacher template).
# The later historical stages reload this SAME SFT checkpoint with a fresh optimizer.
ckpt = torch.load(SFT, map_location="cpu", weights_only=True)
gptconf = GPTConfig(**ckpt["model_args"])
gpt = GPT(gptconf)
state_dict = ckpt["model"]
unwanted_prefix = "_orig_mod."
state_dict = {key.removeprefix(unwanted_prefix):value for key,value in state_dict.items()}
gpt.load_state_dict(state_dict, strict=True)
sft_args = dict(ckpt["model_args"])
assert (sft_args["n_layer"],sft_args["n_head"],sft_args["n_embd"],
        sft_args["vocab_size"],sft_args["block_size"]) == (6,6,348,74,256)
print("Step 4 PASS: original pretrained GPTConfig",sft_args)
# Avoid two simultaneous GPU models during the multi-stage training.
del gpt, ckpt, state_dict

### Step 5: Load Data (**students are required to complete this part!**)

#### Step 5A — Data synthesis source code (included for submission)

The **12 collapsed source-code cells** below contain exact source snapshots of our original preference-data generators and validation helpers. They are part of this single Notebook, so the synthesis implementation is included with the assignment. The next cell verifies all snapshots against matching `scripts/` source files (never silently overwrites changed code).

Our datasets use 12 arithmetic/equation categories, math-verified answer-first Chosen completions, `Sorry, I don't know.` Rejected completions, and canonical fact-level Train/Val/Test isolation. Repeated historical exposures are retained and counted; they are not described as all-unique training facts.

In [ ]:
BUNDLED_GENERATOR_SOURCE = {}


In [ ]:
# Exact upstream source: scripts/generate_round1.py
BUNDLED_GENERATOR_SOURCE["scripts/generate_round1.py"] = r'''#!/usr/bin/env python3
"""Reproducible, exact-math SC3000 NanoGPT-Math Round 1 preference dataset.

Uses Python standard library only. Generates 12 math templates, exact finite
quarters/halves, strong math-fact-based split isolation and a frozen test set.

From repository root:
    python scripts/generate_round1.py --profile mini
    python scripts/generate_round1.py --profile full
    python scripts/generate_round1.py --profile full --install-train

Without --install-train, NEVER replaces dpo/pos_neg_pairs.json.
The existing instructor demo is backed up on first installation.
"""
from __future__ import annotations

import argparse
from collections import Counter
from fractions import Fraction
import hashlib
import json
from pathlib import Path
import pickle
import random
import re
import shutil

REPO = Path(__file__).resolve().parents[1]
KINDS = ("add", "subtract", "multiply", "divide",
         "x_plus_a", "a_plus_x", "x_minus_a", "a_minus_x",
         "x_times_a", "a_times_x", "x_over_a", "a_over_x")
OPERATORS = {"add": "+", "subtract": "-", "multiply": "*", "divide": "/",
             "x_plus_a": "+", "a_plus_x": "+", "x_minus_a": "-", "a_minus_x": "-",
             "x_times_a": "*", "a_times_x": "*", "x_over_a": "/", "a_over_x": "/"}
NUM = r"[+-]?\d+(?:\.\d+)?"
ARITH = re.compile(rf"^({NUM})([+*/-])({NUM})=\?$")
EQUATION = re.compile(rf"^({NUM}|x)([+*/-])({NUM}|x)=({NUM}),x=\?$")
REFUSAL = "Sorry, I don't know."
DEFAULT_CHARS = set("\n '*+,-./=?’0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz")
SEED = 42
# Reserve course demos, Notebook Step 8, and previous independent smoke probes.
RESERVED = {
    "79-7=?": "72", "74+8=?": "82", "x+55=95,x=?": "40", "1*x=6,x=?": "6",
    "17+19=?": "36", "3*17=?": "51", "72/4=?": "18", "72-x=34,x=?": "38",
    "x*11=44,x=?": "4", "7/2=?": "3.5", "5-9=?": "-4",
}


def canonical_number(v: Fraction) -> str:
    """Signed integer / exact terminating decimal with up to 2 decimal places."""
    v = Fraction(v)
    if v.denominator == 1:
        return str(v.numerator)
    n = v * 100
    if n.denominator != 1:
        raise ValueError(f"Not expressible in <=2 decimal places: {v}")
    n = int(n)
    sign = "-" if n < 0 else ""
    whole, rest = divmod(abs(n), 100)
    return (f"{sign}{whole}.{rest:02d}").rstrip("0").rstrip(".")


def signed_int(rng: random.Random, lo: int, hi: int, negative_chance: float = 0) -> Fraction:
    n = rng.randint(lo, hi)
    return Fraction(-n if rng.random() < negative_chance else n)


def fractional(rng: random.Random, lo: int = 1, hi: int = 120, negative_chance: float = 0) -> Fraction:
    # Only .25, .5, .75; no float artifacts, no repeating decimals.
    n = rng.randint(lo * 4, hi * 4)
    while n % 4 == 0:
        n = rng.randint(lo * 4, hi * 4)
    return Fraction(-n if rng.random() < negative_chance else n, 4)


def problem(kind: str, decimal: bool, rng: random.Random) -> tuple[str, str]:
    """Construct true relations backwards; independently verified later."""
    a = signed_int(rng, 1, 100)
    b = signed_int(rng, 1, 100)
    x = fractional(rng, negative_chance=0.2) if decimal else signed_int(rng, 1, 100, .2)
    small = rng.randint(1, 30)
    if kind == "add":
        l, r = (x, b) if decimal else (a, b)
        q, ans = f"{canonical_number(l)}+{canonical_number(r)}=?", l + r
    elif kind == "subtract":
        l, r = (x, b) if decimal else (a, b)
        q, ans = f"{canonical_number(l)}-{canonical_number(r)}=?", l - r
    elif kind == "multiply":
        l, r = (x, Fraction(small)) if decimal else (a, Fraction(small))
        q, ans = f"{canonical_number(l)}*{canonical_number(r)}=?", l * r
    elif kind == "divide":
        if decimal:
            quotient = fractional(rng, lo=1, hi=100)
            divisor = Fraction(4 * rng.randint(1, 10))
            numerator = quotient * divisor
        else:
            quotient = signed_int(rng, 1, 100, .15)
            divisor = Fraction(small)
            numerator = quotient * divisor
        q, ans = f"{canonical_number(numerator)}/{canonical_number(divisor)}=?", quotient
    elif kind == "x_plus_a":
        q, ans = f"x+{canonical_number(a)}={canonical_number(x+a)},x=?", x
    elif kind == "a_plus_x":
        q, ans = f"{canonical_number(a)}+x={canonical_number(a+x)},x=?", x
    elif kind == "x_minus_a":
        q, ans = f"x-{canonical_number(a)}={canonical_number(x-a)},x=?", x
    elif kind == "a_minus_x":
        q, ans = f"{canonical_number(a)}-x={canonical_number(a-x)},x=?", x
    elif kind == "x_times_a":
        q, ans = f"x*{small}={canonical_number(x*small)},x=?", x
    elif kind == "a_times_x":
        q, ans = f"{small}*x={canonical_number(x*small)},x=?", x
    elif kind == "x_over_a":
        divisor = Fraction(rng.choice((2, 4, 5, 10))) if decimal else Fraction(small)
        quotient = fractional(rng, lo=1, hi=100, negative_chance=0.2) if decimal else signed_int(rng, 1, 100, .2)
        x = quotient * divisor
        q, ans = f"x/{canonical_number(divisor)}={canonical_number(quotient)},x=?", x
    elif kind == "a_over_x":
        quotient = signed_int(rng, 1, 30, .2)
        x = fractional(rng, lo=1, hi=100, negative_chance=0.2) if decimal else signed_int(rng, 1, 100, .2)
        numerator = x * quotient
        q, ans = f"{canonical_number(numerator)}/x={canonical_number(quotient)},x=?", x
    else:
        raise ValueError(kind)
    answer = canonical_number(ans)
    if decimal and "." not in (q + answer):
        raise AssertionError(f"Expected decimal case: {q} -> {answer}")
    if not decimal and "." in (q + answer):
        raise AssertionError(f"Integer-only case unexpectedly decimal: {q} -> {answer}")
    return q, answer


def calculate(left: Fraction, op: str, right: Fraction) -> Fraction:
    if op == "+": return left + right
    if op == "-": return left - right
    if op == "*": return left * right
    if op == "/" and right != 0: return left / right
    raise ValueError("Division by zero")


def parse_relation(prompt: str, answer: str):
    """Parse and verify a displayed problem using substitution, not generator logic."""
    answer_f = Fraction(answer)
    match = EQUATION.fullmatch(prompt)
    if match:
        l, op, r, rhs = match.groups()
        if (l == "x") == (r == "x"):
            raise ValueError(f"Must contain exactly one x: {prompt}")
        lv = answer_f if l == "x" else Fraction(l)
        rv = answer_f if r == "x" else Fraction(r)
        cv = Fraction(rhs)
        if calculate(lv, op, rv) != cv:
            raise ValueError(f"Wrong answer: {prompt} -> {answer}")
    else:
        match = ARITH.fullmatch(prompt)
        if not match:
            raise ValueError(f"Invalid math prompt: {prompt}")
        l, op, r = match.groups()
        lv, rv, cv = Fraction(l), Fraction(r), answer_f
        if calculate(lv, op, rv) != cv:
            raise ValueError(f"Wrong answer: {prompt} -> {answer}")

    # Group equivalent arithmetic identities across question templates and rearrangements.
    if op == "+":
        family, left, right, result = "add", lv, rv, cv
    elif op == "-":
        family, left, right, result = "add", rv, cv, lv
    elif op == "*":
        family, left, right, result = "mul", lv, rv, cv
    else:
        family, left, right, result = "mul", rv, cv, lv
    if family == "mul" and left == 0 and right != 0:
        # Zero relations are degenerate for inversion-based group matching.
        pass
    x, y = sorted((left, right))
    return (family, str(x), str(y), str(result))


def stable_split(group_key, seed: int) -> str:
    encoded = json.dumps([seed, *group_key], separators=(",", ":"), ensure_ascii=True).encode()
    point = int.from_bytes(hashlib.sha256(encoded).digest()[:8], "big") % 10000
    return "train" if point < 8000 else "val" if point < 9000 else "test"


def quota(profile: str):
    if profile == "full":
        return {
            "train": {kind: (2560 if i < 4 else 1280) for i, kind in enumerate(KINDS)},
            "val": {kind: (256 if i < 4 else 128) for i, kind in enumerate(KINDS)},
            "test": {kind: (256 if i < 4 else 128) for i, kind in enumerate(KINDS)},
        }
    if profile == "mini":
        return {
            "train": {kind: (16 if i < 4 else 8) for i, kind in enumerate(KINDS)},
            "val": {kind: (4 if i < 4 else 2) for i, kind in enumerate(KINDS)},
            "test": {kind: (4 if i < 4 else 2) for i, kind in enumerate(KINDS)},
        }
    raise ValueError(profile)


def accepted_records(profile: str, seed: int):
    rng = random.Random(seed)
    targets = quota(profile)
    by_split = {split: [] for split in targets}
    seen_questions = set()
    reserved_keys = {parse_relation(p, answer) for p, answer in RESERVED.items()}
    seen_keys_by_split = {split: set() for split in targets}
    attempted = Counter()
    for split in ("train", "val", "test"):
        for kind in KINDS:
            target = targets[split][kind]
            decimal_target = round(target * .15)
            for decimal, count in ((False, target - decimal_target), (True, decimal_target)):
                for _ in range(count):
                    for tries in range(6000):
                        attempted[split] += 1
                        q, ans = problem(kind, decimal, rng)
                        if q in RESERVED or q in seen_questions:
                            continue
                        key = parse_relation(q, ans)
                        if key in reserved_keys or stable_split(key, seed) != split:
                            continue
                        seen_questions.add(q)
                        seen_keys_by_split[split].add(key)
                        by_split[split].append({
                            "prompt": q, "answer": ans, "category": kind,
                            "number_type": "decimal" if decimal else "integer",
                            "fact_key": key,
                        })
                        break
                    else:
                        raise RuntimeError(f"Could not fill {split}:{kind}:decimal={decimal}; expand candidate ranges")
        rng.shuffle(by_split[split])
    groups = list(seen_keys_by_split.values())
    assert not (groups[0] & groups[1]) and not (groups[0] & groups[2]) and not (groups[1] & groups[2])
    return by_split, attempted


def validate_all(by_split, stoi: set[str] | None = None):
    alphabet = stoi if stoi is not None else DEFAULT_CHARS
    prompts = set()
    keys_by_split = {}
    summary = {}
    for split, records in by_split.items():
        keys_by_split[split] = set()
        cats = Counter()
        types = Counter()
        max_pair_length = 0
        for item in records:
            p, a = item["prompt"], item["answer"]
            key = parse_relation(p, a)
            assert tuple(item["fact_key"]) == key
            assert key not in {parse_relation(r, v) for r, v in RESERVED.items()}
            assert p not in prompts
            assert len(p) < 256
            prompts.add(p)
            keys_by_split[split].add(key)
            assert canonical_number(Fraction(a)) == a
            pos = f"{p} The answer is {a}."
            neg = f"{p} {REFUSAL}"
            assert pos != neg
            assert set(pos + neg).issubset(alphabet), (p, sorted(set(pos + neg) - alphabet))
            assert max(len(pos), len(neg)) + 4 <= 256
            max_pair_length = max(max_pair_length, len(pos), len(neg))
            cats[item["category"]] += 1
            types[item["number_type"]] += 1
        summary[split] = {"total": len(records), "categories": dict(cats),
                          "number_types": dict(types), "max_pair_length": max_pair_length,
                          "unique_facts": len(keys_by_split[split])}
    assert not(keys_by_split["train"] & keys_by_split["val"])
    assert not(keys_by_split["train"] & keys_by_split["test"])
    assert not(keys_by_split["val"] & keys_by_split["test"])
    return summary


def write_json(path: Path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


def generate(profile: str, seed: int, out_dir: Path, stoi: set[str] | None):
    by_split, attempts = accepted_records(profile, seed)
    summary = validate_all(by_split, stoi=stoi)
    out_dir.mkdir(parents=True, exist_ok=True)
    paths = {}
    for split, records in by_split.items():
        if split == "train":
            path = out_dir / "pos_neg_pairs.json"
            data = [{"positive": f"{r['prompt']} The answer is {r['answer']}.",
                     "negative": f"{r['prompt']} {REFUSAL}"} for r in records]
        else:
            path = out_dir / f"round1_{split}.json"
            data = [{"prompt": r["prompt"], "answer": r["answer"],
                     "category": r["category"], "number_type": r["number_type"]} for r in records]
        write_json(path, data)
        paths[split] = path
    write_json(out_dir / "round1_train_metadata.json",
               [{"prompt": r["prompt"], "answer": r["answer"],
                 "category": r["category"], "number_type": r["number_type"],
                 "fact_key": list(r["fact_key"])} for r in by_split["train"]])
    hashes = {name: hashlib.sha256(p.read_bytes()).hexdigest() for name, p in paths.items()}
    manifest = {"version": "round1-refusal-v1", "seed": seed, "profile": profile,
                "positive_template": "{prompt} The answer is {answer}.",
                "negative_template": "{prompt} Sorry, I don't know.",
                "sampling": "Exact Fraction; 12 kinds; ~15% decimal; fact-group hash split; no leakage",
                "hash_rule": "sha256(json.dumps([seed, *fact_key], separators=(',', ':'))) first8bytes modulo 10000; train 0-7999 val 8000-8999 test 9000-9999",
                "data": summary, "attempts": dict(attempts), "file_sha256": hashes,
                "reserved_prompts": sorted(RESERVED),
                "test_policy": "Use validation for tuning; do not train on validation/test, including model-mined errors."}
    write_json(out_dir / "round1_manifest.json", manifest)
    return manifest, paths


def install(train_path: Path, repo: Path):
    target = repo / "dpo" / "pos_neg_pairs.json"
    backup = repo / "dpo" / "data" / "course_examples.json"
    existing = json.loads(target.read_text(encoding="utf-8")) if target.exists() else None
    if existing is not None and len(existing) == 4 and not backup.exists():
        backup.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(target, backup)
    elif existing is not None and len(existing) not in (4, 20480, 128):
        raise RuntimeError("Refusing to overwrite a non-demo, non-Round-1 training dataset")
    shutil.copyfile(train_path, target)
    print(f"Installed Round 1 training JSON into {target} (previous 4-demo backed up if applicable)")


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--profile", choices=("mini", "full"), default="full")
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--out-dir", type=Path, help="Defaults to dpo/data/round1_{profile}")
    parser.add_argument("--install-train", action="store_true",
                        help="Install the training data into dpo/pos_neg_pairs.json after validation")
    args = parser.parse_args()
    out = args.out_dir if args.out_dir else REPO / "dpo" / "data" / f"round1_{args.profile}"
    if args.install_train and args.profile != "full":
        parser.error("Only the full dataset may be installed into the formal training path")
    vocab = None
    meta = REPO / "sft" / "meta.pkl"
    if meta.is_file():
        with meta.open("rb") as f:
            vocab = set(pickle.load(f)["stoi"])
    manifest, paths = generate(args.profile, args.seed, out, vocab)
    if args.install_train:
        install(paths["train"], REPO)
    for split, item in manifest["data"].items():
        print(f"{split}: {item['total']:,} unique questions | {item['unique_facts']:,} unique math facts | "
              f"{item['number_types']} | longest pair {item['max_pair_length']} chars")
    print("PASS: exact mathematical verification, no duplicate questions, no fact-level split overlap")
    print("PASS: character vocabulary, 256-character context, correct positive/negative templates")
    print(f"Manifest: {out / 'round1_manifest.json'}")


if __name__ == "__main__":
    main()
'''


In [ ]:
# Exact upstream source: scripts/generate_round1b.py
BUNDLED_GENERATOR_SOURCE["scripts/generate_round1b.py"] = r'''#!/usr/bin/env python3
"""Create a controlled integer-only SC3000 Round 1B DPO dataset.

Reuses Round 1A's deterministic generator and exact mathematical validator.
Keeps ALL 17,408 Round 1A integer training examples and substitutes new
integer examples for the 3,072 decimal training examples, holding all other
experiment dimensions fixed. Does NOT replace dpo/pos_neg_pairs.json or dpo/dpo.pt.

A fixed, category-stratified set of 256 INTEGER validation questions is drawn
from the already-frozen Round 1A validation set (never from training).

Examples:
  python scripts/generate_round1b.py
  python scripts/generate_round1b.py --verify-only
"""
from __future__ import annotations

import argparse
from collections import Counter
from fractions import Fraction
import hashlib
import json
from pathlib import Path
import pickle
import random
import sys

ROOT = Path(__file__).resolve().parents[1]
SCRIPT_DIR = Path(__file__).resolve().parent
sys.path.insert(0, str(SCRIPT_DIR))
import generate_round1 as base  # noqa: E402

SEED = 42
EXTRA_SEED = 14042
VALIDATION_SEED = 24042
OUT = ROOT / 'dpo' / 'experiments' / 'round1b_integer'
OLD_VALIDATION = ROOT / 'dpo' / 'data' / 'round1_full' / 'round1_val.json'
OLD_MANIFEST = ROOT / 'dpo' / 'data' / 'round1_full' / 'round1_manifest.json'
KIND_TARGETS = {k: (2560 if i < 4 else 1280) for i, k in enumerate(base.KINDS)}
VAL_COUNTS = {k: (32 if i < 4 else 16) for i, k in enumerate(base.KINDS)}


def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def save_json(path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(rows, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')


def record_is_integer(row):
    return '.' not in (row['prompt'] + str(row['answer'])) and Fraction(row['answer']).denominator == 1


def make_dataset():
    original_manifest = json.loads(OLD_MANIFEST.read_text(encoding='utf-8'))
    assert original_manifest['seed'] == SEED and original_manifest['profile'] == 'full'
    val_rows = json.loads(OLD_VALIDATION.read_text(encoding='utf-8'))
    if sha256(OLD_VALIDATION) != original_manifest['file_sha256']['val']:
        raise RuntimeError('Frozen Round 1A validation JSON changed; aborting')

    source, attempts = base.accepted_records('full', SEED)
    original_integer = [r for r in source['train'] if r['number_type'] == 'integer']
    original_all_questions = {r['prompt'] for split in source for r in source[split]}
    old_train_fact = {tuple(r['fact_key']) for r in source['train']}
    other_fact = {tuple(r['fact_key']) for split in ('val', 'test') for r in source[split]}
    reserved_fact = {base.parse_relation(q, answer) for q, answer in base.RESERVED.items()}
    assert not (old_train_fact & other_fact)
    assert len(original_integer) == 17408

    selected = list(original_integer)
    rng = random.Random(EXTRA_SEED)
    counts = Counter(r['category'] for r in selected)
    seen_questions = set(original_all_questions)
    for kind in base.KINDS:
        missing = KIND_TARGETS[kind] - counts[kind]
        assert missing > 0
        for _ in range(missing):
            for attempt in range(20000):
                if kind == 'multiply':
                    # Round 1A already occupies most (1..100)*(1..30) train
                    # candidates. Add the same single-step multiplication
                    # template with a modestly extended 31..50 multiplier.
                    left = rng.randint(1, 100)
                    if rng.random() < 0.05:
                        left = -left
                    right = rng.randint(31, 50)
                    question, answer = f'{left}*{right}=?', str(left * right)
                else:
                    question, answer = base.problem(kind, decimal=False, rng=rng)
                if question in seen_questions or question in base.RESERVED:
                    continue
                fact_key = base.parse_relation(question, answer)
                if fact_key in reserved_fact or fact_key in other_fact:
                    continue
                if base.stable_split(fact_key, SEED) != 'train':
                    continue
                selected.append({'prompt': question, 'answer': answer,
                                 'category': kind, 'number_type': 'integer',
                                 'fact_key': fact_key})
                seen_questions.add(question)
                counts[kind] += 1
                break
            else:
                raise RuntimeError(f'Unable to generate unique {kind} integer problem')
    random.Random(34042).shuffle(selected)
    assert len(selected) == 20480
    assert all(counts[k] == KIND_TARGETS[k] for k in base.KINDS)
    assert len({r['prompt'] for r in selected}) == len(selected)
    train_facts = {tuple(r['fact_key']) for r in selected}
    assert not (train_facts & other_fact)
    assert all(base.parse_relation(r['prompt'], r['answer']) == tuple(r['fact_key']) for r in selected)
    assert all(record_is_integer(r) for r in selected)

    valid_old = [r for r in val_rows if record_is_integer(r)]
    assert len(valid_old) == 1744, f'Unexpected old validation integer size: {len(valid_old)}'
    by_kind = {kind: [] for kind in base.KINDS}
    for row in valid_old:
        by_kind[row['category']].append(row)
    val_rng = random.Random(VALIDATION_SEED)
    selected_val = []
    for kind in base.KINDS:
        category_rows = by_kind[kind]
        assert len(category_rows) >= VAL_COUNTS[kind]
        selected_val.extend(val_rng.sample(category_rows, VAL_COUNTS[kind]))
    val_rng.shuffle(selected_val)
    assert len(selected_val) == 256 and len({r['prompt'] for r in selected_val}) == 256
    assert all(base.parse_relation(r['prompt'], str(r['answer'])) not in train_facts for r in selected_val)
    assert all(record_is_integer(r) for r in selected_val)

    meta_path = ROOT / 'sft' / 'meta.pkl'
    stoi = None
    if meta_path.is_file():
        with meta_path.open('rb') as f:
            stoi = set(pickle.load(f)['stoi'])
    base.validate_all({'train': selected, 'val': [], 'test': []}, stoi=stoi)
    for row in selected_val:
        base.parse_relation(row['prompt'], str(row['answer']))

    pairs = [{'positive': f"{r['prompt']} The answer is {r['answer']}.",
              'negative': f"{r['prompt']} {base.REFUSAL}"} for r in selected]
    train_path = OUT / 'pos_neg_pairs.json'
    val_path = OUT / 'common_integer_val_256.json'
    meta_out = OUT / 'round1b_train_metadata.json'
    save_json(train_path, pairs)
    save_json(val_path, selected_val)
    save_json(meta_out, [{k: list(v) if k == 'fact_key' else v for k,v in r.items()}
                         for r in selected])
    manifest = {
        'stage': 'round1b-integer-only', 'seed': SEED, 'extra_seed': EXTRA_SEED,
        'validation_seed': VALIDATION_SEED,
        'training_size': len(pairs), 'validation_size': len(selected_val),
        'original_train_integer_retained': len(original_integer),
        'new_integer_replacements': len(selected)-len(original_integer),
        'train_by_category': {k:counts[k] for k in base.KINDS},
        'validation_by_category': {k:VAL_COUNTS[k] for k in base.KINDS},
        'positive_format':'{prompt} The answer is {answer}.',
        'negative_format':"{prompt} Sorry, I don't know.",
        'original_validation_sha256': sha256(OLD_VALIDATION),
        'original_round1a_train_sha256': original_manifest['file_sha256']['train'],
        'file_sha256': {'train':sha256(train_path),'val':sha256(val_path),'metadata':sha256(meta_out)},
        'notes': 'Integer-only ablation; same 12 categories, 5 epochs, 64 batch, 1,600 steps. Original Round 1A files unchanged. Validation is a frozen integer-only subset of Round 1A validation.'
    }
    save_json(OUT / 'round1b_manifest.json', manifest)
    print('=== ROUND 1B DATASET PASSED ===')
    print('Train:',len(pairs),'| Retained original integer:',len(original_integer),
          '| New integer:',len(selected)-len(original_integer))
    print('Validation (frozen common):',len(selected_val))
    print('Train SHA256:',manifest['file_sha256']['train'])
    print('Output:',OUT)
    return manifest


def verify_only():
    manifest = json.loads((OUT / 'round1b_manifest.json').read_text(encoding='utf-8'))
    for key, path in [('train', OUT / 'pos_neg_pairs.json'),
                      ('val', OUT / 'common_integer_val_256.json'),
                      ('metadata', OUT / 'round1b_train_metadata.json')]:
        assert sha256(path)==manifest['file_sha256'][key], f'SHA mismatch: {key}'
    rows = json.loads((OUT / 'round1b_train_metadata.json').read_text())
    val = json.loads((OUT / 'common_integer_val_256.json').read_text())
    base.validate_all({'train': rows, 'val': [], 'test': []})
    assert all(record_is_integer(r) for r in rows+val)
    train_keys = {base.parse_relation(r['prompt'],r['answer']) for r in rows}
    assert all(base.parse_relation(r['prompt'],r['answer']) not in train_keys for r in val)
    print('PASS: integer-only, exact math, frozen val separation and all hashes')


def main():
    parser=argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--verify-only',action='store_true')
    args=parser.parse_args()
    if args.verify_only:
        verify_only()
    else:
        make_dataset()


if __name__=='__main__':
    main()
'''


In [ ]:
# Exact upstream source: scripts/generate_round1c.py
BUNDLED_GENERATOR_SOURCE["scripts/generate_round1c.py"] = r'''#!/usr/bin/env python3
"""Round 1C: keep Round 1B integer prompts/positives, replace refusals with wrong answers.

All negatives have the same 'The answer is ... .' format, sign and string
length as the correct response. All mathematics is validated independently.
The original Round 1A and Round 1B training data/checkpoints are untouched.

From the repository root:
  python scripts/generate_round1c.py
  python scripts/generate_round1c.py --verify-only
"""
from __future__ import annotations

import argparse
from collections import Counter
import hashlib
import json
from pathlib import Path
import pickle
import random
import sys

ROOT = Path(__file__).resolve().parents[1]
SCRIPTS = Path(__file__).resolve().parent
sys.path.insert(0, str(SCRIPTS))
import generate_round1 as base  # noqa: E402

SEED = 31042
SOURCE = ROOT / "dpo/experiments/round1b_integer"
DEST = ROOT / "dpo/experiments/round1c_incorrect"
EXPECTED_SOURCE_SHA = "0c641927173c7d0e9d221050ded48d273eac9b25da2752b1c6f0569015512164"
PREFIX = " The answer is "


def sha256(p: Path) -> str:
    return hashlib.sha256(p.read_bytes()).hexdigest()


def write_json(p: Path, value) -> None:
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


def wrong_number(original: str, rng: random.Random) -> tuple[str, str]:
    """Generate a wrong integer with exactly the same sign and character length."""
    assert original and original.lstrip("-").isdigit(), original
    negative = original.startswith("-")
    digits = original[1:] if negative else original
    assert digits == "0" or digits[0] != "0", original
    assert not (negative and digits == "0")

    strategy = rng.choices(["digit_change", "off_by_one", "place_error"], [6, 3, 1])[0]
    n = int(digits)
    wrong: int | None = None
    if strategy != "digit_change":
        offset = 1 if strategy == "off_by_one" else 10 ** rng.randrange(1, len(digits)) if len(digits) > 1 else 1
        for candidate in rng.sample([n - offset, n + offset], 2):
            if candidate >= (1 if negative else 0) and len(str(candidate)) == len(digits) and candidate != n:
                wrong = candidate
                break
    if wrong is None:
        strategy = "digit_change"  # stable fallback if near-miss changes number of digits/sign
        idx = rng.randrange(len(digits))
        options = [str(d) for d in range(10)
                   if str(d) != digits[idx] and (idx != 0 or len(digits) == 1 or d != 0)
                   and (not negative or len(digits) != 1 or d != 0)]
        if not options:
            raise RuntimeError(f"No legal digit substitution for {original}")
        replacement = rng.choice(options)
        candidate = digits[:idx] + replacement + digits[idx + 1:]
        wrong = int(candidate)
    result = ("-" if negative else "") + str(wrong)
    assert result != original and len(result) == len(original)
    assert result.startswith("-") == negative
    return result, strategy


def source_files():
    manifest_file = SOURCE / "round1b_manifest.json"
    manifest = json.loads(manifest_file.read_text(encoding="utf-8"))
    paths = {
        "train": SOURCE / "pos_neg_pairs.json",
        "metadata": SOURCE / "round1b_train_metadata.json",
        "validation": SOURCE / "common_integer_val_256.json",
    }
    for key, f in paths.items():
        assert f.exists(), f"Missing Round 1B source: {f}"
        mkey = "val" if key == "validation" else key
        if sha256(f) != manifest["file_sha256"][mkey]:
            raise RuntimeError(f"Round 1B {key} changed since generation")
    if sha256(paths["train"]) != EXPECTED_SOURCE_SHA:
        raise RuntimeError("Round 1B training hash differs from previously audited source")
    return paths, manifest


def verify_records(pairs, meta, validation):
    if len(pairs) != len(meta) or len(pairs) != 20480:
        raise AssertionError("Wrong Round 1C size")
    if len(validation) != 256:
        raise AssertionError("Expected frozen common integer validation N=256")
    prompts, facts, kinds = set(), set(), Counter()
    max_len = 0
    with (ROOT / "sft/meta.pkl").open("rb") as f:
        stoi = pickle.load(f)["stoi"]
    for pair, r in zip(pairs, meta):
        p, answer, wrong = r["prompt"], r["correct_answer"], r["wrong_answer"]
        assert p not in prompts
        prompts.add(p)
        fact = base.parse_relation(p, answer)  # validates exact printed question and correct value
        facts.add(fact)
        assert p.endswith("=?"), p
        assert pair["positive"] == f"{p}{PREFIX}{answer}."
        assert pair["negative"] == f"{p}{PREFIX}{wrong}."
        assert len(pair["positive"]) == len(pair["negative"])  # avoids whole-sequence-length shortcut
        assert answer != wrong and answer.lstrip("-").isdigit() and wrong.lstrip("-").isdigit()
        assert answer.startswith("-") == wrong.startswith("-")
        assert len(answer) == len(wrong)
        try:
            base.parse_relation(p, wrong)
        except ValueError:
            pass
        else:
            raise AssertionError(f"Incorrect answer is unexpectedly mathematically valid: {p} -> {wrong}")
        for v in pair.values():
            assert not (set(v) - set(stoi)), f"Tokenizer OOV in {v!r}"
            assert len(v) + 4 <= 256
            max_len = max(max_len, len(v))
        kinds[r["category"]] += 1
    for r in validation:
        assert r["prompt"] not in prompts
        key = base.parse_relation(r["prompt"], str(r["answer"]))
        assert key not in facts, "Validation math-fact overlaps training"
        assert "." not in r["prompt"] + str(r["answer"])
    return {"max_pair_chars": max_len, "types": dict(sorted(kinds.items())),
            "unique_prompts": len(prompts), "unique_facts": len(facts)}


def create():
    paths, source_manifest = source_files()
    source_pairs = json.loads(paths["train"].read_text(encoding="utf-8"))
    source_meta = json.loads(paths["metadata"].read_text(encoding="utf-8"))
    frozen_val = json.loads(paths["validation"].read_text(encoding="utf-8"))
    rng = random.Random(SEED)
    pairs, meta = [], []
    for old, r in zip(source_pairs, source_meta):
        prompt, answer = r["prompt"], str(r["answer"])
        assert old["positive"] == f"{prompt}{PREFIX}{answer}."
        wrong, strategy = wrong_number(answer, rng)
        pairs.append({"positive": old["positive"], "negative": f"{prompt}{PREFIX}{wrong}."})
        meta.append({"prompt": prompt, "correct_answer": answer, "wrong_answer": wrong,
                     "category": r["category"], "strategy": strategy,
                     "fact_key": r["fact_key"]})
    summary = verify_records(pairs, meta, frozen_val)
    train_path = DEST / "pos_neg_pairs.json"
    meta_path = DEST / "round1c_train_metadata.json"
    val_path = DEST / "common_integer_val_256.json"
    write_json(train_path, pairs)
    write_json(meta_path, meta)
    write_json(val_path, frozen_val)
    manifest = {
        "stage": "round1c-incorrect-negative", "seed": SEED,
        "training_size": len(pairs), "validation_size": len(frozen_val),
        "init_checkpoint": "sft/gpt.pt",
        "source_round1b_train_sha256": sha256(paths["train"]),
        "source_round1b_validation_sha256": sha256(paths["validation"]),
        "source_round1b_manifest_sha256": sha256(SOURCE / "round1b_manifest.json"),
        "templates": {"positive": "{prompt} The answer is {correct}.",
                      "negative": "{prompt} The answer is {wrong}."},
        "rules": "Wrong integer; same sign and string length; exact math verifier; fixed seed; source prompts and positives unchanged.",
        "summary": summary,
        "strategies": dict(Counter(r["strategy"] for r in meta)),
        "file_sha256": {"train": sha256(train_path), "metadata": sha256(meta_path), "val": sha256(val_path)},
        "note": "Round 1C differs from Round 1B only in the negative completion. A negative that is a plausible wrong number is synthetic, not model-mined.",
    }
    write_json(DEST / "round1c_manifest.json", manifest)
    print("=== ROUND 1C DATASET PASSED ===")
    print("Training pairs:", len(pairs), "| Frozen common validation:", len(frozen_val))
    print("Same positive/prompts as Round 1B:", len(pairs))
    print("Wrong-answer strategies:", manifest["strategies"])
    print("Max pair length:", summary["max_pair_chars"])
    print("Train SHA256:", manifest["file_sha256"]["train"])
    print("Output:", DEST)


def verify_only():
    paths, _ = source_files()
    manifest = json.loads((DEST / "round1c_manifest.json").read_text(encoding="utf-8"))
    assert sha256(paths["train"]) == manifest["source_round1b_train_sha256"]
    assert sha256(paths["validation"]) == manifest["source_round1b_validation_sha256"]
    outputs = {"train": DEST / "pos_neg_pairs.json",
               "metadata": DEST / "round1c_train_metadata.json",
               "val": DEST / "common_integer_val_256.json"}
    for kind, path in outputs.items():
        assert sha256(path) == manifest["file_sha256"][kind], f"Hash mismatch: {kind}"
    meta = json.loads(outputs["metadata"].read_text(encoding="utf-8"))
    pairs = json.loads(outputs["train"].read_text(encoding="utf-8"))
    val = json.loads(outputs["val"].read_text(encoding="utf-8"))
    summary = verify_records(pairs, meta, val)
    assert summary == manifest["summary"]
    original = json.loads(paths["train"].read_text(encoding="utf-8"))
    assert all(a["positive"] == b["positive"] for a,b in zip(pairs,original))
    print("PASS: hashes, math correctness, integer-only, equal lengths, identical positives, frozen validation")


def main():
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument("--verify-only", action="store_true")
    args = p.parse_args()
    if args.verify_only:
        verify_only()
    else:
        create()


if __name__ == "__main__":
    main()
'''


In [ ]:
# Exact upstream source: scripts/generate_round3_structured.py
BUNDLED_GENERATOR_SOURCE["scripts/generate_round3_structured.py"] = r'''#!/usr/bin/env python3
"""Round 3: structured, integer-only, one-step-explanation SC3000 preference data.

Core design: dense 0..100 arithmetic facts and related equation templates,
exact arithmetic verification, stable *fact-group* isolation, target 100,000
training rows (deliberate repeat exposure; report actual unique prompts).
Refusal negatives and the instructor's original course loss are kept fixed.

No checkpoints or notebook files are changed. Never automatically install
the dataset into dpo/pos_neg_pairs.json.

Usage:
  python scripts/generate_round3_structured.py --profile mini
  python scripts/generate_round3_structured.py --profile full
  python scripts/generate_round3_structured.py --profile full --verify-only
  python scripts/generate_round3_structured.py --profile full --write-control
"""
from __future__ import annotations

import argparse
from collections import Counter, defaultdict
import hashlib
import json
from pathlib import Path
import pickle
import random
import sys

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "scripts"))
from generate_round1 import RESERVED, parse_relation, stable_split  # noqa: E402

SEED = 42
REFUSAL = "Sorry, I don't know."
KINDS = (
    "add", "subtract", "multiply", "divide",
    "x_plus_a", "a_plus_x", "x_minus_a", "a_minus_x",
    "x_times_a", "a_times_x", "x_over_a", "a_over_x",
)
TRAIN_QUOTA = {
    "add": 14000, "subtract": 10000, "multiply": 11000, "divide": 9000,
    "x_plus_a": 7000, "a_plus_x": 7000, "x_minus_a": 7000, "a_minus_x": 7000,
    "x_times_a": 7000, "a_times_x": 7000, "x_over_a": 7000, "a_over_x": 7000,
}
assert sum(TRAIN_QUOTA.values()) == 100000
DEFAULT_OUT = ROOT / "dpo/data/round3_structured_full"


def make_record(prompt: str, answer: int, explanation: tuple[int, str, int],
                category: str) -> dict:
    """Construct a correctly explained mathematical statement."""
    a, op, b = explanation
    expected = {"+": a + b, "-": a - b, "*": a * b}
    if op == "/":
        if b == 0 or a % b:
            raise ValueError(f"Non-exact explanation division: {a}/{b}")
        value = a // b
    else:
        value = expected[op]
    if value != answer:
        raise ValueError(f"Incorrect reasoning: {prompt}, explanation={explanation}, answer={answer}")
    fact_key = parse_relation(prompt, str(answer))
    explanation_text = f"{a}{op}{b} equals {answer}"
    pos = f"{prompt} The answer is {answer} because {explanation_text}."
    control = f"{prompt} The answer is {answer}."
    neg = f"{prompt} {REFUSAL}"
    if pos.count(".") != 1 or control.count(".") != 1 or neg.count(".") != 1:
        raise ValueError("Unexpected period in integer-only examples")
    if "?" not in prompt or not prompt.endswith("=?"):
        raise ValueError("Bad problem form")
    return {
        "prompt": prompt, "answer": str(answer), "category": category,
        "fact_key": fact_key,
        "positive": pos, "control_positive": control, "negative": neg,
    }


def sum_relations(a: int, b: int):
    """One a+b=c fact, six question families in both orientations."""
    c = a + b
    yield make_record(f"{a}+{b}=?", c, (a, "+", b), "add")
    yield make_record(f"{c}-{a}=?", b, (c, "-", a), "subtract")
    yield make_record(f"x+{a}={c},x=?", b, (c, "-", a), "x_plus_a")
    yield make_record(f"{a}+x={c},x=?", b, (c, "-", a), "a_plus_x")
    yield make_record(f"x-{a}={b},x=?", c, (b, "+", a), "x_minus_a")
    yield make_record(f"{c}-x={a},x=?", b, (c, "-", a), "a_minus_x")


def product_relations(a: int, b: int):
    """One a*b=c fact. Never create equations with nonunique solutions."""
    c = a * b
    yield make_record(f"{a}*{b}=?", c, (a, "*", b), "multiply")
    if a > 0:
        yield make_record(f"{c}/{a}=?", b, (c, "/", a), "divide")
        yield make_record(f"x*{a}={c},x=?", b, (c, "/", a), "x_times_a")
        yield make_record(f"{a}*x={c},x=?", b, (c, "/", a), "a_times_x")
        yield make_record(f"x/{a}={b},x=?", c, (b, "*", a), "x_over_a")
        # x is the denominator, must be nonzero.
        if b > 0:
            yield make_record(f"{c}/x={a},x=?", b, (c, "/", a), "a_over_x")


def curriculum_tag(record: dict) -> str:
    q = record["prompt"]
    ans = int(record["answer"])
    key = record["fact_key"]
    a, b = int(key[1]), int(key[2])
    if a in (0, 1) or b in (0, 1):
        return "identity_0_1"
    if a <= 10 and b <= 10:
        return "basic_facts"
    if a % 10 == 0 or b % 10 == 0:
        return "tens_and_round_numbers"
    if key[0] == "add" and (a % 10 + b % 10 >= 10):
        return "carry_borrow"
    if key[0] == "mul" and a <= 12 and b <= 12:
        return "small_times_table"
    if ans >= 100:
        return "larger_results"
    return "standard"


TAG_WEIGHT = {
    "identity_0_1": 4.0,
    "basic_facts": 3.0,
    "tens_and_round_numbers": 2.5,
    "carry_borrow": 2.0,
    "small_times_table": 3.0,
    "larger_results": 1.0,
    "standard": 1.0,
}


def existing_protected_keys():
    """Also protect the previous common val-256 when available on TC1."""
    keys = {parse_relation(p, ans) for p, ans in RESERVED.items()
            if "." not in p + ans}  # integer portion of course demos
    old = ROOT / "dpo/experiments/round1c_incorrect/common_integer_val_256.json"
    if old.is_file():
        for r in json.loads(old.read_text(encoding="utf-8")):
            keys.add(parse_relation(r["prompt"], str(r["answer"])))
    return keys


def build_pools(seed: int):
    held_out = existing_protected_keys()
    pools = {split: {kind: {} for kind in KINDS}
             for split in ("train", "val", "test")}
    candidates = Counter()
    for op in ("add", "mul"):
        if op == "add":
            # 0..100 densely, all unordered sum facts, then both directions.
            fact_pairs = ((a, b) for a in range(101) for b in range(a, 101))
            builder = sum_relations
        else:
            # Strictly limited products; zero identities include only
            # valid, uniquely solved equations.
            fact_pairs = ((a, b) for a in range(101) for b in range(max(1, a), 101)
                          if a * b <= 400 and (a > 0 or b > 0))
            builder = product_relations
        for a, b in fact_pairs:
            for left, right in ((a, b), (b, a)) if a != b else ((a, b),):
                for row in builder(left, right):
                    candidates["generated"] += 1
                    key = row["fact_key"]
                    if key in held_out:
                        continue
                    split = stable_split(key, seed)
                    # Deduplicate within each split and category, while allowing
                    # repeated exposure in the final 100k training rows.
                    pools[split][row["category"]].setdefault(row["prompt"], row)
    for split, categories in pools.items():
        for category, items in categories.items():
            if not items:
                raise RuntimeError(f"No examples in pool {split}/{category}")
    return pools, candidates, len(held_out)


def sample_rows(pools: dict, profile: str, seed: int):
    rng = random.Random(seed + 3000)
    if profile == "mini":
        train_quota = {k: 12 for k in KINDS}
        heldout_quota = {k: 4 for k in KINDS}
    else:
        train_quota = TRAIN_QUOTA
        heldout_quota = {k: 64 for k in KINDS}

    result = {}
    for split in ("train", "val", "test"):
        out = []
        for kind in KINDS:
            pool = list(pools[split][kind].values())
            rng.shuffle(pool)
            count = train_quota[kind] if split == "train" else heldout_quota[kind]
            if split == "train":
                # Unique items first, then weighted repeat exposure. Do not
                # misreport 100k rows as 100k unique math problems.
                if len(pool) >= count:
                    chosen = rng.sample(pool, count)
                else:
                    weights = [TAG_WEIGHT[curriculum_tag(row)] for row in pool]
                    chosen = pool + rng.choices(pool, weights=weights, k=count - len(pool))
                assert len(chosen) == count
            else:
                if count > len(pool):
                    raise ValueError(f"Only {len(pool)} unique {split} records for {kind}; requested {count}")
                chosen = rng.sample(pool, count)
            out.extend(chosen)
        rng.shuffle(out)
        result[split] = out
    return result


def verify_records(records: dict, seed: int, vocab: set[str] | None):
    # Called before writing ANY outputs.
    manifest_splits = {}
    seen_fact_keys = {split: set() for split in records}
    seen_prompts = {split: set() for split in records}
    for split, rows in records.items():
        max_len = 0
        cats, tags = Counter(), Counter()
        for row in rows:
            p, ans, pos, neg = row["prompt"], row["answer"], row["positive"], row["negative"]
            key = parse_relation(p, ans)
            if key != row["fact_key"] or stable_split(key, seed) != split:
                raise RuntimeError(f"Fact key/split mismatch: {p}")
            # Independently re-check the explained arithmetic expression
            # encoded in the Positive string, rather than trusting a metadata field.
            stem = f"{p} The answer is {ans} because "
            if not pos.startswith(stem) or not pos.endswith(f" equals {ans}."):
                raise RuntimeError("Bad Positive format: " + pos)
            expression = pos[len(stem):-(len(f" equals {ans}."))]
            import re
            m = re.fullmatch(r"(-?\d+)([+\-*/])(-?\d+)", expression)
            if not m:
                raise RuntimeError("Bad mathematical explanation: " + expression)
            a, op, b = int(m[1]), m[2], int(m[3])
            if op == "+":
                actual = a + b
            elif op == "-":
                actual = a - b
            elif op == "*":
                actual = a * b
            else:
                if b == 0 or a % b:
                    raise RuntimeError("Invalid division explanation")
                actual = a // b
            if actual != int(ans):
                raise RuntimeError(f"Incorrect explanation: {pos}")
            if neg != f"{p} {REFUSAL}" or row["control_positive"] != f"{p} The answer is {ans}.":
                raise RuntimeError("Pair template inconsistency")
            if vocab is not None:
                unknown = set(pos + neg + row["control_positive"]) - vocab
                if unknown:
                    raise RuntimeError(f"Tokenizer OOV: {sorted(unknown)} from {p}")
            longest = max(len(pos), len(neg), len(row["control_positive"])) + 4
            if longest > 256:
                raise RuntimeError(f"Context >256: {p}")
            max_len = max(max_len, longest)
            seen_prompts[split].add(p)
            seen_fact_keys[split].add(key)
            cats[row["category"]] += 1
            tags[curriculum_tag(row)] += 1
        manifest_splits[split] = {
            "rows": len(rows), "unique_prompts": len(seen_prompts[split]),
            "unique_math_facts": len(seen_fact_keys[split]),
            "by_category": dict(cats), "by_curriculum_tag": dict(tags),
            "max_required_sequence_length": max_len,
        }
    for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
        if seen_fact_keys[a] & seen_fact_keys[b] or seen_prompts[a] & seen_prompts[b]:
            raise RuntimeError(f"Train/validation/test math-fact leakage: {a} and {b}")
    return manifest_splits


def json_bytes(obj) -> bytes:
    return (json.dumps(obj, ensure_ascii=False, separators=(",", ":")) + "\n").encode("utf-8")


def write_new(path: Path, value) -> str:
    path.parent.mkdir(parents=True, exist_ok=True)
    data = json_bytes(value)
    with path.open("xb") as f:
        f.write(data)
    return hashlib.sha256(data).hexdigest()


def main():
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--profile", choices=("mini", "full"), default="mini")
    ap.add_argument("--seed", type=int, default=SEED)
    ap.add_argument("--out-dir", type=Path)
    ap.add_argument("--write-control", action="store_true",
                    help="Also write matching answer-only preferences for later ablation")
    ap.add_argument("--verify-only", action="store_true",
                    help="Build and validate in memory without writing outputs")
    args = ap.parse_args()
    if args.seed != SEED:
        ap.error("For controlled comparison keep seed=42")
    out = (args.out_dir if args.out_dir else
           ROOT / f"dpo/data/round3_structured_{args.profile}").resolve()
    if out.exists() and not args.verify_only:
        ap.error(f"Refusing to overwrite existing output: {out}")

    vocab_path = ROOT / "sft/meta.pkl"
    vocab = None
    if vocab_path.exists():
        with vocab_path.open("rb") as f:
            vocab = set(pickle.load(f)["stoi"])
    pools, attempted, protected_count = build_pools(args.seed)
    rows = sample_rows(pools, args.profile, args.seed)
    info = verify_records(rows, args.seed, vocab)
    print("=== ROUND 3 STRUCTURED MATH DATA VERIFIED ===")
    for split in ("train", "val", "test"):
        v = info[split]
        print(f"{split:5} rows={v['rows']:6} unique_prompts={v['unique_prompts']:6} "
              f"unique_facts={v['unique_math_facts']:5} "
              f"max_sequence={v['max_required_sequence_length']}")
    print(f"protected_existing_math_facts={protected_count}")
    if args.verify_only:
        print("PASS: verify-only; no data files written")
        return
    hashes = {}
    hashes["train"] = write_new(
        out / "pos_neg_pairs.json",
        [{"positive": r["positive"], "negative": r["negative"]} for r in rows["train"]])
    if args.write_control:
        hashes["answer_only_control"] = write_new(
            out / "pos_neg_pairs_answer_only.json",
            [{"positive": r["control_positive"], "negative": r["negative"]}
             for r in rows["train"]])
    hashes["train_metadata"] = write_new(out / "train_metadata.json",
        [{"prompt": r["prompt"], "answer": r["answer"], "category": r["category"],
          "fact_key": list(r["fact_key"]), "curriculum_tag": curriculum_tag(r)}
         for r in rows["train"]])
    for split in ("val", "test"):
        hashes[split] = write_new(out / f"{split}.json",
            [{"prompt": r["prompt"], "answer": r["answer"],
              "category": r["category"], "fact_key": list(r["fact_key"]),
              "curriculum_tag": curriculum_tag(r)} for r in rows[split]])
    rng = random.Random(SEED + 100)
    sample = rng.sample(rows["train"], min(256, len(rows["train"])))
    hashes["train_probe"] = write_new(out / "train_probe_256.json",
        [{"prompt": r["prompt"], "answer": r["answer"], "category": r["category"]}
         for r in sample])
    manifest = {
        "version": "round3-structured-because-v1",
        "status": "GENERATED_NOT_TRAINED",
        "seed": args.seed, "profile": args.profile,
        "sampling": ("All distinct train records sampled first where possible; "
                     "weighted repeat exposure for 0/1, 1-digit facts, round numbers "
                     "and carrying; exactly 100k train rows in full mode."),
        "split": ("Stable sha256 split by canonical addition/multiplication math fact; "
                  "80% train / 10% val / 10% test; same fact never crosses splits."),
        "note": ("Older common_integer_val_256 facts are excluded when that source "
                 "file is available locally during generation. They remain OOD "
                 "and are NOT a substitute for the new fixed validation set."),
        "positive": "{prompt} The answer is {answer} because {verified expression} equals {answer}.",
        "negative": f"{{prompt}} {REFUSAL}",
        "loss_plan": "Instructor's original course loss; start from sft/gpt.pt",
        "held_out_keys_protected": protected_count,
        "candidate_count_before_protection": attempted["generated"],
        "per_split": info,
        "sha256": hashes,
        "write_control": args.write_control,
    }
    write_new(out / "manifest.json", manifest)
    print(f"PASS: exact-answer and explanation checks, vocabulary, context, and fact-group splits")
    print(f"Dataset directory: {out}")
    print("Files never overwrite dpo/pos_neg_pairs.json or course checkpoints.")
    print("Next: run small GPU smoke and original course-loss training separately.")


if __name__ == "__main__":
    main()
'''


In [ ]:
# Exact upstream source: scripts/build_round4_data.py
BUNDLED_GENERATOR_SOURCE["scripts/build_round4_data.py"] = r'''#!/usr/bin/env python3
"""Round 4: matched continuation-vs-replay data from frozen Round 3 train facts.

Experiment design:
    Replay-only: 70k SAME old pairs + 30k other old pairs.
    Treatment:   70k SAME old pairs + 15k harder old train pairs +
                 15k NEW hard multiplication/inverse facts (products 401..999).
Both start from the exact same Round 3 checkpoint, with same training schedule.

Do not inspect or train on the sealed Round3 test set (its fact keys are used
only to guard against leakage). New examples require stable_split(key,42)=train
and must not share facts with Round3 val/test or old Round1C common val.

CPU only:
    python scripts/build_round4_data.py --profile mini --verify-only
    python scripts/build_round4_data.py --profile full --verify-only
    python scripts/build_round4_data.py --profile full
"""
from __future__ import annotations
import argparse
from collections import Counter, defaultdict
import hashlib
import json
from pathlib import Path
import pickle
import random
import sys

ROOT=Path(__file__).resolve().parents[1]
sys.path.insert(0,str(ROOT/"scripts"))
from generate_round1 import parse_relation, stable_split, RESERVED
from generate_round3_structured import product_relations

SOURCE=ROOT/"dpo/data/round3_structured_full"
OLD_VAL=ROOT/"dpo/experiments/round1c_incorrect/common_integer_val_256.json"
SEED=42
MULTIPLICATIVE=("multiply","divide","x_times_a","a_times_x","x_over_a","a_over_x")
FOCUS_QUOTA={"multiply":4000,"x_over_a":4000,"x_times_a":2000,
             "a_times_x":2000,"divide":1500,"a_over_x":1500}
EXTEND_QUOTA={c:2500 for c in MULTIPLICATIVE}
assert sum(FOCUS_QUOTA.values())==sum(EXTEND_QUOTA.values())==15000


def sha(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def read(path):
    return json.loads(path.read_text(encoding="utf-8"))


def refuse_existing(out):
    if out.exists():
        raise FileExistsError(f"Refusing to overwrite existing dataset directory: {out}")


def validate_sources():
    manifest=read(SOURCE/"manifest.json")
    pairs=read(SOURCE/"pos_neg_pairs.json")
    meta=read(SOURCE/"train_metadata.json")
    if len(pairs)!=100000 or len(meta)!=100000:
        raise RuntimeError("Unexpected Round3 source length")
    required={"train":"pos_neg_pairs.json","train_metadata":"train_metadata.json",
              "val":"val.json","test":"test.json","train_probe":"train_probe_256.json"}
    for k,file in required.items():
        if sha(SOURCE/file)!=manifest["sha256"][k]:
            raise RuntimeError(f"Round3 source hash mismatch: {file}")
    keys_val={parse_relation(r["prompt"],str(r["answer"])) for r in read(SOURCE/"val.json")}
    keys_test={parse_relation(r["prompt"],str(r["answer"])) for r in read(SOURCE/"test.json")}
    if keys_val & keys_test:
        raise RuntimeError("Round3 val/test leakage")
    protected=keys_val | keys_test
    if not OLD_VAL.is_file():
        raise RuntimeError("Old 256 OOD validation missing: cannot certify isolation")
    old=read(OLD_VAL)
    if len(old)!=256:raise RuntimeError("Old frozen validation is not 256")
    old_keys={parse_relation(r["prompt"],str(r["answer"])) for r in old}
    protected |= old_keys
    protected |= {parse_relation(q,a) for q,a in RESERVED.items() if "." not in q+a}
    for pair,m in zip(pairs,meta):
        if not pair["positive"].startswith(m["prompt"]+" The answer is "+str(m["answer"])+" because "):
            raise RuntimeError("Round3 pair/metadata misaligned")
        if parse_relation(m["prompt"],str(m["answer"]))!=tuple(m["fact_key"]):
            raise RuntimeError("Incorrect Round3 fact")
        if tuple(m["fact_key"]) in protected:
            raise RuntimeError("Round3 train already overlaps a protected fact")
    return pairs, meta, protected, {
        "round3_manifest_sha256":sha(SOURCE/"manifest.json"),
        "round3_train_sha256":sha(SOURCE/"pos_neg_pairs.json"),
        "round3_metadata_sha256":sha(SOURCE/"train_metadata.json"),
        "round3_val_sha256":sha(SOURCE/"val.json"),
        "round3_test_sha256":sha(SOURCE/"test.json"),
        "old_val_sha256":sha(OLD_VAL),
    }


def get_extension_pool(protected, split="train"):
    by_kind={k:{} for k in MULTIPLICATIVE}
    for a in range(2,101):
        for b in range(a,101):
            if not (401 <= a*b <= 999):continue
            for left,right in ((a,b),(b,a)) if a!=b else ((a,b),):
                for row in product_relations(left,right):
                    key=row["fact_key"]
                    if key in protected or stable_split(key,SEED)!=split:
                        continue
                    by_kind[row["category"]].setdefault(row["prompt"],row)
    return by_kind


def build(profile):
    old,meta,protected,source_hashes=validate_sources()
    rng=random.Random(42004)
    if profile=="full":
        replay_count=70000
        focus_quota=FOCUS_QUOTA
        extend_quota=EXTEND_QUOTA
    else:
        replay_count=448
        focus_quota={k:16 for k in MULTIPLICATIVE}
        extend_quota={k:16 for k in MULTIPLICATIVE}
    focus_n=sum(focus_quota.values())
    extend_n=sum(extend_quota.values())
    assert replay_count+focus_n+extend_n==(100000 if profile=="full" else 640)

    # SAME shared 70% in both arms (sample old source without replacement).
    order=list(range(len(old)))
    rng.shuffle(order)
    shared_idx=order[:replay_count]
    remaining_idx=order[replay_count:]
    # Replay-only control contains old pairs, with the same total sample count.
    control_idx=shared_idx + rng.sample(remaining_idx, focus_n+extend_n)
    treatment_pairs=[old[i] for i in shared_idx]
    focus_counts=Counter()
    for category,count in focus_quota.items():
        candidates=[]
        for i,m in enumerate(meta):
            if m["category"]!=category:continue
            f=tuple(m["fact_key"])
            if f[0]!="mul" or int(f[1])<=1 or int(f[2])<=1:
                continue
            ans=int(m["answer"])
            # Target numerically hard examples, including inverse operations.
            if category in ("multiply","x_over_a") and ans<100:
                continue
            if category in ("divide","x_times_a","a_times_x","a_over_x") and ans<10:
                continue
            candidates.append(i)
        if not candidates:
            raise RuntimeError(f"No harder in-domain samples for {category}")
        selected=rng.choices(candidates,k=count)
        treatment_pairs.extend(old[i] for i in selected)
        focus_counts[category]=count
    extension=get_extension_pool(protected, split="train")
    extension_val_pool=get_extension_pool(protected, split="val")
    extension_val=[]
    for category in MULTIPLICATIVE:
        candidates=list(extension_val_pool[category].values())
        if len(candidates)<64:
            raise RuntimeError(f"Not enough independent extended validation examples: {category}")
        extension_val.extend(rng.sample(candidates,64))
    rng.shuffle(extension_val)
    extension_rows=[]
    extension_stats={}
    for category,count in extend_quota.items():
        pool=list(extension[category].values())
        if len(pool)<count:
            # May repeat only when truly needed. Manifest reports uniqueness.
            chosen=pool+rng.choices(pool,k=count-len(pool))
        else:
            chosen=rng.sample(pool,count)
        if len(chosen)!=count:
            raise AssertionError("Extended example count mismatch")
        extension_rows.extend(chosen)
        extension_stats[category]={"total_rows":len(chosen),
                                   "unique_prompts":len({x["prompt"] for x in chosen}),
                                   "unique_math_facts":len({x["fact_key"] for x in chosen})}
    treatment_pairs.extend({"positive":r["positive"],"negative":r["negative"]}
                           for r in extension_rows)
    control_pairs=[old[i] for i in control_idx]
    rng.shuffle(control_pairs)
    rng.shuffle(treatment_pairs)
    if len(control_pairs)!=len(treatment_pairs):
        raise AssertionError("Control and treatment length mismatch")
    # Preserve the instructor's get_batches() behavior: any final <64 rows are
    # skipped after shuffling. Both arms have identical count and skipped rows.
    # Audit ALL newly created samples; neither arm can introduce fact leakage.
    vocab_path=ROOT/"sft/meta.pkl"
    with vocab_path.open("rb") as f:vocab=set(pickle.load(f)["stoi"])
    existing_train_facts={tuple(m["fact_key"]) for m in meta}
    extra_facts=set()
    max_len=0
    for r in extension_rows:
        k=parse_relation(r["prompt"],r["answer"])
        if k!=r["fact_key"] or k in protected or k in existing_train_facts:
            raise RuntimeError("New extension fact leaks into old train or val/test")
        if stable_split(k,SEED)!="train":
            raise RuntimeError("Wrong split for extension")
        if not (401<=int(k[3])<=999):
            raise RuntimeError(f"Unexpected product difficulty: {k}")
        if set(r["positive"]+r["negative"])-vocab:
            raise RuntimeError("Tokenizer OOV")
        if max(len(r["positive"]),len(r["negative"]))+4>256:
            raise RuntimeError("Context exceeded")
        extra_facts.add(k)
    for arm in (control_pairs,treatment_pairs):
        for p in arm:
            if set(p["positive"]+p["negative"])-vocab:
                raise RuntimeError("OOV in training pair")
            max_len=max(max_len,len(p["positive"])+4,len(p["negative"])+4)
            if max_len>256:raise RuntimeError("Context exceeded")
    info={
      "profile":profile,"seed":SEED,"source_sha256":source_hashes,
      "old_validation_protected_count":len(protected),
      "replay_shared_count":replay_count,"replay_only_extra_count":focus_n+extend_n,
      "treatment_in_domain_focus_count":focus_n,
      "treatment_extended_count":extend_n,
      "treatment_focus_categories":dict(focus_counts),
      "new_extension_unique_math_facts":len(extra_facts),
      "new_extension_unique_prompts":len({r["prompt"] for r in extension_rows}),
      "new_extension_by_category":extension_stats,
      "max_sequence_with_padding":max_len,
      "training_arm_size":len(control_pairs),
      "full_batches_per_epoch_batch64":len(control_pairs)//64,
      "discarded_incomplete_batch_per_epoch":len(control_pairs)%64,
      "training_rows_not_unique_prompts":True,
      "training_plan":"Round3 checkpoint; original Course Loss; 2 epochs, batch 64, lr=2e-5; replay-only control",
      "evaluation":"Existing Round3 768 Val + previous Round1C 256 OOD; never Round3 final Test",
      "note":"All 70% shared old examples are identical between arms. Treatment additionally reweights hard old train facts and adds product 401..999 facts; control replays older training examples only.",
    }
    info["extension_val_count"]=len(extension_val)
    info["extension_val_unique_math_facts"]=len({r["fact_key"] for r in extension_val})
    if {r["fact_key"] for r in extension_rows} & {r["fact_key"] for r in extension_val}:
        raise RuntimeError("Extended validation intersects training facts")
    for row in extension_val:
        if stable_split(row["fact_key"],SEED)!="val" or row["fact_key"] in protected:
            raise RuntimeError("Invalid extended validation assignment")
        if set(row["positive"]+row["negative"])-vocab:
            raise RuntimeError("Invalid extended validation vocabulary")
    return control_pairs,treatment_pairs,extension_val,info


def write_new(path,data):
    path.parent.mkdir(parents=True,exist_ok=True)
    blob=(json.dumps(data,ensure_ascii=False,separators=(",",":"))+"\n").encode("utf-8")
    with path.open("xb") as f:f.write(blob)
    return hashlib.sha256(blob).hexdigest()


def main():
    p=argparse.ArgumentParser(description=__doc__)
    p.add_argument("--profile",choices=("mini","full"),default="mini")
    p.add_argument("--verify-only",action="store_true")
    p.add_argument("--out-dir",type=Path)
    a=p.parse_args()
    out=(a.out_dir if a.out_dir else ROOT/f"dpo/data/round4_matched_{a.profile}").resolve()
    if out.exists() and not a.verify_only:refuse_existing(out)
    control,treatment,extension_val,info=build(a.profile)
    print("=== ROUND 4 MATCHED DATA CHECK ===")
    print(f"Replay-only={len(control)}, Treatment={len(treatment)}; "
          f"shared_old={info['replay_shared_count']}, "
          f"old_focus={info['treatment_in_domain_focus_count']}, "
          f"new_extension={info['treatment_extended_count']}")
    print(f"New unique math facts={info['new_extension_unique_math_facts']}; "
          f"new unique prompts={info['new_extension_unique_prompts']}; "
          f"max_len={info['max_sequence_with_padding']}")
    if a.verify_only:
        print("PASS: mathematically disjoint, no files written")
        return
    out.mkdir(parents=True,exist_ok=False)
    hashes={}
    hashes["replay_only"]=write_new(out/"replay_only_pairs.json",control)
    hashes["treatment"]=write_new(out/"treatment_pairs.json",treatment)
    hashes["extension_val"]=write_new(out/"extension_val.json",[
        {"prompt":r["prompt"],"answer":r["answer"],"category":r["category"],
         "fact_key":list(r["fact_key"])} for r in extension_val])
    info["file_sha256"]=hashes
    write_new(out/"manifest.json",info)
    print("PASS: no protected validation/test facts entered the new training data")
    print(f"Saved: {out}")


if __name__=="__main__":main()
'''


In [ ]:
# Exact upstream source: scripts/build_round5_decomposition.py
BUNDLED_GENERATOR_SOURCE["scripts/build_round5_decomposition.py"] = r'''#!/usr/bin/env python3
"""Round5: paired original-vs-decomposition explanations on SAME Round4 Treatment rows.

Both output files have exactly 100,000 rows, identical order/prompts/answers/
negatives. Arm A copies original Positive. Arm B modifies ONLY the explanatory
tail of mathematically eligible product-answer questions.

No model weights or original dataset are changed.
Usage:
  python scripts/build_round5_decomposition.py --verify-only
  python scripts/build_round5_decomposition.py
"""
from __future__ import annotations
import argparse
from collections import Counter
import hashlib
import json
from pathlib import Path
import pickle
import re
import sys

ROOT=Path(__file__).resolve().parents[1]
sys.path.insert(0,str(ROOT/"scripts"))
from generate_round1 import parse_relation
from generate_round3_structured import REFUSAL

SOURCE=ROOT/"dpo/data/round4_matched_full"
SOURCE_FILE=SOURCE/"treatment_pairs.json"
DEST=ROOT/"dpo/data/round5_decomposition"
NUM=r"[0-9]+"
DIRECT_RE=re.compile(rf"^({NUM})\*({NUM})=\?$")
UNKNOWN_RE=re.compile(rf"^x/({NUM})=({NUM}),x=\?$")
POS_RE=re.compile(
    rf"^(?P<prompt>.+?=\?) The answer is (?P<answer>{NUM}) "
    rf"because (?P<explanation>.+)\.$"
)
KINDS=("multiply","x_over_a")


def digest(path:Path)->str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def load_source():
    manifest=json.loads((SOURCE/"manifest.json").read_text(encoding="utf-8"))
    if manifest.get("profile")!="full" or manifest.get("training_arm_size")!=100000:
        raise RuntimeError("Round4 source is not the frozen full 100k experiment")
    if digest(SOURCE_FILE)!=manifest["file_sha256"]["treatment"]:
        raise RuntimeError("Round4 treatment SHA256 mismatch")
    pairs=json.loads(SOURCE_FILE.read_text(encoding="utf-8"))
    if len(pairs)!=100000:
        raise RuntimeError("Expected exactly 100,000 Round4 treatment rows")
    return manifest,pairs


def exact_factors(prompt:str,answer:int):
    m=DIRECT_RE.fullmatch(prompt)
    if m:
        a,b=int(m.group(1)),int(m.group(2))
        kind="multiply"
    else:
        m=UNKNOWN_RE.fullmatch(prompt)
        if not m:return None
        denominator,quotient=int(m.group(1)),int(m.group(2))
        a,b=quotient,denominator
        kind="x_over_a"
    if a*b!=answer or answer<100 or a<2 or b<2:
        return None
    return a,b,kind


def split_factors(a:int,b:int):
    """Choose an arithmetic split with two strictly positive partial products.

    Prefer split of the second operand to follow question order (23*17).
    If necessary, split the first operand. Never use +0 disguised as teaching.
    """
    for left,right in ((a,b),(b,a)):
        tens=right//10*10
        ones=right%10
        if tens>=10 and ones>=1:
            return left,tens,ones
    return None


def decomposition(pair:dict):
    pos,neg=pair["positive"],pair["negative"]
    m=POS_RE.fullmatch(pos)
    if not m:
        raise ValueError(f"Unexpected original Positive format: {pos[:90]}")
    prompt,ans=m.group("prompt"),int(m.group("answer"))
    if neg!=f"{prompt} {REFUSAL}":
        raise ValueError(f"Negative is not the unchanged course refusal: {prompt}")
    # Every row is independently verified against its printed mathematical task.
    parse_relation(prompt,str(ans))
    factors=exact_factors(prompt,ans)
    if factors is None:
        return pos,None
    a,b,kind=factors
    split=split_factors(a,b)
    if split is None:
        return pos,None
    x,tens,ones=split
    first=x*tens
    second=x*ones
    if first+second!=ans or not(first>0 and second>0):
        raise AssertionError(f"Arithmetic decomposition wrong for {prompt}")
    # Canonical phrase; all intermediate expressions have been checked.
    explanation=(f"{x}*{tens}={first} and "
                 f"{x}*{ones}={second} and "
                 f"{first}+{second}={ans}")
    new_pos=f"{prompt} The answer is {ans} because {explanation}."
    if new_pos==pos or new_pos.count(".")!=1:
        raise AssertionError(f"Invalid decomposition text: {new_pos}")
    return new_pos, {
        "prompt":prompt,"answer":ans,"category":kind,
        "factors":[a,b],"split":[x,tens,ones],
        "partial_products":[first,second],
    }


def assemble():
    source_manifest,pairs=load_source()
    a_pairs=[]
    b_pairs=[]
    examples=[]
    counts=Counter()
    eligible_by_answer=Counter()
    reasons=Counter()
    vocab_path=ROOT/"sft/meta.pkl"
    with vocab_path.open("rb") as f:
        vocab=set(pickle.load(f)["stoi"])
    max_length=0
    for i,pair in enumerate(pairs):
        old=pair["positive"];neg=pair["negative"]
        new,detail=decomposition(pair)
        if detail is not None:
            counts[detail["category"]]+=1
            band = ("100-400" if detail["answer"]<=400 else
                    "401-999" if detail["answer"]<=999 else "1000+")
            eligible_by_answer[band]+=1
            if len(examples)<12:examples.append({"row":i,"original":old,"decomposition":new,**detail})
        else:
            if "=? The answer is " not in old:
                raise AssertionError(f"Unexpected math template: {old[:90]}")
            reasons["unchanged"]+=1
        if set(old+new+neg)-vocab:
            raise RuntimeError(f"Tokenizer OOV in row {i}")
        max_length=max(max_length,len(old)+4,len(new)+4,len(neg)+4)
        if max_length>256:
            raise RuntimeError(f"Sequence exceeds model block 256 at row {i}: {len(new)}")
        a_pairs.append({"positive":old,"negative":neg})
        b_pairs.append({"positive":new,"negative":neg})
    changed=sum(counts.values())
    if changed==0:
        raise RuntimeError("No eligible decomposition samples found; check source data")
    if changed<5000:
        print(f"WARNING: only {changed} decomposition-modified rows; "
              "interpret a null result as possibly limited treatment intensity",flush=True)
    for pa,pb in zip(a_pairs,b_pairs):
        if pa["negative"]!=pb["negative"]:
            raise AssertionError("Control negative differed")
        ap=POS_RE.fullmatch(pa["positive"])
        bp=POS_RE.fullmatch(pb["positive"])
        if ap.group("prompt")!=bp.group("prompt") or ap.group("answer")!=bp.group("answer"):
            raise AssertionError("Control prompt/answer differed")
    report={
        "version":"round5-decomposition-v1",
        "source_round4_treatment_sha256":digest(SOURCE_FILE),
        "source_round4_manifest_sha256":digest(SOURCE/"manifest.json"),
        "source_round4_training_rows":len(pairs),
        "arm_A":"original one-step because",
        "arm_B":"exact distributive decomposition after correct numerical answer",
        "same_rows_order_answers_negatives":True,
        "modified_rows":changed,
        "unmodified_rows":len(pairs)-changed,
        "modified_fraction":changed/len(pairs),
        "modified_by_category":dict(counts),
        "modified_by_answer_band":dict(eligible_by_answer),
        "max_required_length_with_padding":max_length,
        "max_model_context":256,
        "selected_examples":examples,
        "training_plan":"both from Round4 Treatment Epoch1, 1 epoch, same seed/batch/LR, original course Loss",
        "no_training_or_test_evaluation":True,
    }
    return a_pairs,b_pairs,report


def write_new(path,data):
    path.parent.mkdir(parents=True,exist_ok=True)
    blob=(json.dumps(data,ensure_ascii=False,separators=(",",":"))+"\n").encode("utf-8")
    with path.open("xb") as f:f.write(blob)
    return hashlib.sha256(blob).hexdigest()


def main():
    p=argparse.ArgumentParser(description=__doc__)
    p.add_argument("--verify-only",action="store_true")
    p.add_argument("--out-dir",type=Path,default=DEST)
    args=p.parse_args()
    out=args.out_dir if args.out_dir.is_absolute() else ROOT/args.out_dir
    if out.exists() and not args.verify_only:
        p.error(f"Refusing to overwrite existing Round5 folder: {out}")
    a,b,manifest=assemble()
    print("=== ROUND 5 DATA AUDIT ===")
    print(f"Training rows per arm: {len(a)}")
    print(f"Decomposition-modified rows: {manifest['modified_rows']} "
          f"({manifest['modified_fraction']:.2%})")
    print(f"By category: {manifest['modified_by_category']}")
    print(f"By answer band: {manifest['modified_by_answer_band']}")
    print(f"Unmodified rows: {manifest['unmodified_rows']}")
    print(f"Max sequence length including 4 padding characters: {manifest['max_required_length_with_padding']}")
    print("Example A:",manifest["selected_examples"][0]["original"])
    print("Example B:",manifest["selected_examples"][0]["decomposition"])
    if args.verify_only:
        print("PASS: verified without creating files")
        return
    out.mkdir(parents=True,exist_ok=False)
    hashes={}
    hashes["original"]=write_new(out/"original_pairs.json",a)
    hashes["decomposition"]=write_new(out/"decomposition_pairs.json",b)
    manifest["file_sha256"]=hashes
    write_new(out/"manifest.json",manifest)
    print("PASS: paired datasets and manifest saved to",out)


if __name__=="__main__":
    main()
'''


In [ ]:
# Exact upstream source: scripts/build_round7_targeted_data.py
BUNDLED_GENERATOR_SOURCE["scripts/build_round7_targeted_data.py"] = r'''#!/usr/bin/env python3
"""Round7: mathematically isolated replay vs targeted multiplication curriculum.

Only OLD TRAIN rows may be replayed. New 11..99 times 11..99 facts with product
1000..4999 use the original stable 42 TRAIN partition, excluding the existing
Val768, Old OOD256, Extended384 and all previously trained facts.

NEVER open the independent sealed Round3 test or the prior-year public blind
questions. Only predeclared skill-level strata from earlier analysis guide
the synthetic math generation.

Usage:
  python scripts/build_round7_targeted_data.py --profile mini --verify-only
  python scripts/build_round7_targeted_data.py --profile full --verify-only
  python scripts/build_round7_targeted_data.py --profile full
"""
from __future__ import annotations

import argparse
from collections import Counter
import hashlib
import json
from pathlib import Path
import pickle
import random
import re
import sys

ROOT=Path(__file__).resolve().parents[1]
sys.path.insert(0,str(ROOT/"scripts"))
from generate_round1 import parse_relation,stable_split,RESERVED
from generate_round3_structured import product_relations,REFUSAL

SEED=42
DATA=ROOT/"dpo/data/round5_decomposition"
OLD=DATA/"original_pairs.json"
R3=ROOT/"dpo/data/round3_structured_full"
R4=ROOT/"dpo/data/round4_matched_full"
OOD=ROOT/"dpo/experiments/round1c_incorrect/common_integer_val_256.json"

# Errors in Round5-A were concentrated in direct large-product arithmetic.
# Keep smaller inverse-form training to avoid losing them.
QUOTA={"multiply":3750,"x_over_a":3750,"x_times_a":2250,
       "a_times_x":2250,"divide":1500,"a_over_x":1500}
assert sum(QUOTA.values())==15000

POS_RE=re.compile(r"^(?P<prompt>.+?=\?) The answer is (?P<answer>\d+) because .+\.$")
FAMILY_PATTERNS={
    "multiply":re.compile(r"\d+\*\d+=\?"),
    "x_over_a":re.compile(r"x/\d+=\d+,x=\?"),
    "x_times_a":re.compile(r"x\*\d+=\d+,x=\?"),
    "a_times_x":re.compile(r"\d+\*x=\d+,x=\?"),
    "divide":re.compile(r"\d+/\d+=\?"),
    "a_over_x":re.compile(r"\d+/x=\d+,x=\?"),
}


def sha(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def load(path):
    return json.loads(path.read_text(encoding="utf-8"))


def source_and_guards():
    m=load(DATA/"manifest.json")
    if m.get("source_round4_training_rows")!=100000:
        raise ValueError("Round5 source is not 100k")
    if sha(OLD)!=m["file_sha256"]["original"]:
        raise ValueError("Round5 source train sha differs from manifest")
    rm=load(R3/"manifest.json")
    fm=load(R4/"manifest.json")
    if sha(R3/"val.json")!=rm["sha256"]["val"]:
        raise ValueError("Round3 frozen validation modified")
    if sha(OOD)!=fm["source_sha256"]["old_val_sha256"]:
        raise ValueError("Old OOD validation modified")
    if sha(R4/"extension_val.json")!=fm["file_sha256"]["extension_val"]:
        raise ValueError("Extended validation modified")
    val=load(R3/"val.json")
    ood=load(OOD)
    ext=load(R4/"extension_val.json")
    if (len(val),len(ood),len(ext))!=(768,256,384):
        raise ValueError("Unexpected number of frozen validation records")
    protected=set()
    for records in (val,ood,ext):
        for r in records:
            protected.add(parse_relation(r["prompt"],str(r["answer"])))
    protected.update(parse_relation(p,a) for p,a in RESERVED.items()
                     if "." not in p+a)
    old=load(OLD)
    if len(old)!=100000:
        raise ValueError("Expected 100k Round5 original training rows")
    parsed=[]
    prior_facts=set()
    by_cat={}
    for cat in QUOTA:by_cat[cat]=[]
    for i,p in enumerate(old):
        match=POS_RE.fullmatch(p["positive"])
        if match is None:raise ValueError(f"Unexpected original training positive row {i}")
        prompt,answer=match["prompt"],match["answer"]
        if p["negative"]!=f"{prompt} {REFUSAL}":
            raise ValueError(f"Negative preference template changed at {i}")
        fact=parse_relation(prompt,answer)
        if fact in protected:
            raise ValueError(f"Original training data overlaps a protected validation fact: {prompt}")
        prior_facts.add(fact)
        cat=next((k for k,pattern in FAMILY_PATTERNS.items()
                  if pattern.fullmatch(prompt)),None)
        product=int(fact[3]) if fact[0]=="mul" else None
        if cat in QUOTA and product is not None and product>=101:
            # Replay difficult mathematical facts already seen at training,
            # including direct products 101..999 and reverse equations.
            by_cat[cat].append(i)
        parsed.append((prompt,answer,fact,cat))
    for category,candidates in by_cat.items():
        if not candidates:
            raise RuntimeError(f"No prior training examples for {category}")
    source_hashes={
        "old_100k_sha256":sha(OLD),
        "round5_manifest_sha256":sha(DATA/"manifest.json"),
        "round3_val_sha256":sha(R3/"val.json"),
        "old_ood_sha256":sha(OOD),
        "extended_val_sha256":sha(R4/"extension_val.json"),
    }
    # We intentionally do NOT open R3/test.json.
    # New fact stable_split == train proves it cannot be in the old test split.
    return old,prior_facts,protected,by_cat,source_hashes


def novel_pool(prior_facts,protected):
    pools={kind:{} for kind in QUOTA}
    facts=set()
    for a in range(11,100):
        for b in range(a,100):
            product=a*b
            if not 1000<=product<=4999:
                continue
            for left,right in (((a,b),(b,a)) if a!=b else ((a,b),)):
                for r in product_relations(left,right):
                    cat=r["category"]
                    if cat not in pools:continue
                    key=parse_relation(r["prompt"],r["answer"])
                    if (key in prior_facts or key in protected or
                        stable_split(key,SEED)!="train"):
                        continue
                    if key[0]!="mul" or int(key[3])!=product:
                        raise ValueError("Bad synthetic multiplication fact")
                    pools[cat].setdefault(r["prompt"],r)
                    facts.add(key)
    return {k:list(v.values()) for k,v in pools.items()},facts


def write_once(path,content):
    path.parent.mkdir(parents=True,exist_ok=True)
    blob=(json.dumps(content,ensure_ascii=False,separators=(",",":"))+"\n").encode("utf-8")
    with path.open("xb") as f:f.write(blob)
    return hashlib.sha256(blob).hexdigest()


def assemble(profile):
    rng=random.Random(42007)
    old,prior_facts,protected,hard_by_cat,source_hashes=source_and_guards()
    pool,possible_facts=novel_pool(prior_facts,protected)
    if profile=="full":
        shared=70000
        old_quota=QUOTA
        new_quota=QUOTA
    else:
        shared=448
        old_quota={cat:16 for cat in QUOTA}
        new_quota={cat:16 for cat in QUOTA}
    hard_n=sum(old_quota.values())
    new_n=sum(new_quota.values())
    if shared+hard_n+new_n!=(100000 if profile=="full" else 640):
        raise AssertionError("Round7 arm length wrong")
    permutation=list(range(len(old)))
    rng.shuffle(permutation)
    common=permutation[:shared]
    remaining=permutation[shared:]
    # SAME shared rows; control fills with leftover ordinary old TRAIN rows.
    control_extra=rng.sample(remaining,hard_n+new_n)
    control=[old[i] for i in common+control_extra]
    target=[old[i] for i in common]
    hard_counts={}
    for cat,count in old_quota.items():
        selected=rng.choices(hard_by_cat[cat],k=count)
        target.extend(old[i] for i in selected)
        hard_counts[cat]={
            "rows":count,"distinct_source_rows":len(set(selected)),
            "available_train_source_rows":len(hard_by_cat[cat])
        }
    new_counts={}
    novel_rows=[]
    for cat,count in new_quota.items():
        choices=pool[cat]
        if len(choices)<count:
            raise RuntimeError(f"New TRAIN-only pool {cat}: {len(choices)} < {count}")
        chosen=rng.sample(choices,count)
        novel_rows.extend(chosen)
        target.extend({"positive":r["positive"],"negative":r["negative"]}
                      for r in chosen)
        new_counts[cat]={
            "rows":count,"available_pool_prompts":len(choices),
            "unique_prompts":len({r["prompt"] for r in chosen}),
        }
    new_facts={tuple(r["fact_key"]) for r in novel_rows}
    if new_facts & protected or new_facts & prior_facts:
        raise RuntimeError("Novel candidate collides with previous training or validation")
    if any(stable_split(k,SEED)!="train" for k in new_facts):
        raise RuntimeError("New math facts must be in the original train partition")
    rng.shuffle(control)
    rng.shuffle(target)
    if len(control)!=len(target):
        raise RuntimeError("Unequal experimental training rows")
    vocab_path=ROOT/"sft/meta.pkl"
    with vocab_path.open("rb") as f:vocab=set(pickle.load(f)["stoi"])
    max_len=0
    for arm,rows in (("replay",control),("targeted",target)):
        for i,p in enumerate(rows):
            pos,neg=p["positive"],p["negative"]
            invalid=set(pos+neg)-vocab
            if invalid:
                raise ValueError(f"{arm} tokenizer OOV row {i}: {sorted(invalid)}")
            length=max(len(pos)+4,len(neg)+4)
            max_len=max(max_len,length)
            if length>256:
                raise ValueError(f"{arm} exceeds model context 256 row {i}: {length}")
            match=POS_RE.fullmatch(pos)
            if match is None:
                raise ValueError(f"{arm} unusual positive row {i}")
            p_prompt,p_answer=match["prompt"],match["answer"]
            if neg!=f"{p_prompt} {REFUSAL}":
                raise ValueError("Course negative preference was changed")
            key=parse_relation(p_prompt,p_answer)
            if key in protected or (arm=="targeted" and
                                    key not in prior_facts and key not in new_facts):
                raise ValueError(f"{arm} protected/unknown math fact: {p_prompt}")
    info={
        "version":"round7_matched_mult_v1",
        "profile":profile,"seed":42007,
        "source_training_sha256":source_hashes,
        "base_checkpoint":"dpo/experiments/round5_original/dpo.pt",
        "strategy":"train-only multiplication failure-type reinforcement",
        "same_shared_old_records":shared,
        "control_other_old_records":hard_n+new_n,
        "target_old_hard_records":hard_n,
        "target_new_math_records":new_n,
        "old_hard_by_category":hard_counts,
        "new_by_category":new_counts,
        "new_product_band":"1000..4999 (two-digit by two-digit)",
        "new_unique_math_facts":len(new_facts),
        "new_unique_prompts":len({r["prompt"] for r in novel_rows}),
        "possible_new_unique_math_facts":len(possible_facts),
        "previous_training_math_facts":len(prior_facts),
        "validation_protected_math_facts":len(protected),
        "every_new_fact_stable_split_train":True,
        "no_sealed_test_read":True,
        "no_public_blind_read":True,
        "max_sequence_length_with_padding":max_len,
        "rows_per_arm":len(control),
        "batch_size":64,
        "optimizer_updates_per_arm":len(control)//64,
        "discarded_incomplete_batch":len(control)%64,
        "init_model_same_for_both_arms":True,
        "training_plan":"from Round5-A; 1 epoch; Course Loss; same batch64 LR1e-5 seed; evaluate fixed primary",
    }
    if profile=="full" and len(new_facts)<1000:
        raise RuntimeError(f"Insufficient novel mathematical facts: {len(new_facts)}")
    print("=== ROUND7 MATCHED TRAINING DATA ===")
    print(f"control={len(control)} targeted={len(target)} shared_old={shared}")
    print(f"target old hard={hard_n}, target NEW TRAIN-split rows={new_n}")
    print("old hard:",json.dumps(hard_counts,sort_keys=True))
    print("new:",json.dumps(new_counts,sort_keys=True))
    print("new unique math facts:",len(new_facts),"old math facts:",len(prior_facts))
    print("protected facts:",len(protected),"max tokens incl padding:",max_len)
    print("No sealed final test or public blind tests opened.")
    return control,target,info


def main():
    p=argparse.ArgumentParser(description=__doc__)
    p.add_argument("--profile",choices=("mini","full"),default="mini")
    p.add_argument("--verify-only",action="store_true")
    p.add_argument("--out-dir",type=Path)
    args=p.parse_args()
    out=(args.out_dir if args.out_dir is not None else
         ROOT/f"dpo/data/round7_targeted_{args.profile}").resolve()
    if out.exists() and not args.verify_only:
        p.error(f"Cannot overwrite Round7 folder: {out}")
    control,target,info=assemble(args.profile)
    if args.verify_only:
        print("PASS: Round7 balanced TRAIN-only data audited, nothing written")
        return
    out.mkdir(parents=True,exist_ok=False)
    hashes={
        "replay":write_once(out/"replay_pairs.json",control),
        "targeted":write_once(out/"targeted_pairs.json",target),
    }
    info["file_sha256"]=hashes
    write_once(out/"manifest.json",info)
    print("PASS: saved",out)


if __name__=="__main__":
    main()
'''


In [ ]:
# Exact upstream source: scripts/build_round9_data.py
BUNDLED_GENERATOR_SOURCE["scripts/build_round9_data.py"] = r'''#!/usr/bin/env python3
"""Round9 data: matched Replay / Random / Structured from ONE candidate universe.

All rows retain the instructor Course-Loss answer-first format and refusal
negative.  Only TRAIN-partition canonical facts enter newly synthesized rows.
Known validation / public / teacher-style 680 facts are excluded from new rows.
The historical 70k replay may contain 680 Seen facts (report, do not hide).

CPU:
 python scripts/build_round9_data.py --self-test-only
 python scripts/build_round9_data.py --verify-only
 python scripts/build_round9_data.py --build
"""
from __future__ import annotations
import argparse
from collections import Counter, defaultdict
import hashlib
import json
from pathlib import Path
import pickle
import random
import re
import sys

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "scripts"))
from generate_round1 import parse_relation, stable_split
from generate_round3_structured import (
    KINDS, REFUSAL, sum_relations, product_relations
)
from build_teacher_style_400 import source_and_protected, count_carries
from evaluate_teacher_style_680 import validate as validate_680

SEED = 42
OUT = ROOT / "dpo/data/round9_balanced_full"
OLD = ROOT / "dpo/data/round5_decomposition/original_pairs.json"
OLD_MANIFEST = ROOT / "dpo/data/round5_decomposition/manifest.json"
BASE = ROOT / "dpo/experiments/round7_replay/dpo.pt"
REPLAY_N, NEW_PER_CAT, ROWS = 70000, 2500, 100000
BINS = ("b0", "b1", "b2", "b3")
ADDITIVE = set(KINDS[:2]) | {"x_plus_a","a_plus_x","x_minus_a","a_minus_x"}
POS = re.compile(r"^(?P<prompt>.+?=\?) The answer is (?P<answer>-?\d+) because .+\.$")


def read(path):
    return json.loads(path.read_text(encoding="utf-8"))


def digest(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def bytes_json(value):
    return (json.dumps(value, ensure_ascii=False, separators=(",", ":")) + "\n").encode()


def check_one(row, stoi, protected, new):
    m = POS.fullmatch(row["positive"])
    if not m:
        raise ValueError("Positive must be full answer-first because format")
    p, ans = m["prompt"], m["answer"]
    key = parse_relation(p, ans)
    if row["negative"] != p + " " + REFUSAL:
        raise ValueError("Non-refusal negative or mismatched prompt")
    if max(len(row["positive"]), len(row["negative"])) + 4 > 256:
        raise ValueError("Exceeds block_size=256")
    if set(row["positive"] + row["negative"]) - stoi:
        raise ValueError("Tokenizer out-of-vocabulary")
    if new and (key in protected or stable_split(key, SEED) != "train"):
        raise ValueError("New example contains protected/non-train math fact")
    return p, ans, key


def bin_for(category, key):
    """Four feasible mathematical strata per category; no fake examples."""
    if category in ADDITIVE:
        if key[0] != "add":
            raise ValueError("Wrong additive family")
        a, b = int(key[1]), int(key[2])
        # Balanced ranges + any carry; subtraction borrow is the inverse carry.
        return "b" + str(2 * int(max(a, b) > 30) + int(count_carries(a, b) > 0))
    if key[0] != "mul":
        raise ValueError("Wrong multiplicative family")
    a, b, product = int(key[1]), int(key[2]), int(key[3])
    if product <= 99: return "b0"
    if product <= 400 and min(a, b) <= 9: return "b1"
    if product <= 400: return "b2"
    if product <= 999: return "b3"
    raise ValueError("Product exceeds Round9 training domain (999)")


def pool_records(protected):
    pools = {c: {} for c in KINDS}
    def admit(record):
        key = tuple(record["fact_key"])
        if key in protected or stable_split(key, SEED) != "train":
            return
        cat = record["category"]
        bucket = bin_for(cat, key)
        pools[cat].setdefault(record["prompt"], {
            "positive": record["positive"], "negative": record["negative"],
            "prompt": record["prompt"], "answer": record["answer"],
            "fact_key": list(key), "category": cat, "bin": bucket
        })
    for a in range(0, 101):
        for b in range(a, 101):
            for x, y in (((a,b),(b,a)) if a != b else ((a,b),)):
                for rec in sum_relations(x, y):
                    admit(rec)
    for a in range(1, 100):
        for b in range(a, 100):
            if a * b > 999: continue
            for x, y in (((a,b),(b,a)) if a != b else ((a,b),)):
                for rec in product_relations(x, y):
                    admit(rec)
    by_cat = {}
    for cat in KINDS:
        records = list(pools[cat].values())
        counts = Counter(r["bin"] for r in records)
        if not records or any(counts[b] == 0 for b in BINS):
            raise RuntimeError(f"Insufficient eligible TRAIN-only candidates: {cat} {dict(counts)}")
        by_cat[cat] = records
        print(f"POOL {cat:12s}: n={len(records):5d}, "
              + " ".join(f"{b}={counts[b]}" for b in BINS), flush=True)
    return by_cat


def make():
    # Both benchmark parts MUST be frozen and hash-valid before generating training.
    core, extension, _, _ = validate_680()
    seen, protected, source_hashes = source_and_protected()
    teacher_keys = {tuple(r["fact_key"]) for r in core + extension}
    protected |= teacher_keys
    if len(core) + len(extension) != 680: raise RuntimeError("Frozen benchmark incomplete")
    if not OLD.is_file() or not BASE.is_file(): raise FileNotFoundError("Old data / Round7 replay missing")
    source_m = read(OLD_MANIFEST)
    if digest(OLD) != source_m["file_sha256"]["original"]:
        raise RuntimeError("Old Round5 TRAIN bytes changed from manifest")
    with (ROOT / "sft/meta.pkl").open("rb") as f:
        vocab = set(pickle.load(f)["stoi"])
    old = read(OLD)
    if len(old) != ROWS: raise RuntimeError("Old replay source must have 100000 rows")
    for pair in old:
        check_one(pair, vocab, protected, new=False)
    pools = pool_records(protected)
    rng = random.Random(42009)
    old_ids = list(range(ROWS))
    rng.shuffle(old_ids)
    common = [old[i] for i in old_ids[:REPLAY_N]]
    replay_extra = [old[i] for i in old_ids[REPLAY_N:]]
    random_extra, structured_extra = [], []
    for cat in KINDS:
        candidates = pools[cat]
        random_extra.extend(rng.choices(candidates, k=NEW_PER_CAT))
        by_bin = {b:[x for x in candidates if x["bin"] == b] for b in BINS}
        for bucket in BINS:
            structured_extra.extend(rng.choices(by_bin[bucket], k=NEW_PER_CAT // 4))
    if len(random_extra) != 30000 or len(structured_extra) != 30000:
        raise RuntimeError("30k newly sampled rows missing")
    full = {
        "replay": common + replay_extra,
        "random": common + random_extra,
        "structured": common + structured_extra
    }
    # Use the SAME positional shuffle for all three arms and metadata.
    order = list(range(ROWS))
    random.Random(420091).shuffle(order)
    pairs = {
        arm: [{"positive": src[i]["positive"], "negative": src[i]["negative"]}
              for i in order] for arm, src in full.items()
    }
    # Metadata mirrors structured arm row-for-row (helps exact error mining).
    structured_meta = []
    summary = {}
    for arm, src in full.items():
        rows = [src[i] for i in order]
        new_rows = [src[i] for i in order if i >= REPLAY_N]
        for r in rows:
            check_one(r, vocab, protected, new=False)
        for r in new_rows:
            if arm != "replay":
                check_one(r, vocab, protected, new=True)
        keys = {parse_relation(POS.fullmatch(r["positive"])["prompt"],
                               POS.fullmatch(r["positive"])["answer"]) for r in new_rows}
        summary[arm] = {
            "total_rows": len(rows), "extra_rows": len(new_rows),
            "extra_unique_facts": len(keys), "extra_new_to_historical_facts": len(keys - seen),
            "extra_unique_prompts": len({POS.fullmatch(r["positive"])["prompt"] for r in new_rows}),
            "extra_by_category": (dict(Counter(r["category"] for r in new_rows))
                                  if arm != "replay" else {}),
            "extra_by_category_bin": (
                {k:dict(Counter(r["bin"] for r in new_rows if r["category"]==k))
                 for k in KINDS} if arm != "replay" else {})
        }
        if arm == "structured":
            for idx, r in zip(order, rows):
                m = POS.fullmatch(r["positive"])
                key = parse_relation(m["prompt"], m["answer"])
                # Old replay rows can include other values / categories.
                structured_meta.append({
                    "prompt":m["prompt"], "answer":m["answer"],
                    "source":"old" if idx < REPLAY_N else "new",
                    "category": (r["category"] if idx >= REPLAY_N else None),
                    "bin": (r["bin"] if idx >= REPLAY_N else None),
                    "fact_key":list(key)
                })
    if pairs["structured"] == pairs["random"]:
        raise RuntimeError("Random and Structured samples unexpectedly identical")
    if len(structured_meta) != ROWS: raise RuntimeError("Misaligned metadata")
    manifest = {
        "version":"round9_structured_v1", "seed":SEED,
        "base_checkpoint_sha256":digest(BASE),
        "old_100k_sha256":digest(OLD),
        "old_manifest_sha256":digest(OLD_MANIFEST),
        "source_hashes":source_hashes,
        "core400_sha256":digest(ROOT/"dpo/benchmarks/teacher_style_400_v1/questions.json"),
        "extension280_sha256":digest(ROOT/"dpo/benchmarks/teacher_style_extension_280_v1/questions.json"),
        "protected_teacher_fact_count":len(teacher_keys),
        "train_rows": ROWS,"shared_old":REPLAY_N,
        "new_per_category":NEW_PER_CAT,"bin_quota_per_category":NEW_PER_CAT//4,
        "strata_additive":"max underlying addend <=30/>30 x zero/positive carry",
        "strata_multiplicative":"product<=99 / 100-400 one-digit factor / 100-400 two-digit factors / 401-999",
        "shared_permutation_seed":420091,
        "statistics":summary,
        "files_sha256":{}
    }
    for arm in pairs: manifest["files_sha256"][arm] = hashlib.sha256(bytes_json(pairs[arm])).hexdigest()
    manifest["files_sha256"]["structured_metadata"] = hashlib.sha256(bytes_json(structured_meta)).hexdigest()
    return pairs, structured_meta, manifest


def main():
    p=argparse.ArgumentParser(description=__doc__)
    p.add_argument("--self-test-only",action="store_true")
    p.add_argument("--verify-only",action="store_true")
    p.add_argument("--build",action="store_true")
    args=p.parse_args()
    if args.self_test_only:
        for a,b in ((7,8),(29,17),(40,51)):
            for record in sum_relations(a,b):
                assert bin_for(record["category"],tuple(record["fact_key"])) in BINS
        for a,b in ((8,7),(8,24),(15,18),(41,19)):
            for record in product_relations(a,b):
                assert bin_for(record["category"],tuple(record["fact_key"])) in BINS
        print("SELFTEST PASS: 12 categories, canonical fact bins")
        return
    if args.build == args.verify_only:
        p.error("Choose exactly one of --verify-only or --build")
    if OUT.exists() and args.build:
        raise FileExistsError(f"Refusing to overwrite dataset: {OUT}")
    pairs, meta, manifest = make()
    for a, s in manifest["statistics"].items(): print(a,s,flush=True)
    if args.verify_only:
        print("AUDIT PASS; no files written")
        return
    OUT.mkdir(parents=True,exist_ok=False)
    for arm in ("replay","random","structured"):
        (OUT/f"{arm}_pairs.json").write_bytes(bytes_json(pairs[arm]))
    (OUT/"structured_metadata.json").write_bytes(bytes_json(meta))
    (OUT/"manifest.json").write_bytes(bytes_json(manifest))
    for item, hash_value in manifest["files_sha256"].items():
        path=OUT/("structured_metadata.json" if item=="structured_metadata" else f"{item}_pairs.json")
        if digest(path) != hash_value: raise RuntimeError(f"Output SHA mismatch: {path}")
    print(f"BUILD PASS: {OUT}; SHA256 and protected facts verified")


if __name__ == "__main__":
    main()
'''


In [ ]:
# Exact upstream source: scripts/build_teacher_style_400.py
BUNDLED_GENERATOR_SOURCE["scripts/build_teacher_style_400.py"] = r'''#!/usr/bin/env python3
"""Freeze a NEW 400-question SC3000 teacher-style, ICLR-2024-inspired benchmark.

NOT a verbatim ICLR dataset. The paper's balanced digit/carry idea is adapted
to five SC3000 example categories, with exactly four subskills x 20 questions.
Separately label Seen versus Unseen canonical mathematical facts relative to
the frozen Round3/Round4/Round5 training source. Existing Val/OOD/Extended,
prior-year public blind questions and professor's examples are excluded.

ALL selected facts must belong to the ORIGINAL stable_split(key,42) TRAIN
partition, even for UNSEEN questions. This is necessary to keep the sealed
Round3 independent TEST partition untouched without reading the Test file.

Commands (CPU):
  python scripts/build_teacher_style_400.py --self-test-only
  python scripts/build_teacher_style_400.py --audit-only
  python scripts/build_teacher_style_400.py --freeze

Once written, NEVER regenerate, rebalance or modify its 400 rows for tuning.
"""
from __future__ import annotations

import argparse
from collections import Counter,defaultdict
from fractions import Fraction
import hashlib
import json
from pathlib import Path
import random
import re
import sys

ROOT=Path(__file__).resolve().parents[1]
sys.path.insert(0,str(ROOT/"scripts"))
from generate_round1 import parse_relation,stable_split,RESERVED
from pipeline_smoke import verify_math

R3=ROOT/"dpo/data/round3_structured_full"
R4=ROOT/"dpo/data/round4_matched_full"
R5=ROOT/"dpo/data/round5_decomposition"
OOD=ROOT/"dpo/experiments/round1c_incorrect/common_integer_val_256.json"
PUBLIC=ROOT/"dpo/reference_2025_blind_sets.json"
OUT=ROOT/"dpo/benchmarks/teacher_style_400_v1"

SEED=20261030
GROUPS=("add","multiply","divide","a_minus_x","x_times_a")
SUBSKILL=20
PER_GROUP=4*SUBSKILL
EXPECTED_TOTAL=5*PER_GROUP

def read(path):
    return json.loads(path.read_text(encoding="utf-8"))

def sha(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def count_carries(a,b):
    """Digit sum identity from the paper's carry-balancing data code."""
    ds=lambda n:sum(int(ch) for ch in str(n))
    return (ds(a)+ds(b)-ds(a+b))//9

def attr(group,prompt,answer):
    m=re.fullmatch(r"(\d+)\+(\d+)=\?",prompt) if group=="add" else None
    if m:
        a,b=map(int,m.groups())
        u=(a%10+b%10)>=10
        t=(a//10+b//10+int(u))>=10
        assert count_carries(a,b)==int(u)+int(t)
        return f"ones_carry{int(u)}_tens_carry{int(t)}"
    m=re.fullmatch(r"(\d+)\*(\d+)=\?",prompt) if group=="multiply" else None
    if m:
        a,b=map(int,m.groups())
        return f"result_{len(str(int(answer)))}digit_ones_carry{int(a*(b%10)>=10)}"
    m=re.fullmatch(r"(\d+)/(\d+)=\?",prompt) if group=="divide" else None
    if m:
        a,b=map(int,m.groups())
        return f"quotient_{len(str(int(answer)))}digit_divisor_{'even' if b%2==0 else 'odd'}"
    m=re.fullmatch(r"(\d+)-x=(\d+),x=\?",prompt) if group=="a_minus_x" else None
    if m:
        a,b=map(int,m.groups())
        return f"answer_{len(str(int(answer)))}digit_borrow{int(a%10<b%10)}"
    m=re.fullmatch(r"x\*(\d+)=(\d+),x=\?",prompt) if group=="x_times_a" else None
    if m:
        a,b=map(int,m.groups())
        return f"answer_{len(str(int(answer)))}digit_multiplier_{len(str(a))}digit"
    raise ValueError(f"Unrecognized {group} expression {prompt}")

STRATA={
    "add":tuple(f"ones_carry{u}_tens_carry{t}" for u in (0,1) for t in (0,1)),
    "multiply":tuple(f"result_{n}digit_ones_carry{c}" for n in (2,3) for c in (0,1)),
    "divide":tuple(f"quotient_{n}digit_divisor_{p}" for n in (1,2) for p in ("even","odd")),
    "a_minus_x":tuple(f"answer_{n}digit_borrow{b}" for n in (1,2) for b in (0,1)),
    "x_times_a":tuple(f"answer_{n}digit_multiplier_{m}digit" for n in (1,2) for m in (1,2)),
}

def examples():
    """Enumerate the intended elementary difficulty, no 4-digit product."""
    for a in range(10,100):
        for b in range(10,100):
            yield "add",f"{a}+{b}=?",str(a+b)
    for a in range(2,10):
        for b in range(10,100):
            yield "multiply",f"{a}*{b}=?",str(a*b)
    for divisor in range(2,10):
        for quotient in range(1,100):
            dividend=divisor*quotient
            if 1<=dividend<=99:
                yield "divide",f"{dividend}/{divisor}=?",str(quotient)
    for known in range(20,100):
        for result in range(10,known):
            x=known-result
            if 1<=x<=99:
                yield "a_minus_x",f"{known}-x={result},x=?",str(x)
    for multiplier in range(2,16):
        for x in range(1,80):
            product=multiplier*x
            if product<=999:
                yield "x_times_a",f"x*{multiplier}={product},x=?",str(x)

def source_and_protected():
    r3m=read(R3/"manifest.json")
    r4m=read(R4/"manifest.json")
    r5m=read(R5/"manifest.json")
    sources={
        "round3_train_meta":R3/"train_metadata.json",
        "round3_val":R3/"val.json",
        "round4_treatment":R4/"treatment_pairs.json",
        "round4_extended_val":R4/"extension_val.json",
        "round5_train":R5/"original_pairs.json",
        "old_ood":OOD,
        "published_public_blind":PUBLIC,
    }
    if sha(sources["round3_train_meta"])!=r3m["sha256"]["train_metadata"]:
        raise RuntimeError("Round3 original training metadata SHA differs")
    if sha(sources["round3_val"])!=r3m["sha256"]["val"]:
        raise RuntimeError("Round3 frozen val SHA differs")
    if sha(sources["round4_treatment"])!=r4m["file_sha256"]["treatment"]:
        raise RuntimeError("Round4 treatment training SHA differs")
    if sha(sources["round4_extended_val"])!=r4m["file_sha256"]["extension_val"]:
        raise RuntimeError("Round4 extended val SHA differs")
    if sha(sources["round5_train"])!=r5m["file_sha256"]["original"]:
        raise RuntimeError("Round5 original training SHA differs")
    if sha(sources["old_ood"])!=r4m["source_sha256"]["old_val_sha256"]:
        raise RuntimeError("Old OOD validation SHA differs")
    seen=set()
    r3_train=read(sources["round3_train_meta"])
    if len(r3_train)!=100000:raise RuntimeError("Missing old 100k train source")
    for r in r3_train:
        seen.add(tuple(r["fact_key"]))
    regex=re.compile(r"^(?P<prompt>.+?=\?) The answer is (?P<answer>\d+) because ")
    for name in ("round4_treatment","round5_train"):
        pairs=read(sources[name])
        if len(pairs)!=100000:raise RuntimeError(f"Unexpected 100k rows: {name}")
        for pair in pairs:
            m=regex.match(pair["positive"])
            if not m:raise ValueError(f"Old TRAIN positive not recognized: {name}")
            seen.add(parse_relation(m["prompt"],m["answer"]))
    protected=set()
    for name in ("round3_val","old_ood","round4_extended_val"):
        for row in read(sources[name]):
            protected.add(parse_relation(row["prompt"],str(row["answer"])))
    for prompt,answer in RESERVED.items():
        if "." not in prompt+answer:
            protected.add(parse_relation(prompt,answer))
    public=read(PUBLIC)
    for group in ("blind61","blind41"):
        for row in public["groups"][group]["records"]:
            protected.add(parse_relation(row["prompt"],str(row["answer"])))
    if not seen.isdisjoint(protected):
        # Some protected public questions are known to overlap historical TRAIN.
        # This is acceptable: those facts are EXCLUDED from new benchmark.
        print("NOTE: previously used public/instructor facts overlap old TRAIN; "
              "they are excluded from the new benchmark.")
    return seen,protected,{k:sha(v) for k,v in sources.items()}

def candidates(seen,protected):
    buckets=defaultdict(list)
    encounter=set()
    rejection=Counter()
    for group,prompt,answer in examples():
        fact=parse_relation(prompt,answer)
        if fact in protected:
            rejection["protected"]+=1;continue
        if stable_split(fact,42)!="train":
            rejection["not_original_train_partition"]+=1;continue
        if (group,prompt) in encounter:continue
        encounter.add((group,prompt))
        sub=attr(group,prompt,answer)
        if sub not in STRATA[group]:
            raise ValueError(f"Unexpected subskill {group} {sub}")
        if not verify_math(prompt,answer):
            raise ValueError(f"Incorrect generated answer: {prompt} -> {answer}")
        if "?" not in prompt or any(ord(ch)>127 for ch in prompt):
            raise ValueError(prompt)
        buckets[(group,sub)].append({
            "prompt":prompt,"answer":answer,
            "category":group,"subskill":sub,
            "fact_key":list(fact),
            "seen_status":"seen" if fact in seen else "unseen",
        })
    return buckets,rejection

def select_group(group,buckets,used_global,rng):
    """Jointly allocate 20 distinct facts to each of a group's four subskills.

    Greedy, one-subskill-at-a-time selection can consume the same math fact in
    two neighboring strata. For example 8*7=56 could appear as 56/7=8 or
    56/8=7, across odd/even divisor groups. A bipartite augmenting-path
    matching reassigns previously chosen facts so a feasible 4x20 solution is
    found instead of failing due to the order of sampling.

    Historical Seen/Unseen is a preference, NOT a hard quota. Never invent
    missing unseen facts just to force 10/10. Preserve 400 distinct prompts
    and distinct canonical facts WITHIN each operation group.
    """
    subs=STRATA[group]
    # Within each subskill multiple surface variants can represent one fact.
    # Keep ONE representative per (subskill,fact_key) deterministically.
    per_sub={}
    for sub in subs:
        rows=list(buckets[(group,sub)])
        rng.shuffle(rows)
        unique={}
        for row in rows:
            unique.setdefault(tuple(row["fact_key"]),row)
        per_sub[sub]=unique
        if len(unique)<SUBSKILL:
            counts=Counter(x["seen_status"] for x in unique.values())
            raise RuntimeError(
                f"{group}/{sub}: only {len(unique)} unique mathematical facts "
                f"available, fewer than {SUBSKILL}; Seen/Unseen={dict(counts)}."
            )

    fact_degree=Counter()
    for unique in per_sub.values():
        fact_degree.update(unique.keys())

    # Give scarce strata first access and minimize overlap where possible.
    # The augmenter may subsequently move any selected fact to free a
    # constrained slot; there is NO irreversible greedy reservation.
    sub_order=sorted(subs,key=lambda sub:(len(per_sub[sub]),sub))
    slot_order=[(sub,i) for sub in sub_order for i in range(SUBSKILL)]
    edges={}
    for sub,i in slot_order:
        keys=list(per_sub[sub])
        rng.shuffle(keys)
        # Prefer Seen in first 10 slots, Unseen in last 10. Scarcity of the
        # other label is allowed; the output reports actual counts.
        preferred="seen" if i<SUBSKILL//2 else "unseen"
        keys.sort(key=lambda k:(
            0 if per_sub[sub][k]["seen_status"]==preferred else 1,
            fact_degree[k],
            0 if k not in used_global else 1,
        ))
        edges[(sub,i)]=keys

    owner={}      # canonical math fact -> slot holding it
    assignment={} # slot -> canonical math fact

    def augment(slot,visited):
        for key in edges[slot]:
            if key in visited:
                continue
            visited.add(key)
            previous_owner=owner.get(key)
            if previous_owner is None or augment(previous_owner,visited):
                owner[key]=slot
                assignment[slot]=key
                return True
        return False

    missing=[]
    for slot in slot_order:
        if not augment(slot,set()):
            missing.append(slot)

    # The elementary division universe is finite: with 1-digit quotients
    # and divisors 2..9, there may be FEWER than 40 distinct available
    # canonical math facts after the original train split. We may still
    # fairly test 40 DIFFERENT division prompts such as 56/7=? and 56/8=?.
    # Prefer a maximum matching first; when mathematically impossible,
    # permit cross-subskill reuse of a math fact but NEVER duplicate within
    # one subskill or reuse any EXACT PROMPT.
    within_group_reused=0
    for slot in missing:
        sub=slot[0]
        current={assignment[(sub,i)] for i in range(SUBSKILL)
                 if (sub,i) in assignment}
        available=[k for k in edges[slot] if k not in current]
        if not available:
            raise RuntimeError(
                f"{group}/{sub}: even allowing different question forms, "
                f"not enough distinct eligible facts to fill {SUBSKILL} "
                "prompts in this subskill. Do not copy validation/test facts."
            )
        # Reuse the least-often-present fact first, and prefer new facts
        # relative to previously selected OTHER operation groups.
        usage=Counter(assignment.values())
        available.sort(key=lambda k:(
            usage[k],
            0 if k not in used_global else 1,
            fact_degree[k],
        ))
        assignment[slot]=available[0]
        within_group_reused+=int(usage[available[0]]>0)

    result=[]
    for sub in subs:
        rows=[per_sub[sub][assignment[(sub,i)]] for i in range(SUBSKILL)]
        if len({tuple(x["fact_key"]) for x in rows})!=SUBSKILL:
            raise RuntimeError(f"Matching produced duplicate facts WITHIN {group}/{sub}")
        if len({x["prompt"] for x in rows})!=SUBSKILL:
            raise RuntimeError(f"Matching produced duplicate prompts in {group}/{sub}")
        result.extend(rows)
    expected=len(subs)*SUBSKILL
    if len(result)!=expected:
        raise RuntimeError(f"Joint matching failed {group} row count")
    if len({x["prompt"] for x in result})!=expected:
        raise RuntimeError(f"Joint matching failed {group} prompt uniqueness")
    unique_facts=len({tuple(x["fact_key"]) for x in result})
    print(f"  Joint selection {group}: {expected} different prompts, "
          f"{unique_facts} different canonical facts, "
          f"{expected-unique_facts} cross-subskill fact reuses",flush=True)
    return result


def test_matching():
    """Exercise the REAL matcher on Hall-tight synthetic overlapping strata."""
    # A has 29 math facts, B has 30, overlap 19 and union precisely 40.
    # A naive sequential sampling can waste the overlap and fail, although
    # a valid 20+20 matching exists. This case tests displacement/reassignment.
    label="__two_subskill_match_test__"
    if label in STRATA:
        raise AssertionError("Synthetic test group unexpectedly exists")
    STRATA[label]=("A","B")
    def make(i):
        return {"fact_key":["mul",str(i),"2",str(i*2)],
                "seen_status":"seen" if i%2 else "unseen",
                "prompt":str(i),"subskill":"A"}
    buckets={
        (label,"A"):[make(i) for i in range(29)],
        (label,"B"):[dict(make(i),subskill="B")
                     for i in list(range(19))+list(range(29,40))]
    }
    try:
        chosen=select_group(label,buckets,set(),random.Random(42))
    finally:
        del STRATA[label]
    assert len(chosen)==40
    assert Counter(x["subskill"] for x in chosen)=={"A":20,"B":20}
    assert len({tuple(x["fact_key"]) for x in chosen})==40
    # When only 36 facts exist for 40 questions, the matcher MUST use four
    # distinct prompt forms with repeated facts ACROSS two subskills.
    label2="__scarce_match_test__"
    STRATA[label2]=("A","B")
    tiny={
        (label2,"A"):[dict(make(i),prompt=f"A{i}",subskill="A") for i in range(29)],
        (label2,"B"):[dict(make(i),prompt=f"B{i}",subskill="B")
                      for i in list(range(18))+list(range(29,36))]
    }
    # B has only 25 eligible facts; 2x20 needs 40 different prompt strings.
    try:
        fallback=select_group(label2,tiny,set(),random.Random(42))
    finally:
        del STRATA[label2]
    assert len(fallback)==40
    assert len({r["prompt"] for r in fallback})==40
    assert len({tuple(r["fact_key"]) for r in fallback})==36
    assert all(len({tuple(r["fact_key"]) for r in fallback
                    if r["subskill"]==sub})==20 for sub in ("A","B"))
    print("PASS real joint matcher: full disjoint case and impossible "
          "40-questions/36-facts case, no repeated prompt")

def build():
    rng=random.Random(SEED)
    seen,protected,source_hashes=source_and_protected()
    buckets,rejection=candidates(seen,protected)
    chosen=[]
    used=set()
    # Constrained group matching, not greedy stratum-by-stratum selection.
    # This preserves 20 facts per stratum and 80 unique facts per operation.
    for group in GROUPS:
        print(f"Candidate availability for {group} (unique facts per subskill):",flush=True)
        for sub in STRATA[group]:
            unique={tuple(r["fact_key"]):r for r in buckets[(group,sub)]}
            counts=Counter(r["seen_status"] for r in unique.values())
            print(f"  {sub:38} unique={len(unique):4} "
                  f"seen={counts['seen']:4} unseen={counts['unseen']:4}",flush=True)
        rows=select_group(group,buckets,used,rng)
        chosen.extend(rows)
        used.update(tuple(r["fact_key"]) for r in rows)
    order={group:i for i,group in enumerate(GROUPS)}
    suborder={(g,s):i for g in GROUPS for i,s in enumerate(STRATA[g])}
    chosen.sort(key=lambda r:(order[r["category"]],
                              suborder[(r["category"],r["subskill"])],
                              r["prompt"]))
    if len(chosen)!=EXPECTED_TOTAL:
        raise RuntimeError(f"Wrong total: {len(chosen)}")
    if len({r["prompt"] for r in chosen})!=EXPECTED_TOTAL:
        raise RuntimeError("Benchmark contains duplicate exact prompts")
    if any(len({r["prompt"] for r in chosen if r["category"]==g})!=PER_GROUP
           for g in GROUPS):
        raise RuntimeError("Within-group benchmark has duplicate exact prompts")
    for g in GROUPS:
        for sub in STRATA[g]:
            selected=[r for r in chosen if r["category"]==g and r["subskill"]==sub]
            if len({tuple(r["fact_key"]) for r in selected})!=SUBSKILL:
                raise RuntimeError(f"Duplicate math fact WITHIN stratum {g}/{sub}")
    by_group=Counter(r["category"] for r in chosen)
    by_stratum=Counter((r["category"],r["subskill"]) for r in chosen)
    if any(by_group[g]!=PER_GROUP for g in GROUPS):
        raise RuntimeError(f"Wrong group quotas: {by_group}")
    if any(by_stratum[(g,s)]!=SUBSKILL for g in GROUPS for s in STRATA[g]):
        raise RuntimeError(f"Wrong subskill quotas: {by_stratum}")
    by_status=defaultdict(Counter)
    for r in chosen:
        fact=tuple(r["fact_key"])
        if fact in protected or stable_split(fact,42)!="train":
            raise RuntimeError(f"Protected or non-TRAIN fact: {r['prompt']}")
        if (fact in seen)!=(r["seen_status"]=="seen"):
            raise RuntimeError("Incorrect seen/unseen label")
        if not verify_math(r["prompt"],r["answer"]):
            raise RuntimeError(f"Incorrect arithmetic gold: {r}")
        if r["category"]=="multiply" and int(r["answer"])>=1000:
            raise RuntimeError("Four digit multiplication is not teacher style")
        by_status[r["category"]][r["seen_status"]]+=1
    manifest={
        "version":"teacher_style_400_v1",
        "seed":SEED,
        "paper":"Nayoung Lee et al., Teaching Arithmetic to Small Transformers, ICLR 2024",
        "paper_repo":"https://github.com/lee-ny/teaching_arithmetic",
        "note":"New SC3000 benchmark inspired by digit/carry balance; NOT paper's raw dataset",
        "teacher_example_questions_kept_separate":True,
        "groups":list(GROUPS),
        "quota_per_group":PER_GROUP,
        "subskill_quota":SUBSKILL,
        "subskills":{g:list(STRATA[g]) for g in GROUPS},
        "selected_n":len(chosen),
        "by_seen_status":{g:dict(by_status[g]) for g in GROUPS},
        "total_seen":sum(r["seen_status"]=="seen" for r in chosen),
        "total_unseen":sum(r["seen_status"]=="unseen" for r in chosen),
        "original_stable_partition":"train for ALL benchmark facts",
        "fact_disjoint_with_existing_val_ood_extended_public_and_teacher_examples":True,
        "all_unique_prompts":True,
        "unique_fact_keys_total":len({tuple(r["fact_key"]) for r in chosen}),
        "math_fact_reuses_global_total":
            EXPECTED_TOTAL-len({tuple(r["fact_key"]) for r in chosen}),
        "math_fact_reuses_between_operation_groups":
            sum(len({tuple(r["fact_key"]) for r in chosen if r["category"]==g})
                for g in GROUPS)-len({tuple(r["fact_key"]) for r in chosen}),
        "within_each_subskill_unique_fact_keys":True,
        "unique_facts_by_operation":{g:len({tuple(r["fact_key"]) for r in chosen
                                          if r["category"]==g}) for g in GROUPS},
        "math_fact_reuses_within_operation":{
            g:PER_GROUP-len({tuple(r["fact_key"]) for r in chosen if r["category"]==g})
            for g in GROUPS
        },
        "selection_strategy":"maximum matching first; unavoidable cross-subskill fact reuse disclosed",
        "no_four_digit_multiplication_results":True,
        "no_sealed_round3_test_file_read":True,
        "source_sha256":source_hashes,
        "new_candidate_rejections":dict(rejection),
        "no_future_training_on_any_of_these_400_facts":True,
        "evaluation":"same original greedy model.py stop and strict starts-with-The answer is scoring; TA extractor unknown",
    }
    print("=== NEW TEACHER-STYLE 400 QUESTION BENCHMARK ===")
    for g in GROUPS:
        print(f"{g:12} {by_group[g]}/80 seen={by_status[g]['seen']} "
              f"unseen={by_status[g]['unseen']}")
        for sub in STRATA[g]:
            rs=[x for x in chosen if x["category"]==g and x["subskill"]==sub]
            ss=Counter(x["seen_status"] for x in rs)
            print(f"  {sub:38} {len(rs):2} seen={ss['seen']:2} unseen={ss['unseen']:2}")
    print("400 unique prompt strings;",
          len({tuple(r["fact_key"]) for r in chosen}),
          "unique canonical math facts (cross-task reuse disclosed); no old validation/public facts; "
          "sealed old Test NEVER opened.")
    return chosen,manifest

def write(path,obj):
    return (json.dumps(obj,ensure_ascii=False,indent=2)+"\n").encode("utf-8")

def freeze():
    if OUT.exists():
        raise FileExistsError(f"Refusing to overwrite frozen benchmark folder {OUT}")
    rows,manifest=build()
    OUT.mkdir(parents=True,exist_ok=False)
    blob=write(OUT/"questions.json",rows)
    with (OUT/"questions.json").open("xb") as f:f.write(blob)
    manifest["questions_sha256"]=hashlib.sha256(blob).hexdigest()
    with (OUT/"manifest.json").open("xb") as f:f.write(write(OUT/"manifest.json",manifest))
    print("FROZEN benchmark:",OUT)
    print("questions_sha256:",manifest["questions_sha256"])
    print("Commit questions.json + manifest.json before Round8 mining/training.")

def validate_frozen():
    questions=OUT/"questions.json";manifest=OUT/"manifest.json"
    if not questions.is_file() or not manifest.is_file():
        raise FileNotFoundError(
            "Benchmark not yet frozen. Run --audit-only successfully, then "
            "--freeze, then --check-frozen."
        )
    m=read(manifest); rows=read(questions)
    if m["version"]!="teacher_style_400_v1" or len(rows)!=400:
        raise RuntimeError("Wrong frozen benchmark version/size")
    if sha(questions)!=m["questions_sha256"]:
        raise RuntimeError("Frozen benchmark SHA MISMATCH: do not silently regenerate")
    if len({r["prompt"] for r in rows})!=400:
        raise RuntimeError("Duplicate exact prompts")
    if len({tuple(r["fact_key"]) for r in rows})!=m["unique_fact_keys_total"]:
        raise RuntimeError("Frozen canonical fact count changed")
    for r in rows:
        if tuple(r["fact_key"])!=parse_relation(r["prompt"],r["answer"]):
            raise RuntimeError("Benchmark fact key corrupted")
        if not verify_math(r["prompt"],r["answer"]):
            raise RuntimeError("Benchmark label incorrect")
    print("PASS frozen 400 and SHA:",m["questions_sha256"])

def selftest():
    assert count_carries(17,19)==1
    assert count_carries(55,66)==2
    assert attr("add","17+19=?","36")=="ones_carry1_tens_carry0"
    assert attr("multiply","3*17=?","51")=="result_2digit_ones_carry1"
    assert attr("divide","72/4=?","18")=="quotient_2digit_divisor_even"
    assert attr("a_minus_x","72-x=34,x=?","38")=="answer_2digit_borrow1"
    assert attr("x_times_a","x*11=44,x=?","4")=="answer_1digit_multiplier_2digit"
    assert sum(1 for _ in GROUPS)==5
    assert all(len(v)==4 for v in STRATA.values())
    assert 5*4*20==400
    test_matching()
    print("PASS 5 categories, 20 strata x20, carry/borrow labels, original teacher samples")

def main():
    p=argparse.ArgumentParser(description=__doc__)
    action=p.add_mutually_exclusive_group(required=True)
    action.add_argument("--self-test-only",action="store_true")
    action.add_argument("--audit-only",action="store_true")
    action.add_argument("--freeze",action="store_true")
    action.add_argument("--check-frozen",action="store_true")
    args=p.parse_args()
    if args.self_test_only:selftest()
    elif args.audit_only:build();print("AUDIT PASS; no files written")
    elif args.freeze:freeze()
    else:validate_frozen()

if __name__=="__main__": main()
'''


In [ ]:
# Exact upstream source: scripts/evaluate_teacher_style_680.py
BUNDLED_GENERATOR_SOURCE["scripts/evaluate_teacher_style_680.py"] = r'''#!/usr/bin/env python3
"""Uniform score on the two immutable teacher-style benchmark files (400+280).

DEVELOPMENT benchmark only. Both candidate checkpoints receive EXACT same
680 prompts, greedy decoding, model.py original stop and strict answer-first
grading. Sealed independent Test and prior-year public tests are not read.

Slurm: sbatch scripts/teacher_style_680_eval.sbatch
"""
from __future__ import annotations

import argparse
from collections import Counter, defaultdict
import hashlib
import json
from pathlib import Path
import pickle
import sys

ROOT=Path(__file__).resolve().parents[1]
sys.path.insert(0,str(ROOT))
sys.path.insert(0,str(ROOT/"scripts"))
CORE=ROOT/"dpo/benchmarks/teacher_style_400_v1"
EXT=ROOT/"dpo/benchmarks/teacher_style_extension_280_v1"
OUT=ROOT/"dpo/experiments/teacher_style_680_v1_eval"
MODELS={
    "round5_A":ROOT/"dpo/experiments/round5_original/dpo.pt",
    "round7_replay":ROOT/"dpo/experiments/round7_replay/dpo.pt",
    "round6_A":ROOT/"dpo/experiments/round6_from_sft/original/dpo.pt",
    "round6_D":ROOT/"dpo/experiments/round6_from_sft/decomposition_before/dpo.pt",
}
CORE_GROUPS=("add","multiply","divide","a_minus_x","x_times_a")
EXT_GROUPS=("subtract","x_plus_a","a_plus_x","x_minus_a","a_times_x","x_over_a","a_over_x")

def read(path):
    return json.loads(path.read_text(encoding="utf-8"))

def sha(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def frozen(path,version,n,unique_key):
    p=path/"questions.json"
    manifest=path/"manifest.json"
    if not p.exists() or not manifest.exists():
        raise FileNotFoundError(f"First freeze the two benchmark parts: {path}")
    meta=read(manifest);records=read(p)
    if meta.get("version")!=version or len(records)!=n:
        raise RuntimeError(f"Incorrect frozen benchmark version or size {path}")
    if sha(p)!=meta["questions_sha256"]:
        raise RuntimeError(f"Frozen benchmark checksum mismatch {path}")
    if len({r["prompt"] for r in records})!=n:
        raise RuntimeError(f"Benchmark contains repeated prompt {path}")
    if len({tuple(r["fact_key"]) for r in records})!=meta[unique_key]:
        raise RuntimeError(f"Benchmark unique fact count changed: {path}")
    return records,meta

def validate():
    from generate_round1 import parse_relation,stable_split
    from pipeline_smoke import verify_math
    core,cm=frozen(CORE,"teacher_style_400_v1",400,"unique_fact_keys_total")
    ext,em=frozen(EXT,"teacher_style_extension_280_v1",280,"unique_math_facts_total")
    if em["core_reference"]["core_questions_sha256"]!=sha(CORE/"questions.json"):
        raise RuntimeError("Extension was frozen against a different Core400")
    if em["core_reference"]["core_manifest_sha256"]!=sha(CORE/"manifest.json"):
        raise RuntimeError("Core400 manifest was modified after extension selection")
    if len({r["prompt"] for r in core+ext})!=680:
        raise RuntimeError("Core+Extension duplicate prompt")
    core_facts={tuple(r["fact_key"]) for r in core}
    ext_facts={tuple(r["fact_key"]) for r in ext}
    if core_facts & ext_facts:
        raise RuntimeError("Core and Extension share canonical arithmetic facts")
    if [r["category"] for r in core] != sorted(
        [r["category"] for r in core],key=lambda k:CORE_GROUPS.index(k)
    ):
        raise RuntimeError("Core group order has changed")
    groups=Counter(r["category"] for r in core+ext)
    for cat in CORE_GROUPS:
        if groups[cat]!=80:raise RuntimeError(f"Wrong core group size {cat}")
    for cat in EXT_GROUPS:
        if groups[cat]!=40:raise RuntimeError(f"Wrong extension group size {cat}")
    for label,rr in (("core400",core),("extension280",ext)):
        for r in rr:
            f=parse_relation(r["prompt"],str(r["answer"]))
            if (f!=tuple(r["fact_key"]) or stable_split(f,42)!="train" or
                not verify_math(r["prompt"],r["answer"])):
                raise RuntimeError(f"Invalid frozen benchmark item {label}: {r['prompt']}")
    print("PASS frozen Core400 + Extension280: 680 different prompts, "
          f"{len(core_facts)+len(ext_facts)} unique disjoint math facts; "
          "two independent manifests intact.")
    return core,ext,cm,em

def stats(rows):
    n=len(rows)
    correct=sum(r["strict_status"]=="correct" for r in rows)
    return {"n":n,"correct":correct,
            "accuracy":correct/n if n else None,
            "math_error":sum(r["strict_status"]=="math_error" for r in rows),
            "format_failure":sum(r["strict_status"]=="format_failure" for r in rows)}

def analyze(records):
    result={"overall":stats(records),"by_section":{},"by_category":{},
            "by_category_subskill":{},"by_seen_status":{},"by_section_seen_status":{}}
    for section in ("core400","extension280"):
        subset=[x for x in records if x["section"]==section]
        result["by_section"][section]=stats(subset)
        result["by_section_seen_status"][section]={
            label:stats([x for x in subset if x["seen_status"]==label])
            for label in ("seen","unseen")
        }
    for cat in CORE_GROUPS+EXT_GROUPS:
        subset=[x for x in records if x["category"]==cat]
        result["by_category"][cat]=stats(subset)
        result["by_category_subskill"][cat]={
            sub:stats([x for x in subset if x["subskill"]==sub])
            for sub in sorted({x["subskill"] for x in subset})
        }
    for label in ("seen","unseen"):
        result["by_seen_status"][label]=stats(
            [x for x in records if x["seen_status"]==label])
    return result

def diff(a,b):
    if len(a)!=len(b):raise RuntimeError("Models have different prompt counts")
    c=Counter()
    for x,y in zip(a,b):
        if x["prompt"]!=y["prompt"] or x["answer"]!=y["answer"]:
            raise RuntimeError("Question list differs between checkpoints")
        left=x["strict_status"]=="correct"
        right=y["strict_status"]=="correct"
        c["n"]+=1;c["a_correct"]+=left;c["b_correct"]+=right
        c["a_only"]+=left and not right;c["b_only"]+=right and not left
    return dict(c)

def main():
    p=argparse.ArgumentParser(description=__doc__)
    p.add_argument("--device",choices=("cpu","cuda"),default="cuda")
    p.add_argument("--max-new-tokens",type=int,default=80)
    p.add_argument("--models",default="round5_A,round7_replay",
                   help="Comma-separated known checkpoint aliases")
    p.add_argument("--out-dir",type=Path,default=OUT)
    p.add_argument("--check-only",action="store_true",
                   help="Validate both frozen benchmark parts without GPU inference")
    args=p.parse_args()
    core,ext,cm,em=validate()
    if args.check_only:
        print("PASS 680 read-only benchmark freeze validation; no GPU run")
        return
    selected=[n.strip() for n in args.models.split(",")]
    if (not selected or len(set(selected))!=len(selected) or
        any(n not in MODELS for n in selected)):
        p.error("Unknown or duplicate model name")
    if args.max_new_tokens<1:p.error("Max new tokens must be positive")
    output=(args.out_dir if args.out_dir.is_absolute() else ROOT/args.out_dir).resolve()
    experiments=(ROOT/"dpo/experiments").resolve()
    if not output.is_relative_to(experiments) or output==experiments or output.exists():
        p.error(f"Refusing existing/outside output directory {output}")
    import torch
    from model import GPT,GPTConfig
    from evaluate_ta_integer import greedy_original_stop
    from train_round3_course import evaluate_text
    if args.device=="cuda" and not torch.cuda.is_available():
        p.error("No CUDA GPU allocation; run through Slurm")
    with (ROOT/"sft/meta.pkl").open("rb") as f:
        meta=pickle.load(f)
    stoi,itos=meta["stoi"],meta["itos"]
    all_rows=[dict(x,section="core400") for x in core]+[
        dict(x,section="extension280") for x in ext]
    if any(set(row["prompt"])-set(stoi) for row in all_rows):
        raise RuntimeError("Unknown tokenizer characters in benchmark")
    for name in selected:
        if not MODELS[name].is_file():raise FileNotFoundError(MODELS[name])
    output.mkdir(parents=True,exist_ok=False)
    report={
        "benchmark":"teacher_style_680_v1","core_questions_sha256":sha(CORE/"questions.json"),
        "extension_questions_sha256":sha(EXT/"questions.json"),
        "max_new_tokens":args.max_new_tokens,
        "decode":"greedy NanoGPT, stop on first period or newline",
        "primary_scorer":"strict The answer is <integer> at beginning of completion, NOT verified TA",
        "benchmark_role":"fixed teacher-style development competency suite; seen facts are NOT unseen generalization",
        "models":{},"paired":{}
    }
    predictions_by_model={}
    for name in selected:
        file=MODELS[name]
        ckpt=torch.load(file,map_location="cpu",weights_only=True)
        model=GPT(GPTConfig(**ckpt["model_args"]))
        weights=ckpt.get("model_state_dict",ckpt.get("model"))
        if weights is None:raise RuntimeError(f"Missing model weights {file}")
        model.load_state_dict({k.removeprefix("_orig_mod."):v
                               for k,v in weights.items()},strict=True)
        model.to(args.device).eval()
        outputs=[]
        for i,row in enumerate(all_rows):
            completion,stop=greedy_original_stop(
                model,row["prompt"],stoi,itos,args.max_new_tokens,args.device)
            status,prediction,proof=evaluate_text(completion,row["answer"])
            outputs.append({
                **row,"completion":completion,"stop_reason":stop,
                "strict_status":status,"strict_prediction":prediction,
                "explanation_valid":proof
            })
            if (i+1)%40==0:
                previous=outputs[-40:]
                print(f"{name:14} section={row['section']:12} "
                      f"category={row['category']:12} "
                      f"correct={stats(previous)['correct']:2}/40",flush=True)
        scores=analyze(outputs)
        predictions_by_model[name]=outputs
        report["models"][name]={"checkpoint_sha256":sha(file),
                                "scores":scores}
        with (output/f"{name}_predictions.json").open("x",encoding="utf-8") as f:
            json.dump(outputs,f,indent=2,ensure_ascii=False);f.write("\n")
        print(f"FINAL {name:14} total={scores['overall']['correct']}/680 "
              f"core={scores['by_section']['core400']['correct']}/400 "
              f"extension={scores['by_section']['extension280']['correct']}/280 "
              f"seen={scores['by_seen_status']['seen']['correct']}/"
              f"{scores['by_seen_status']['seen']['n']}, "
              f"unseen={scores['by_seen_status']['unseen']['correct']}/"
              f"{scores['by_seen_status']['unseen']['n']}",flush=True)
        del model
        if args.device=="cuda":torch.cuda.empty_cache()
    if len(selected)==2:
        a,b=selected
        for section in ("all","core400","extension280"):
            left=[r for r in predictions_by_model[a]
                  if section=="all" or r["section"]==section]
            right=[r for r in predictions_by_model[b]
                   if section=="all" or r["section"]==section]
            d=diff(left,right)
            report["paired"][section]={"left":a,"right":b,**d}
            print(f"PAIRED {section}: {a}={d['a_correct']} {b}={d['b_correct']}, "
                  f"{b}-{a}={d['b_correct']-d['a_correct']:+d} "
                  f"({a}-only={d['a_only']}; {b}-only={d['b_only']})")
    with (output/"summary.json").open("x",encoding="utf-8") as f:
        json.dump(report,f,indent=2,ensure_ascii=False);f.write("\n")
    print("SAVED",output/"summary.json")

if __name__=="__main__": main()
'''


In [ ]:
# Exact upstream source: scripts/pipeline_smoke.py
BUNDLED_GENERATOR_SOURCE["scripts/pipeline_smoke.py"] = r'''#!/usr/bin/env python3
"""End-to-end SC3000 DPO pipeline smoke test against the ACTUAL notebook cells.

No permanent training data or final dpo.pt is touched. Runs Notebook Steps 2-8
(except Step 1 pip installs), with a tiny disposable preference dataset.

From repository root, inside the prepared .venv:
    python scripts/pipeline_smoke.py --self-test-only
    python scripts/pipeline_smoke.py --device cpu
    python scripts/pipeline_smoke.py --device cuda --samples 96 --batch-size 16
    python scripts/pipeline_smoke.py --device cuda --out-dir dpo/smoke_outputs/run1

For TC1 GPU use an interactive allocation or sbatch. Do not run CUDA training
on a login node. The sample generator here is ONLY for pipeline verification;
Round 1 will use a separate, fully audited dataset generator.
"""

import argparse
import contextlib
import json
import os
import random
import re
import sys
import tempfile
from collections import Counter
from fractions import Fraction
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
# The final submission notebook is a multi-stage self-contained trainer.
# Earlier experiment scripts still need original course Step2–8 cells;
# keep their reference frozen under a .json filename (only one .ipynb
# exists for the final submission). This fallback preserves old checkout use.
COURSE_REFERENCE = ROOT / "dpo" / "reference_course_notebook_v1.json"
NOTEBOOK = COURSE_REFERENCE if COURSE_REFERENCE.is_file() else ROOT / "dpo" / "dpo.ipynb"
sys.path.insert(0, str(Path(__file__).resolve().parent))
from math_eval_utils import greedy_completion, grade_completion, parse_expected, self_test as test_eval  # noqa: E402

RESERVED_PROBES = {"79-7=?", "x+55=95,x=?", "7/2=?", "5-9=?"}
KINDS = (
    "add", "subtract", "multiply", "divide",
    "x_plus_a", "a_plus_x", "x_minus_a", "a_minus_x",
    "x_times_a", "a_times_x", "x_over_a", "a_over_x",
)
NUM = r"[+-]?\d+(?:\.\d+)?"
ARITH_RE = re.compile(rf"^({NUM})([+*/-])({NUM})=\?$")
EQUATION_RE = re.compile(rf"^({NUM}|x)([+*/-])({NUM}|x)=({NUM}),x=\?$")


def fmt_number(value):
    """Render fractions whose decimal expansion ends within two digits."""
    value = Fraction(value)
    if value.denominator == 1:
        return str(value.numerator)
    scaled = value * 100
    if scaled.denominator != 1:
        raise ValueError(f"Non-terminating or >2-place decimal: {value}")
    v = int(scaled)
    sign = "-" if v < 0 else ""
    whole, decimal = divmod(abs(v), 100)
    return (f"{sign}{whole}.{decimal:02d}").rstrip("0").rstrip(".")


def apply_op(left, op, right):
    if op == "+":
        return left + right
    if op == "-":
        return left - right
    if op == "*":
        return left * right
    if op == "/" and right != 0:
        return left / right
    raise ValueError(f"Invalid math operation or division by zero: {op}")


def verify_math(prompt, answer):
    """Independent verification by parsing the printed question and substitution."""
    target = parse_expected(answer)
    equation = EQUATION_RE.fullmatch(prompt)
    if equation:
        lhs, op, rhs_term, rhs = equation.groups()
        if (lhs == "x") == (rhs_term == "x"):
            raise ValueError(f"Equation does not have exactly one x: {prompt}")
        left = target if lhs == "x" else parse_expected(lhs)
        right = target if rhs_term == "x" else parse_expected(rhs_term)
        return apply_op(left, op, right) == parse_expected(rhs)
    arithmetic = ARITH_RE.fullmatch(prompt)
    if arithmetic:
        left, op, right = arithmetic.groups()
        return apply_op(parse_expected(left), op, parse_expected(right)) == target
    raise ValueError(f"Unrecognized math prompt: {prompt}")


def new_problem(kind, rng, decimal=False):
    a = rng.randint(1, 35)
    b = rng.randint(1, 35)
    x = Fraction(rng.randint(1, 35), 1)
    if decimal:
        x = Fraction(rng.randrange(1, 70, 2), 2)
    if kind == "add":
        left = x if decimal else Fraction(a)
        prompt, result = f"{fmt_number(left)}+{b}=?", left + b
    elif kind == "subtract":
        left = x if decimal else Fraction(a)
        prompt, result = f"{fmt_number(left)}-{b}=?", left - b
    elif kind == "multiply":
        right = rng.randrange(1, 16, 2) if decimal else rng.randint(1, 15)
        left = x if decimal else Fraction(a)
        prompt, result = f"{fmt_number(left)}*{right}=?", left * right
    elif kind == "divide":
        if decimal:
            dividend, divisor = x, rng.choice((1, 2, 5))
            result = dividend / divisor
        else:
            dividend, divisor = a * b, b
            result = Fraction(a)
        prompt = f"{fmt_number(dividend)}/{divisor}=?"
    elif kind == "x_plus_a":
        prompt, result = f"x+{a}={fmt_number(x+a)},x=?", x
    elif kind == "a_plus_x":
        prompt, result = f"{a}+x={fmt_number(a+x)},x=?", x
    elif kind == "x_minus_a":
        prompt, result = f"x-{a}={fmt_number(x-a)},x=?", x
    elif kind == "a_minus_x":
        prompt, result = f"{a}-x={fmt_number(a-x)},x=?", x
    elif kind == "x_times_a":
        prompt, result = f"x*{a}={fmt_number(x*a)},x=?", x
    elif kind == "a_times_x":
        prompt, result = f"{a}*x={fmt_number(a*x)},x=?", x
    elif kind == "x_over_a":
        divisor = rng.choice((1, 2, 5)) if decimal else b
        if not decimal:
            x = Fraction(rng.randint(1, 20) * divisor)
        prompt, result = f"x/{divisor}={fmt_number(x/divisor)},x=?", x
    elif kind == "a_over_x":
        quotient = rng.randint(1, 20)
        prompt, result = f"{fmt_number(x*quotient)}/x={quotient},x=?", x
    else:
        raise ValueError(f"Unknown problem kind {kind}")
    result = fmt_number(result)
    if not verify_math(prompt, result):
        raise AssertionError(f"Invalid positive label: {prompt} -> {result}")
    return prompt, result


def make_pairs(n, seed):
    rng = random.Random(seed)
    records, seen = [], set()
    for i in range(n):
        kind = KINDS[i % len(KINDS)]
        for attempt in range(2000):
            prompt, answer = new_problem(kind, rng, decimal=(i % 7 == 0))
            if prompt not in seen and prompt not in RESERVED_PROBES:
                seen.add(prompt)
                break
        else:
            raise RuntimeError(f"Cannot get a unique prompt for kind {kind}")
        records.append({
            "positive": f"{prompt} The answer is {answer}.",
            "negative": f"{prompt} Sorry, I don't know.",
            "prompt": prompt, "answer": answer, "category": kind,
        })
    return records


def notebook_steps():
    with NOTEBOOK.open(encoding="utf-8") as handle:
        nb = json.load(handle)
    by_step = {}
    current = None
    for cell in nb["cells"]:
        source = cell.get("source", [])
        code = "".join(source) if isinstance(source, list) else source
        if cell["cell_type"] == "markdown":
            found = re.search(r"###\s+Step\s+(\d+)\s*:", code)
            if found:
                current = int(found.group(1))
        elif cell["cell_type"] == "code" and current is not None:
            by_step.setdefault(current, []).append(code)
    for step in range(2, 9):
        if step not in by_step:
            raise RuntimeError(f"Notebook is missing Step {step} code")
    return by_step


def execute_step(step, chunks, globals_dict):
    for i, source in enumerate(chunks[step]):
        if source.lstrip().startswith(("!", "%")):
            raise RuntimeError(f"Cannot execute notebook shell/magic at Step {step}")
        exec(compile(source, f"{NOTEBOOK}:step{step}:cell{i}", "exec"), globals_dict, globals_dict)


def compare_predictions(model, stoi, itos, prompts, token_budget, device):
    model.eval()
    rows = []
    for prompt, answer in prompts:
        missing = sorted(set(prompt) - set(stoi))
        if missing:
            raise ValueError(f"Evaluation prompt has unknown characters: {missing}")
        completion, stop = greedy_completion(
            model, [stoi[ch] for ch in prompt], itos, token_budget, device
        )
        status, prediction = grade_completion(completion, answer)
        rows.append({"prompt": prompt, "expected": answer, "completion": completion,
                     "prediction": prediction, "status": status, "stop_reason": stop})
    return rows


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--self-test-only", action="store_true")
    parser.add_argument("--device", choices=("auto", "cpu", "cuda"), default="auto")
    parser.add_argument("--samples", type=int, default=48)
    parser.add_argument("--batch-size", type=int, default=16)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--max-new-tokens", type=int, default=40)
    parser.add_argument("--out-dir", help="Optional NEW directory for persistent smoke artifacts")
    args = parser.parse_args()

    test_eval()
    for record in make_pairs(240, 42):
        assert verify_math(record["prompt"], record["answer"])
        assert record["positive"].startswith(record["prompt"])
        assert record["negative"].startswith(record["prompt"])
    print("PASS: all 12 synthetic problem templates, exact answers, unique prompts")
    if args.self_test_only:
        print("SELF-TEST PASSED (model not loaded; no gradient steps run)")
        return

    if args.samples < 12 or args.batch_size < 1 or args.samples % args.batch_size:
        parser.error("Use samples >= 12, batch_size >= 1, and samples divisible by batch_size")
    if args.max_new_tokens < 2:
        parser.error("--max-new-tokens must be >= 2")
    if not (ROOT / "sft" / "gpt.pt").is_file():
        parser.error("Missing trusted course checkpoint: sft/gpt.pt")
    if not (ROOT / "sft" / "meta.pkl").is_file():
        parser.error("Missing course tokenizer: sft/meta.pkl")

    import torch

    device = ("cuda" if torch.cuda.is_available() else "cpu") if args.device == "auto" else args.device
    if device == "cuda" and not torch.cuda.is_available():
        parser.error("CUDA unavailable: use an allocated GPU Slurm job or --device cpu")
    steps = notebook_steps()
    records = make_pairs(args.samples, args.seed)

    # Four independent probes; never use their prompts for smoke training.
    probes = [("79-7=?", "72"), ("x+55=95,x=?", "40"),
              ("7/2=?", "3.5"), ("5-9=?", "-4")]
    train_prompts = {row["prompt"] for row in records}
    if train_prompts.intersection(prompt for prompt, _ in probes):
        raise AssertionError("Smoke probes unexpectedly overlap training data")

    os.chdir(ROOT)  # Notebook Step 2 deliberately checks cwd.
    with contextlib.ExitStack() as stack:
        if args.out_dir:
            out = Path(args.out_dir).expanduser().resolve()
            smoke_root = (ROOT / "dpo" / "smoke_outputs").resolve()
            if out == smoke_root or not out.is_relative_to(smoke_root):
                parser.error("--out-dir must be a new subdirectory of dpo/smoke_outputs/")
            out.mkdir(parents=True, exist_ok=False)
        else:
            out = Path(stack.enter_context(tempfile.TemporaryDirectory(prefix="sc3000_dpo_smoke_")))
        (out / "pos_neg_pairs.json").write_text(
            json.dumps([{"positive": r["positive"], "negative": r["negative"]}
                        for r in records], ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
        )

        env = {"__name__": "__dpo_smoke__"}
        print("=== Executing the actual dpo.ipynb Step 2 ===")
        execute_step(2, steps, env)
        env["device"] = device
        env["batch_size"] = args.batch_size
        env["epochs"] = 1
        env["dpo_dir"] = out  # prevents final data/checkpoint overwrite
        print(f"Device={device} | disposable pairs={len(records)} | batch={args.batch_size}")

        print("=== Executing Notebook Steps 3-4: helpers and baseline weights ===")
        execute_step(3, steps, env)
        execute_step(4, steps, env)
        assert int(env["gpt"].config.block_size) == 256, "Unexpected model context length"
        assert env["gpt"].config.vocab_size == len(env["stoi"])
        for row in records:
            for field in ("positive", "negative"):
                unknown = set(row[field]) - set(env["stoi"])
                if unknown:
                    raise ValueError(f"OOV characters in {field}: {unknown}")
                if len(row[field]) + 4 > env["gpt"].config.block_size:
                    raise ValueError("Sample exceeds model block_size")
        print("PASS: tokenizer, model config, full text sequence lengths")

        itos, stoi = env["itos"], env["stoi"]
        before = compare_predictions(env["gpt"], stoi, itos, probes, args.max_new_tokens, device)
        tracked_parameter = env["gpt"].transformer.h[0].attn.c_attn.weight
        before_parameter = tracked_parameter.detach().cpu().clone()

        print("=== Executing Notebook Steps 5-7: data, optimizer and actual training ===")
        execute_step(5, steps, env)
        assert env["data_path"].resolve() == (out / "pos_neg_pairs.json").resolve()
        assert len(env["lines"]) == len(records)
        execute_step(6, steps, env)
        assert env["steps_per_epoch"] == args.samples // args.batch_size
        assert env["epochs"] == 1 and env["dpo_dir"] == out
        execute_step(7, steps, env)
        assert len(env["training_loss"]) == 1
        assert all(__import__("math").isfinite(v) for v in env["training_loss"])
        delta = (tracked_parameter.detach().cpu() - before_parameter).abs().max().item()
        if delta <= 0:
            raise AssertionError("No parameter update detected")
        print(f"PASS: gradient updates, finite loss, optimizer/scheduler | max parameter delta={delta:.8g}")

        print("=== Executing Notebook Step 8: checkpoint loading ===")
        checkpoint_path = out / "dpo.pt"
        assert checkpoint_path.is_file() and checkpoint_path.stat().st_size > 0
        expected_after = tracked_parameter.detach().cpu().clone()
        execute_step(8, steps, env)  # Current Notebook only reloads; evaluation is TODO.
        restored = env["gpt"].transformer.h[0].attn.c_attn.weight.detach().cpu()
        if not torch.equal(expected_after, restored):
            raise AssertionError("Checkpoint reload does not reproduce trained weights")
        print("PASS: dpo.pt saved and reloaded exactly using Notebook Step 8")
        after = compare_predictions(env["gpt"], stoi, itos, probes, args.max_new_tokens, device)

        for b, a in zip(before, after):
            print(f"PROBE {b['prompt']!r} (answer {b['expected']}):")
            print(f"  baseline: {b['completion']!r} [{b['status']}, {b['stop_reason']}]")
            print(f"  trained:  {a['completion']!r} [{a['status']}, {a['stop_reason']}]")

        report = {"notebook": str(NOTEBOOK), "device": device, "samples": len(records),
                  "batch_size": args.batch_size, "train_steps": env["steps_per_epoch"],
                  "training_loss": env["training_loss"], "parameter_max_change": delta,
                  "categories": dict(Counter(r["category"] for r in records)),
                  "baseline_probes": before, "post_smoke_probes": after,
                  "checkpoint_bytes": checkpoint_path.stat().st_size,
                  "note": "Pipeline only: not a formal accuracy baseline or Round 1."}
        (out / "smoke_report.json").write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
        print("=== PIPELINE SMOKE TEST PASSED ===")
        print(f"Artifacts: {out}")
        if not args.out_dir:
            print("Temporary artifacts will be removed; use --out-dir to keep them.")


if __name__ == "__main__":
    main()
'''


In [ ]:
# Exact upstream source: scripts/math_eval_utils.py
BUNDLED_GENERATOR_SOURCE["scripts/math_eval_utils.py"] = r'''"""Shared exact-numeric grading and decimal-safe greedy decoding for NanoGPT-Math.

The course model's generate() stops at the first '.', which truncates decimals.
These helpers leave the supplied model.py unchanged.
"""

from fractions import Fraction
import re

NUMBER_PATTERN = re.compile(r"[+-]?\d+(?:\.\d+)?\Z")
ANSWER_PATTERN = re.compile(
    r"^\s*(?:The answer is\s+)?([+-]?\d+(?:\.\d+)?)(?=\s*(?:\.(?!\d)|because\b|$))",
    flags=re.IGNORECASE,
)
REFUSAL_PATTERN = re.compile(
    r"\b(?:sor+y|unknown)\b|\b(?:do not know|don't know|cannot answer|can't answer)\b",
    flags=re.IGNORECASE,
)


def parse_expected(value):
    """Parse an integer or finite decimal exactly, without binary floats."""
    s = str(value).strip()
    if not NUMBER_PATTERN.fullmatch(s):
        raise ValueError(f"Expected an integer or finite decimal, got {value!r}")
    return Fraction(s)


def grade_completion(completion, expected):
    """Return (correct/math_error/refusal/format_failure, parsed_answer_or_None)."""
    match = ANSWER_PATTERN.match(completion)
    if match:
        answer = match.group(1)
        result = "correct" if Fraction(answer) == parse_expected(expected) else "math_error"
        return result, answer
    if REFUSAL_PATTERN.search(completion):
        return "refusal", None
    return "format_failure", None


def decode_from_next(predict_next, prompt_ids, itos, max_new_tokens):
    """Greedy token streaming. Peek after a numeric '.' to distinguish 1.5 from 1.

    predict_next(input_ids) -> int, using the already-emitted token IDs.
    A non-digit lookahead is discarded, just like the original period stop.
    """
    dot_id = next((i for i, ch in itos.items() if ch == "."), None)
    newline_id = next((i for i, ch in itos.items() if ch == "\n"), None)
    digit_ids = {i for i, ch in itos.items() if ch in "0123456789"}
    if dot_id is None or newline_id is None or max_new_tokens <= 0:
        raise ValueError("Tokenizer must contain '.' and newline, and token budget must be positive")

    context = list(prompt_ids)
    generated = []
    stop_reason = "max_new_tokens"
    while len(generated) < max_new_tokens:
        token = int(predict_next(context))
        if token == newline_id:
            stop_reason = "newline"
            break
        prev_is_digit = bool(generated) and generated[-1] in digit_ids
        context.append(token)
        generated.append(token)
        if token == dot_id:
            if prev_is_digit and len(generated) < max_new_tokens:
                next_token = int(predict_next(context))
                if next_token in digit_ids:
                    context.append(next_token)
                    generated.append(next_token)
                    continue
            stop_reason = "period"
            break
    return "".join(itos[token] for token in generated), stop_reason


def greedy_completion(model, prompt_ids, itos, max_new_tokens, device):
    import torch

    block_size = model.config.block_size

    def next_token(context):
        ids = torch.tensor([context[-block_size:]], dtype=torch.long, device=device)
        logits, _ = model(ids)
        return int(logits[0, -1].argmax().item())

    with torch.inference_mode():
        return decode_from_next(next_token, prompt_ids, itos, max_new_tokens)


def self_test():
    assert parse_expected("1.50") == parse_expected("1.5")
    assert parse_expected("-3") == Fraction(-3)
    for bad in ("1/3", "nan", "1e4", "1.2.3"):
        try:
            parse_expected(bad)
        except ValueError:
            pass
        else:
            raise AssertionError(f"Expected rejection of {bad!r}")
    assert grade_completion(" The answer is 1.50.", "1.5")[0] == "correct"
    assert grade_completion(" The answer is 1.", "1.5")[0] == "math_error"
    assert grade_completion(" The answer is 1.5.0", "1.5")[0] == "format_failure"
    assert grade_completion(" Sorry, I don't know.", "2")[0] == "refusal"
    assert grade_completion(" Something else.", "2")[0] == "format_failure"
    assert grade_completion(" The answer is -4.", "-4")[0] == "correct"

    chars = "\n .-0123456789The answer isSory,Idn'tkow"
    unique = list(dict.fromkeys(chars))
    itos = dict(enumerate(unique))
    stoi = {v: k for k, v in itos.items()}
    for answer, expected, outcome in (
        (" The answer is 1.5.", "1.5", "correct"),
        (" The answer is -4.", "-4", "correct"),
        (" Sorry, I don't know.", "2", "refusal"),
    ):
        scripted = [stoi[x] for x in answer]
        prefix = [stoi["T"]]

        def scripted_predict(tokens):
            i = len(tokens) - len(prefix)
            return scripted[i] if i < len(scripted) else stoi["\n"]

        completion, stop = decode_from_next(scripted_predict, prefix, itos, 50)
        assert (completion, stop) == (answer, "period"), (completion, stop)
        assert grade_completion(completion, expected)[0] == outcome
    print("PASS: decimal-aware stopping, exact decimal grading, refusal grading")


if __name__ == "__main__":
    self_test()
'''


In [ ]:
# Verify bundled source files BEFORE Step 5 data generation. Never overwrite
# a source with different contents; doing so would alter the experiment.
for relative, embedded_text in BUNDLED_GENERATOR_SOURCE.items():
    path=repo_root/relative
    if path.exists():
        if path.read_text(encoding="utf-8")!=embedded_text:
            raise RuntimeError(
                f"Generator source has changed since this notebook was frozen: {relative}. "
                "Use the matching Git revision; do not silently rebuild data.")
    else:
        path.parent.mkdir(parents=True,exist_ok=True)
        path.write_text(embedded_text,encoding="utf-8")
    compile(embedded_text,str(path),"exec")
print(f"BUNDLED GENERATOR PASS: {len(BUNDLED_GENERATOR_SOURCE)} source files "
      f"reconciled; no generator modification",flush=True)

#### Step 5B — Generate and load the actual training data

The four demo preference pairs from the course are insufficient for training. This project generates the full 100,000-row preference data for Round3, Round4 Treatment, Round5 Original and Round7 Replay, automatically rebuilding only missing datasets and checking SHA256 manifests. **Round9 Random needs the learned Round7 model and is prepared later during Step 7.** No frozen data or checkpoint is overwritten. The sealed Test is excluded from evaluation.

In [ ]:
# Step 5 — Generate and validate missing historical preference datasets
# All builder programs live in this same GitHub repository, and their
# exact source SHA256 is stored in the final run provenance.
GENERATOR_PLAN = [
    ("Round1A frozen source", repo_root/"dpo/data/round1_full/round1_manifest.json",
     "scripts/generate_round1.py", ["--profile", "full"]),
    ("Round1B integer source", repo_root/"dpo/experiments/round1b_integer/round1b_manifest.json",
     "scripts/generate_round1b.py", []),
    ("Round1C fixed validation", repo_root/"dpo/experiments/round1c_incorrect/round1c_manifest.json",
     "scripts/generate_round1c.py", []),
    ("Round3 structured 100k", repo_root/"dpo/data/round3_structured_full/manifest.json",
     "scripts/generate_round3_structured.py", ["--profile", "full"]),
    ("Round4 matched data", repo_root/"dpo/data/round4_matched_full/manifest.json",
     "scripts/build_round4_data.py", ["--profile", "full"]),
    ("Round5 one-step data", repo_root/"dpo/data/round5_decomposition/manifest.json",
     "scripts/build_round5_decomposition.py", []),
    ("Round7 paired replay", repo_root/"dpo/data/round7_targeted_full/manifest.json",
     "scripts/build_round7_targeted_data.py", ["--profile", "full"]),
]
generator_provenance = {}

for name, manifest_path, script_rel, args in GENERATOR_PLAN:
    script = repo_root / script_rel
    if not script.is_file():
        raise FileNotFoundError(f"Reproducibility requires source generator {script_rel}")
    generator_provenance[script_rel] = sha256(script)
    if not manifest_path.exists():
        if not AUTO_BUILD_MISSING_DATA:
            raise FileNotFoundError(f"{name}: missing frozen source {manifest_path}")
        print(f"GENERATE {name}: {script_rel} {args}", flush=True)
        subprocess.run([sys.executable, str(script), *args],
                       cwd=repo_root, check=True)
    if not manifest_path.is_file():
        raise RuntimeError(f"{name} generation did not produce {manifest_path}")
    generator_provenance[str(manifest_path.relative_to(repo_root))] = sha256(manifest_path)
    print(f"PREPARED {name}: {manifest_path.relative_to(repo_root)}", flush=True)

# Frozen teacher-style 680 is for retention diagnostics only; its
# arithmetic facts were historically seen, never to be called unseen OOD.
TEACHER_DIRS = [
    (repo_root/"dpo/benchmarks/teacher_style_400_v1", 400),
    (repo_root/"dpo/benchmarks/teacher_style_extension_280_v1", 280),
]
for directory, expected_n in TEACHER_DIRS:
    questions = read_json(directory/"questions.json")
    manifest = read_json(directory/"manifest.json")
    assert len(questions) == expected_n
    assert sha256(directory/"questions.json") == manifest["questions_sha256"]
    generator_provenance[str((directory/"questions.json").relative_to(repo_root))] = sha256(directory/"questions.json")
print("PASS: 400 + 280 frozen teacher questions", flush=True)

### Step 6: Build the optimizer and scheduler (**students are required to complete this part!**)

#### Step 6 — Preserve optimizer resets and scheduler horizons

Although the original teacher template builds **one** optimizer, our selected E5 follows a documented sequence. The helper above uses the same `AdamW(weight_decay=0.01)` and `CosineAnnealingLR(eta_min=0.1 × lr)` but constructs **a new optimizer at each historical stage**. Round4 saves the first epoch of a two-epoch cosine horizon, while Random E5 is the third epoch of a four-epoch cosine horizon. Collapsing these into five epochs from SFT would change the experiment.

In [ ]:
# Step 6 — Historical optimizer and scheduler hyperparameters
R3 = repo_root / "dpo/data/round3_structured_full"
R4 = repo_root / "dpo/data/round4_matched_full"
R5 = repo_root / "dpo/data/round5_decomposition"
R7 = repo_root / "dpo/data/round7_targeted_full"
R9 = repo_root / "dpo/data/round9_balanced_full"

STAGE1_TO_4=[
    {"id":"round3_structured", "data":R3/"pos_neg_pairs.json",
     "manifest":R3/"manifest.json","hash_section":"sha256","hash_key":"train",
     "lr":1e-4,"actual_epochs":5,"schedule_epochs":5},
    {"id":"round4_treatment_e1","data":R4/"treatment_pairs.json",
     "manifest":R4/"manifest.json","hash_section":"file_sha256","hash_key":"treatment",
     "lr":2e-5,"actual_epochs":1,"schedule_epochs":2},
    {"id":"round5_original","data":R5/"original_pairs.json",
     "manifest":R5/"manifest.json","hash_section":"file_sha256","hash_key":"original",
     "lr":1e-5,"actual_epochs":1,"schedule_epochs":1},
    {"id":"round7_replay","data":R7/"replay_pairs.json",
     "manifest":R7/"manifest.json","hash_section":"file_sha256","hash_key":"replay",
     "lr":1e-5,"actual_epochs":1,"schedule_epochs":1},
]
EXPECTED_RANDOM_SHA="30519fe6883f175e67791a0461afe23e9212f38d974778c4bd5f552d3e26e3fa"

# Basic contract checks (also repeated in the resumable stage runner).
assert [x["id"] for x in STAGE1_TO_4]==[
    "round3_structured","round4_treatment_e1","round5_original","round7_replay"]
assert [x["actual_epochs"] for x in STAGE1_TO_4]==[5,1,1,1]
assert [x["schedule_epochs"] for x in STAGE1_TO_4]==[5,2,1,1]
print("PASS: 4 historical stages and their original cosine schedules")

In [ ]:
# Step 6 — Instructor's original AdamW + cosine scheduler, made reusable
# because the chosen E5 was trained in SEVEN sequential stages.
def build_optimizer_scheduler(gpt,base_lr,steps_per_epoch,schedule_epochs):
    optimizer = torch.optim.AdamW(gpt.parameters(),lr=base_lr,weight_decay=0.01)
    total_steps=schedule_epochs*steps_per_epoch
    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,T_max=total_steps,eta_min=base_lr*0.1)
    return optimizer,scheduler

print("Step 6 PASS: original AdamW, cosine annealing, fresh restart at each stage.")

### Step 7: Begin training (**students are required to complete this part!**)
    
**Two reproducible loss modes:** `course` preserves the instructor's original
whole-sequence mean objective. `answer_focused` uses completion-only
preference sums plus extra correct-number and completion NLL terms. The latter
is a controlled additional experiment and is not standard reference-model DPO.


#### Step 7 — Historical fine-tuning from SFT to Random E5

Following Step 6, train with the original Course Preference Loss (whole-sequence mean log probability; beta=0.5, Chosen bonus 0.1), character-level Teacher Forcing, `random.shuffle`, four newline padding characters, 64 examples per minibatch and gradient clipping at 1.0. Exactly 1,562 full minibatches per 100k-row epoch; the final 32 rows are skipped, consistent with the teacher helper.

The code below trains **13 epochs through seven stages**, including two one-epoch Random continuations with fresh optimizers, and E3–E5 with a continuous optimizer. It saves weights, AdamW state, cosine state and RNG state at each epoch. Rerun **Restart & Run All** if a GPU allocation ends; completed verified epochs are skipped.

In [ ]:
# Step 7 — Course Loss training loop with epoch-boundary resume.
def atomic_save_torch(target, obj):
    target=Path(target)
    if target.exists():
        raise FileExistsError(f"Refusing to overwrite checkpoint: {target}")
    tmp=target.with_suffix(target.suffix+".partial")
    if tmp.exists():
        raise RuntimeError(f"Incomplete temporary checkpoint: {tmp}. Inspect first.")
    torch.save(obj, tmp)
    tmp.rename(target)

def stage_paths(stage_id):
    folder=RUN_DIR/stage_id
    return folder,folder/"progress.json"

def stage_finished_checkpoint(spec):
    folder,_=stage_paths(spec["id"])
    n=1 if FAST_SMOKE else spec["actual_epochs"]
    return folder/f"epoch_{n:02d}.pt"

def train_stage(spec, parent_path):
    assert Path(parent_path).is_file(),f"Missing parent checkpoint {parent_path}"
    data_file=Path(spec["data"])
    source_m=read_json(spec["manifest"])
    expected=source_m[spec["hash_section"]][spec["hash_key"]]
    if sha256(data_file)!=expected:
        raise RuntimeError(f"Data SHA256 does not match frozen source manifest: {data_file}")
    allrows,info=verify_pairs(data_file,smoke=FAST_SMOKE)
    batch=16 if FAST_SMOKE else BATCH_SIZE
    actual=1 if FAST_SMOKE else spec["actual_epochs"]
    schedule_epochs=spec["schedule_epochs"]
    steps=len(allrows)//batch
    if steps<=0:raise RuntimeError("Not enough preferences for one minibatch")
    protocol={
        "stage":spec["id"],"parent_sha256":sha256(parent_path),
        "dataset_sha256":info["sha256"],
        "source_manifest_sha256":sha256(spec["manifest"]),
        "seed":SEED,"batch_size":batch,"steps_per_epoch":steps,
        "actual_epochs":actual,"cosine_schedule_epochs":schedule_epochs,
        "learning_rate":spec["lr"],"loss":"instructor full-sequence mean",
        "max_length":info["max_length"],"beta":BETA,
        "model_block_size":256,"fast_smoke":FAST_SMOKE
    }
    stage_dir,progress_path=stage_paths(spec["id"])
    if progress_path.is_file():
        progress=read_json(progress_path)
        if progress["protocol"]!=protocol:
            raise RuntimeError("Refusing incompatible resume: stage data/schedule/parent changed")
        finished=progress["completed_epochs"]
        if not 1<=finished<=actual:
            raise RuntimeError("Invalid saved epoch offset")
        epoch_file=stage_dir/f"epoch_{finished:02d}.pt"
        if sha256(epoch_file)!=progress["last_checkpoint_sha256"]:
            raise RuntimeError(f"Corrupt stage checkpoint {epoch_file}")
        if finished==actual:
            print(f"SKIP verified {spec['id']}: {finished}/{actual} complete",flush=True)
            return epoch_file
        print(f"RESUME {spec['id']} from epoch {finished}/{actual}",flush=True)
    elif stage_dir.exists():
        # A failed allocation before the first optimizer update may have
        # written protocol.json but no checkpoint yet. Restart that *empty*
        # stage safely without overwriting any saved model.
        existing={p.name for p in stage_dir.iterdir()}
        if existing=={"protocol.json"} and read_json(stage_dir/"protocol.json")==protocol:
            finished=0
            progress=None
            print(f"RESTART EMPTY STAGE {spec['id']}: no completed epoch lost",flush=True)
        else:
            raise RuntimeError(f"Stale/incomplete stage folder: {stage_dir}; "
                               "inspect or rename; never overwrite saved epochs")
    else:
        stage_dir.mkdir(parents=True,exist_ok=False)
        save_json_once(stage_dir/"protocol.json",protocol)
        finished=0
        progress=None

    # Previous Python/CUDA seed reset BEFORE constructing GPT, just like all
    # historical train_round*.py stage entry points.
    random.seed(SEED)
    torch.manual_seed(SEED)
    if device=="cuda":torch.cuda.manual_seed_all(SEED)
    gpt, model_args=model_from_checkpoint(parent_path)
    if model_args["vocab_size"]!=len(stoi) or model_args["block_size"]!=256:
        raise RuntimeError("Original character tokenizer / model architecture changed")
    gpt.to(device).train()
    opt,scheduler=build_optimizer_scheduler(gpt,spec["lr"],steps,schedule_epochs)
    lines=allrows

    if finished:
        ck=torch.load(epoch_file,map_location="cpu",weights_only=True)
        if ck["stage_protocol"]!=protocol or ck["epoch"]!=finished:
            raise RuntimeError("Resume checkpoint metadata mismatch")
        gpt.load_state_dict(ck["model_state_dict"],strict=True)
        opt.load_state_dict(ck["optimizer_state_dict"])
        scheduler.load_state_dict(ck["scheduler_state_dict"])
        lines=ck["shuffled_lines"]
        random.setstate(ck["python_rng_state"])
        torch.set_rng_state(ck["torch_rng_state"])
        if device=="cuda":
            torch.cuda.set_rng_state_all(ck["cuda_rng_states"])
        if len(lines)!=len(allrows):
            raise RuntimeError("Saved shuffle order lost rows")

    for epoch in range(finished+1,actual+1):
        gpt.train()
        accumulated=0.
        updated=0
        for step,(neg,pos) in enumerate(get_batches(lines,batch,info["max_length"],device),1):
            opt.zero_grad(set_to_none=True)
            loss=course_preference_loss(gpt,neg,pos)
            if not bool(torch.isfinite(loss).item()):
                raise RuntimeError(f"Non-finite loss: {spec['id']} epoch={epoch} step={step}")
            loss.backward()
            torch.nn.utils.clip_grad_norm_(gpt.parameters(),max_norm=1.0)
            opt.step()
            scheduler.step()
            updated+=1
            accumulated+=float(loss.item())
            if step==1 or step%250==0 or step==steps:
                print(f"TRAIN {spec['id']} epoch={epoch}/{actual} "
                      f"step={step}/{steps} loss={loss.item():.5f} "
                      f"lr={scheduler.get_last_lr()[0]:.9g}",flush=True)
        if updated!=steps:raise RuntimeError("Optimizer step count mismatch")
        ckpt=stage_dir/f"epoch_{epoch:02d}.pt"
        obj={
            "epoch":epoch,"stage_protocol":protocol,
            "model_state_dict":gpt.state_dict(),"model_args":model_args,
            "optimizer_state_dict":opt.state_dict(),
            "scheduler_state_dict":scheduler.state_dict(),
            "shuffled_lines":lines,"python_rng_state":random.getstate(),
            "torch_rng_state":torch.get_rng_state(),
            "cuda_rng_states":torch.cuda.get_rng_state_all() if device=="cuda" else []
        }
        atomic_save_torch(ckpt,obj)
        history=progress["history"] if progress else []
        history.append({"epoch":epoch,"mean_loss":accumulated/steps,
                        "checkpoint":ckpt.name,"sha256":sha256(ckpt)})
        progress={"protocol":protocol,"completed_epochs":epoch,
                  "last_checkpoint_sha256":sha256(ckpt),"history":history}
        temp=progress_path.with_suffix(".json.partial")
        if temp.exists():raise RuntimeError(f"Stale progress marker: {temp}")
        temp.write_text(json.dumps(progress,indent=2)+"\n",encoding="utf-8")
        temp.replace(progress_path)
        print(f"EPOCH SAVED {spec['id']} {epoch}/{actual} "
              f"mean_loss={accumulated/steps:.6f}",flush=True)
    print(f"STAGE PASS {spec['id']} -> {ckpt.name}",flush=True)
    return ckpt

# Verify the original SFT config without changing any model weights.
print("Step 7 training engine PASS: loss, optimizer, checkpoint and RNG resume ready",flush=True)

In [ ]:
# Step 7A — Train/reuse Round3, Round4 E1, Round5 Original, Round7 Replay
stage_outputs={"sft":SFT}
parent=SFT
for spec in STAGE1_TO_4:
    parent=train_stage(spec,parent)
    stage_outputs[spec["id"]]=parent
    print(f"CHAIN {spec['id']}: SHA256 {sha256(parent)[:20]}",flush=True)

ROUND7_PARENT=stage_outputs["round7_replay"]
print(f"PASS: Round7 Replay predecessor created/verified: {ROUND7_PARENT}",flush=True)

#### Step 7B — Build/verify Random data after Round7 Replay

The Round9 Random generator depends on the historical Round7 checkpoint; its 100k pair file has frozen SHA256 `30519fe6883f175e67791a0461afe23e9212f38d974778c4bd5f552d3e26e3fa`. Reuse the file if present; otherwise rebuild in an isolated run folder. A changed SHA raises an error.

In [ ]:
# Step 7B — Verify/build the Round9 Random 100k training dataset
EXISTING_R9_DATA=R9/"random_pairs.json"
if EXISTING_R9_DATA.exists():
    r9_manifest=read_json(R9/"manifest.json")
    if r9_manifest["version"]!="round9_structured_v1":
        raise RuntimeError("Wrong historical Round9 manifest")
    if sha256(EXISTING_R9_DATA)!=r9_manifest["files_sha256"]["random"]:
        raise RuntimeError("Historical Random100k data SHA256 mismatch")
    random_data_path=EXISTING_R9_DATA
    random_manifest_path=R9/"manifest.json"
else:
    if not AUTO_BUILD_MISSING_DATA:
        raise FileNotFoundError("Missing Round9 Random preference dataset")
    import build_round9_data as b9
    import evaluate_teacher_style_680 as teacher680_module
    # Strictly validate both existing frozen teacher benchmarks.
    teacher680_module.validate()
    generated=RUN_DIR/"notebook_generated_round9"
    if generated.exists():
        if not (generated/"random_pairs.json").exists():
            raise RuntimeError("Incomplete generated Round9 folder; inspect it")
        random_data_path=generated/"random_pairs.json"
        random_manifest_path=generated/"manifest.json"
    else:
        # The original builder expects the parent at a historical absolute
        # path. Use our new Round7 checkpoint, without copying it there.
        historical_base=b9.BASE
        try:
            b9.BASE=ROUND7_PARENT
            r9_pairs,r9_metadata,r9_manifest=b9.make()
        finally:
            b9.BASE=historical_base
        generated.mkdir(parents=True,exist_ok=False)
        for arm in ("replay","random","structured"):
            (generated/f"{arm}_pairs.json").write_bytes(b9.bytes_json(r9_pairs[arm]))
        (generated/"structured_metadata.json").write_bytes(
            b9.bytes_json(r9_metadata))
        (generated/"manifest.json").write_bytes(b9.bytes_json(r9_manifest))
        random_data_path=generated/"random_pairs.json"
        random_manifest_path=generated/"manifest.json"

r9_manifest=read_json(random_manifest_path)
if sha256(random_data_path)!=r9_manifest["files_sha256"]["random"]:
    raise RuntimeError("Generated Random source mismatch with its own manifest")
if sha256(random_data_path)!=EXPECTED_RANDOM_SHA:
    raise RuntimeError(
        "Round9 Random data do NOT match archived E5 training source. "
        "Do not publish an alleged historical reproduction with changed data. "
        f"observed={sha256(random_data_path)} expected={EXPECTED_RANDOM_SHA}")
print(f"PASS: historical Random100k exact pair SHA={sha256(random_data_path)}")
print(f"Dataset location: {random_data_path}",flush=True)

In [ ]:
# Step 7C — Continue original staged training until selected E5
STAGE5_TO_7=[
    {"id":"round9_random_e1", "data":random_data_path,
     "manifest":random_manifest_path,"hash_section":"files_sha256","hash_key":"random",
     "lr":1e-5,"actual_epochs":1,"schedule_epochs":1},
    {"id":"round9_random_e2_restart", "data":random_data_path,
     "manifest":random_manifest_path,"hash_section":"files_sha256","hash_key":"random",
     "lr":1e-5,"actual_epochs":1,"schedule_epochs":1},
    {"id":"round9_random_e3_e5_from_e2", "data":random_data_path,
     "manifest":random_manifest_path,"hash_section":"files_sha256","hash_key":"random",
     "lr":1e-5,"actual_epochs":3,"schedule_epochs":4}
]
assert [s["actual_epochs"] for s in STAGE5_TO_7]==[1,1,3]
assert [s["schedule_epochs"] for s in STAGE5_TO_7]==[1,1,4]

parent=ROUND7_PARENT
for spec in STAGE5_TO_7:
    parent=train_stage(spec,parent)
    stage_outputs[spec["id"]]=parent
    print(f"CHAIN {spec['id']} SHA256={sha256(parent)[:20]}",flush=True)

REPRODUCED_E5=stage_outputs["round9_random_e3_e5_from_e2"]
print(f"FINAL E5 REPLAY CHECKPOINT = {REPRODUCED_E5}",flush=True)

### Step 8: Begin testing (**students are required to complete this part!**)

#### Step 8 — Testing, comparisons and results analysis

The original Notebook generates and prints math answers in Step 8. We use the **same greedy character generation**, stop at `.` or newline, and evaluate the first explicitly marked integer. Explanation arithmetic is checked separately.

Frozen development sets: **Val768** (canonical fact-disjoint), **TrainProbe256** (training probe), **Teacher680** (historical seen-fact retention), and five reserved examples. The sealed final Test is not read or tuned on. Historical E5 development scores were 614/768 Val, 521/680 Teacher and 226/256 TrainProbe; newly measured scores are reported separately.

**Interpretation:** Round9 Random E1→E5 improved development accuracy; later Round10 targeted carry-heavy arithmetic fixed some specific examples (e.g. 3×17) but did not improve overall Val768 and Teacher680 relative to ordinary replay. The selected model remains **Random E5**. Repeated math facts and the original length-normalized full-sequence loss may limit numeric generalization. No claim is made about the hidden grading set.

In [ ]:
# Step 8 — Original greedy generation and answer-first evaluation
ANSWER_RE=re.compile(r"^\s*The answer is\s+([+-]?\d+)(?=\s|[.]|$)",re.I)
EXPLANATION_RE=re.compile(
    r"^\s*The answer is\s+([+-]?\d+)\s+because\s+"
    r"([+-]?\d+)([+\-*/])([+-]?\d+)\s+equals\s+([+-]?\d+)\.\s*$",re.I)

def greedy_original_stop(model,prompt,limit=80):
    tokens=encode(prompt)
    if len(tokens)>model.config.block_size:
        raise RuntimeError("Evaluation prompt exceeds NanoGPT context")
    produced=[]
    stop="max_new_tokens"
    with torch.inference_mode():
        for _ in range(limit):
            x=torch.tensor([tokens[-model.config.block_size:]],
                           dtype=torch.long,device=device)
            logits,_=model(x)
            new_id=int(logits[0,-1].argmax().item())
            if new_id==0:
                stop="newline";break
            tokens.append(new_id)
            produced.append(new_id)
            if new_id==7:
                stop="period";break
    return "".join(itos[t] for t in produced),stop

def grade_completion(response,gold):
    m=ANSWER_RE.match(response)
    if m is None:return "format_failure",None,False
    got=int(m.group(1))
    status="correct" if got==int(gold) else "math_error"
    proof=EXPLANATION_RE.fullmatch(response)
    valid=False
    if proof:
        a,op,b,rhs=int(proof[2]),proof[3],int(proof[4]),int(proof[5])
        if op=="+":value=a+b
        elif op=="-":value=a-b
        elif op=="*":value=a*b
        else:value=(a/b if b else None)
        valid=(value==rhs==got)
    return status,got,valid

def evaluate_frozen(model,records,split):
    output=[]
    for i,item in enumerate(records):
        gold=str(item["answer"])
        response,stop=greedy_original_stop(model,item["prompt"],MAX_NEW_TOKENS)
        status,predicted,proof=grade_completion(response,gold)
        output.append({"prompt":item["prompt"],"answer":gold,
                       "category":item["category"],"completion":response,
                       "status":status,"prediction":predicted,
                       "stop_reason":stop,"explanation_valid":proof})
        if i<5:
            print(f"{split}: {item['prompt']} gold={gold} "
                  f"model={response!r} [{status}]",flush=True)
    n=len(output)
    count=sum(x["status"]=="correct" for x in output)
    by_cat={}
    for category in sorted({x["category"] for x in output}):
        rows=[x for x in output if x["category"]==category]
        by_cat[category]={"correct":sum(x["status"]=="correct" for x in rows),
                          "n":len(rows)}
    metrics={"correct":count,"n":n,"accuracy":count/n,"by_category":by_cat,
             "format_failure":sum(x["status"]=="format_failure" for x in output),
             "valid_explanations":sum(x["explanation_valid"] for x in output)}
    print(f"EVAL {split}: {count}/{n} ({count/n:.2%}) "
          f"format_failure={metrics['format_failure']}",flush=True)
    return output,metrics

r3_manifest=read_json(R3/"manifest.json")
for name,path,tag in [
    ("val",R3/"val.json","val"),
    ("train_probe",R3/"train_probe_256.json","train_probe")
]:
    if sha256(path)!=r3_manifest["sha256"][tag]:
        raise RuntimeError(f"Frozen Round3 {name} checksum mismatch")

teacher_core=read_json(TEACHER_DIRS[0][0]/"questions.json")
teacher_extension=read_json(TEACHER_DIRS[1][0]/"questions.json")
from generate_round1 import RESERVED
reserved_prompts=[
    "17+19=?","3*17=?","72/4=?","72-x=34,x=?","x*11=44,x=?"]
reserved_records=[
    {"prompt":p,"answer":RESERVED[p],
     "category":("multiply" if p=="3*17=?" else
                 "add" if p=="17+19=?" else
                 "divide" if p=="72/4=?" else
                 "a_minus_x" if p=="72-x=34,x=?" else "x_times_a")}
    for p in reserved_prompts]

dev_splits={
    "val":read_json(R3/"val.json"),
    "train_probe":read_json(R3/"train_probe_256.json"),
    "teacher680":teacher_core+teacher_extension,
    "reserved5":reserved_records
}
if FAST_SMOKE:
    dev_splits={name:rows[:12] for name,rows in dev_splits.items()}
assert (FAST_SMOKE or {k:len(v) for k,v in dev_splits.items()}==
        {"val":768,"train_probe":256,"teacher680":680,"reserved5":5})

evaluation_dir=RUN_DIR/"development_e5"
evaluation_dir.mkdir(parents=True,exist_ok=True)
model,_=model_from_checkpoint(REPRODUCED_E5)
model.to(device).eval()
measured={}
for split,rows in dev_splits.items():
    pred_file=evaluation_dir/f"{split}_predictions.json"
    stat_file=evaluation_dir/f"{split}_metrics.json"
    if pred_file.exists() and stat_file.exists():
        pred=read_json(pred_file)
        metric=read_json(stat_file)
        if len(pred)!=len(rows) or metric["correct"]!=sum(x["status"]=="correct" for x in pred):
            raise RuntimeError(f"Saved {split} score was corrupted")
        print(f"REUSE FROZEN EVAL {split}: {metric['correct']}/{metric['n']}")
    elif pred_file.exists() or stat_file.exists():
        raise RuntimeError(f"Only part of frozen {split} evaluation exists")
    else:
        pred,metric=evaluate_frozen(model,rows,split)
        save_json_once(pred_file,pred)
        save_json_once(stat_file,metric)
    measured[split]=metric
del model
if device=="cuda":torch.cuda.empty_cache()

# Historic values from TC1 saved runs; NOT the new run's measured values.
historical_e5={"val":614,"train_probe":226,"teacher680":521,"reserved5":4}
print("ACTUAL fresh-run scores:")
for k,v in measured.items():
    print(f"  {k:13} {v['correct']:4}/{v['n']:<4} "
          f"historical E5 reference: {historical_e5[k]}",flush=True)
# Evaluate the actual exported checkpoint as well when it is the archived
# historical E5, rather than assuming fresh re-training produced the exact
# same weights. These are the same frozen development splits, not final Test.
archived_for_eval=repo_root/"dpo/experiments/round9_random_multi_e3_e6/epoch_e05.pt"
submission_eval_source=(archived_for_eval
                        if FINAL_EXPORT_POLICY=="archived_if_available" and
                           archived_for_eval.is_file()
                        else REPRODUCED_E5)
measured_submission=measured
if submission_eval_source!=REPRODUCED_E5 and not FAST_SMOKE:
    print("EVALUATING SELECTED HISTORICAL E5 for submitted-model scores",flush=True)
    submission_eval_dir=RUN_DIR/"submitted_e5_development"
    submission_eval_dir.mkdir(parents=True,exist_ok=True)
    selected_model,_=model_from_checkpoint(submission_eval_source)
    selected_model.to(device).eval()
    measured_submission={}
    for split,rows in dev_splits.items():
        pred_file=submission_eval_dir/f"{split}_predictions.json"
        stat_file=submission_eval_dir/f"{split}_metrics.json"
        if pred_file.exists() and stat_file.exists():
            pred=read_json(pred_file)
            metric=read_json(stat_file)
            if len(pred)!=len(rows) or metric["correct"]!=sum(
                    x["status"]=="correct" for x in pred):
                raise RuntimeError(f"Corrupt submitted E5 eval: {split}")
        elif pred_file.exists() or stat_file.exists():
            raise RuntimeError(f"Incomplete submitted E5 eval: {split}")
        else:
            pred,metric=evaluate_frozen(selected_model,rows,"submission_"+split)
            save_json_once(pred_file,pred)
            save_json_once(stat_file,metric)
        measured_submission[split]=metric
    del selected_model
    if device=="cuda":torch.cuda.empty_cache()
print("SCORES FOR ACTUAL SUBMITTED CHECKPOINT:")
for k,v in measured_submission.items():
    print(f"  {k:13} {v['correct']:4}/{v['n']:<4}",flush=True)
print("No sealed Round3 final Test labels or predictions accessed.")

In [ ]:
# Step 8 — Compare historical E5, export dpo.pt and save provenance
ARCHIVED_E5=repo_root/"dpo/experiments/round9_random_multi_e3_e6/epoch_e05.pt"
replay_model,_=model_from_checkpoint(REPRODUCED_E5)
parity={"archived_e5_exists":ARCHIVED_E5.is_file()}
if ARCHIVED_E5.is_file():
    historical,_=model_from_checkpoint(ARCHIVED_E5)
    ref_weights=historical.state_dict()
    got_weights=replay_model.state_dict()
    if set(ref_weights)!=set(got_weights):
        raise RuntimeError("Replayed and archived E5 architectures/weight keys differ")
    max_abs_difference=max((got_weights[k]-ref_weights[k]).abs().max().item()
                           for k in ref_weights)
    exactly_equal=all(torch.equal(got_weights[k],ref_weights[k])
                      for k in ref_weights)
    parity.update({"archived_e5_sha256":sha256(ARCHIVED_E5),
                   "max_absolute_weight_delta":max_abs_difference,
                   "bitwise_weights_equal":exactly_equal})
    print("HISTORICAL E5 PARITY:",json.dumps(parity,indent=2),flush=True)
    if not exactly_equal:
        print("WARNING: Fresh run is NOT byte/weight-identical to archived E5. "
              "Do not cite historical E5 scores as the new run's score. "
              "The measured scores above are authoritative for this run.",flush=True)
    del historical
else:
    print("Historical E5 is absent. All stages trained from SFT; parity cannot be checked.")

# The archived E5 is the explicit historical selected model. A full replay
# run provides evidence and a numerical parity check; its weights are NOT
# silently substituted if a different GPU/PyTorch version produces drift.
if FINAL_EXPORT_POLICY == "archived_if_available" and ARCHIVED_E5.is_file():
    export_source=ARCHIVED_E5
    export_model,_=model_from_checkpoint(export_source)
    export_origin="selected_archived_E5"
else:
    export_source=REPRODUCED_E5
    export_model=replay_model
    export_origin="fully_retrained_notebook_E5"
print(f"SUBMISSION SOURCE {export_origin}: {export_source}",flush=True)

# Build the original course-compatible weights-only checkpoint.
clean_checkpoint={"model_args":sft_args,
                  "model_state_dict":{k:v.detach().cpu().clone()
                                      for k,v in export_model.state_dict().items()}}
del export_model,replay_model
if FAST_SMOKE:
    print("SMOKE PASS — refusing to publish a tiny-sample model as formal dpo.pt")
else:
    final_dir=repo_root/"dpo"
    formal=final_dir/"dpo.pt"
    artifact=RUN_DIR/"selected_e5_dpo.pt"
    if artifact.exists():
        a=torch.load(artifact,map_location="cpu",weights_only=True)
        if set(a["model_state_dict"])!=set(clean_checkpoint["model_state_dict"]):
            raise RuntimeError("Existing final E5 artifact keys changed")
        if any(not torch.equal(a["model_state_dict"][k],clean_checkpoint["model_state_dict"][k])
               for k in a["model_state_dict"]):
            raise RuntimeError("Existing final E5 artifact differs; refuse overwrite")
    else:
        atomic_save_torch(artifact,clean_checkpoint)
    # Preserve existing dpo/dpo.pt. The exported checkpoint origin is
    # explicit and never implicitly substituted.
    if formal.is_file() and sha256(formal)!=sha256(artifact):
        backup=RUN_DIR/"backups"/f"old_dpo_{sha256(formal)[:16]}.pt"
        backup.parent.mkdir(parents=True,exist_ok=True)
        if not backup.exists():shutil.copy2(formal,backup)
        print(f"BACKUP original dpo.pt -> {backup}",flush=True)
    if not formal.exists() or sha256(formal)!=sha256(artifact):
        temporary=formal.with_suffix(".pt.partial")
        if temporary.exists():raise RuntimeError(f"Stale final checkpoint file: {temporary}")
        shutil.copy2(artifact,temporary)
        temporary.replace(formal)
    if sha256(formal)!=sha256(artifact):
        raise RuntimeError("Final submission checkpoint copy SHA mismatch")
    print(f"FINAL ARTIFACT READY: {formal} SHA256={sha256(formal)}",flush=True)

report={
    "version":"sc3000_final_notebook_v1",
    "mode":"smoke" if FAST_SMOKE else "full_historical_replay",
    "selected":"Random E5 (third epoch of four-epoch E3–E6 schedule)",
    "sources":env_record, "generator_sources_and_manifests":generator_provenance,
    "source_random_100k_sha256":sha256(random_data_path),
    "trained_e5_epoch_checkpoint_sha256":sha256(REPRODUCED_E5),
    "export_checkpoint_origin":export_origin,
    "export_source_sha256":sha256(export_source),
    "archived_parity":parity,
    "measured_dev_scores_of_replay":measured,
    "measured_dev_scores_of_submitted_checkpoint":measured_submission,
    "historical_dev_reference_only":historical_e5,
    "stage_history":{
        key:{"checkpoint":str(file.relative_to(repo_root)),
             "sha256":sha256(file)}
        for key,file in stage_outputs.items()
    },
    "final_test_used":False,
    "limitations":["Seen-heavy teacher680","development-selected E5",
                   "CUDA nondeterminism possible","archived E5 parity disclosed"]
}
report_path=RUN_DIR/"final_reproduction_manifest.json"
if report_path.exists():
    old=read_json(report_path)
    # Timestamp/environment may differ between runs, but numerical outputs
    # and immutable source/selected checkpoint must remain consistent.
    if old["trained_e5_epoch_checkpoint_sha256"]!=report["trained_e5_epoch_checkpoint_sha256"]:
        raise RuntimeError("Existing submission manifest points at another E5")
else:
    save_json_once(report_path,report)
print(f"FINAL AUDIT PASS: {report_path}",flush=True)

#### Step 8 — Submission checklist

1. Submit the **single** `dpo/dpo.ipynb` with original **Step 1–Step 8** headings and actual saved outputs; include the data, starting SFT weights and `model.py`. Data synthesis implementation is embedded under Step 5.
2. Confirm `dpo/dpo.pt` is exported after training and testing. If archived E5 exists, it remains the selected submission checkpoint with source recorded; otherwise the notebook-retrained E5 is exported.
3. Retain `dpo/experiments/final_notebook_e5_reproduction/final_reproduction_manifest.json` and stage checkpoint SHA256 records. Historical E5 and fresh replay results are reported **separately** if different.
4. GitHub CI syntax success is not GPU reproduction. Actual execution and saved outputs are still required by the course.